In [1]:
!pip uninstall -y torch torchvision torchaudio

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 92.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 79.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 223.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 117.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 221.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 207.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 113.8

In [2]:
import torch, platform, psutil

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("RAM:", round(psutil.virtual_memory().total / 1024**3, 2), "GB")

Python: 3.12.12
PyTorch: 2.12.0+cu126
PyTorch CUDA: 12.6
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
RAM: 31.35 GB


In [3]:
import os
import random
import numpy as np

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print(e)

print("Seed fixed:", SEED)

Seed fixed: 42


In [4]:
import gdown
import numpy as np

# For intra-subject

def download_dataset(file_id):
    download_url = f"https://drive.google.com/uc?id={file_id}"
    output_file = "mimic.npy"
    gdown.download(download_url, output_file, quiet=False)
    data = np.load("mimic.npy", allow_pickle=True).item()
    return data

def load_dataset(dataset):
    X = []  # ECG segments
    y = []  # ABP segments

    # Loop over each patient
    for patient_id, signals in dataset.items():
        ecg_segments = signals['ecg']
        abp_segments = signals['abp']
        # Safety check: skip if segment lengths don't match
        if len(ecg_segments) != len(abp_segments):
            continue

        for ecg_seg, abp_seg in zip(ecg_segments, abp_segments):
            if (
                isinstance(ecg_seg, np.ndarray) and ecg_seg.shape == (625,) and
                isinstance(abp_seg, np.ndarray) and abp_seg.shape == (625,)
            ):
                X.append(ecg_seg)
                y.append(abp_seg)


    # Convert lists to numpy arrays
    X = np.array(X)
    y = np.array(y)
    
    return X, y


links = {
    # this is all from mimic-iii dataset
    "no_pp": "18nRE4Z_RBKjT3grYFUiCV0cjUw6gZNd5",
    "bw": "1mui4ifqbZv9e-5GeE4XyU061Kxi-wZdN",
    "bw+maf": "1wRU1UTPOzVqa_2lIm8GLI94jeLdf4pgh",
    "dwt": "1HYGwjSBrgiMV-4CSOiMxJXFyNzlSDBq8",
    "nk2": "1ZrELLKXvmhsuPhirAcvsZ4TcXtRC1rGQ",

    # this is from mimic-iv
    "no_pp_m4":"1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY",
    "bw_m4":"1tK1fPxBhZicWW5Vd5S8DFrKefnHQFVjQ",
    "bw+maf_m4":"12KmtjeVHuZP4omk-AvEBpfwmI1cPn1R7",
    "nk2_m4":"12AHW7_3ytaBs34hGHsIJ-ruRC8vpNE9q",
    "dwt_m4":"1Mmb9NxnwvKBSa1dmNJtAcF63V8N8BxYR"
  }
data = download_dataset(links["no_pp_m4"])
X,y = load_dataset(data)
print(X.shape, y.shape)

Downloading...
From (original): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY
From (redirected): https://drive.google.com/uc?id=1mjalXJyJz50nnFtlsi7D7EuD8jrDI9aY&confirm=t&uuid=21034fa1-134d-46eb-8e10-6fa2a56d3a49
To: /kaggle/working/mimic.npy
100%|██████████| 384M/384M [00:06<00:00, 60.8MB/s] 


(25319, 625) (25319, 625)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [8]:
X_ecg = X
y_abp = y

X_train, X_test, y_train, y_test = train_test_split(
    X_ecg,
    y_abp,
    test_size=0.2,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=SEED
)

print("Raw train:", X_train.shape, y_train.shape)
print("Raw val:", X_val.shape, y_val.shape)
print("Raw test:", X_test.shape, y_test.shape)

Raw train: (16204, 625) (16204, 625)
Raw val: (4051, 625) (4051, 625)
Raw test: (5064, 625) (5064, 625)


In [9]:
X_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = X_scaler.fit_transform(
    X_train.reshape(-1, 1)
).reshape(X_train.shape)

X_val_scaled = X_scaler.transform(
    X_val.reshape(-1, 1)
).reshape(X_val.shape)

X_test_scaled = X_scaler.transform(
    X_test.reshape(-1, 1)
).reshape(X_test.shape)

y_train_scaled = y_scaler.fit_transform(
    y_train.reshape(-1, 1)
).reshape(y_train.shape)

y_val_scaled = y_scaler.transform(
    y_val.reshape(-1, 1)
).reshape(y_val.shape)

y_test_scaled = y_scaler.transform(
    y_test.reshape(-1, 1)
).reshape(y_test.shape)

In [10]:
X_train_scaled = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)
X_val_scaled = torch.tensor(X_val_scaled, dtype=torch.float32).unsqueeze(1)
X_test_scaled = torch.tensor(X_test_scaled, dtype=torch.float32).unsqueeze(1)

y_train_scaled = torch.tensor(y_train_scaled, dtype=torch.float32)
y_val_scaled = torch.tensor(y_val_scaled, dtype=torch.float32)
y_test_scaled = torch.tensor(y_test_scaled, dtype=torch.float32)

print("Train:", X_train_scaled.shape, y_train_scaled.shape)
print("Val:", X_val_scaled.shape, y_val_scaled.shape)
print("Test:", X_test_scaled.shape, y_test_scaled.shape)

Train: torch.Size([16204, 1, 625]) torch.Size([16204, 625])
Val: torch.Size([4051, 1, 625]) torch.Size([4051, 625])
Test: torch.Size([5064, 1, 625]) torch.Size([5064, 625])


In [11]:
g = torch.Generator()
g.manual_seed(SEED)

train_loader = DataLoader(
    TensorDataset(X_train_scaled, y_train_scaled),
    batch_size=32,
    shuffle=True,
    generator=g
)

val_loader = DataLoader(
    TensorDataset(X_val_scaled, y_val_scaled),
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_scaled, y_test_scaled),
    batch_size=32,
    shuffle=False
)

print("Loaders ready")

Loaders ready


In [12]:
!pip install -q braindecode einops huggingface_hub

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from braindecode.models import BENDR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.6/575.6 kB 11.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.8/189.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 11.8 MB/s eta 0:00:00


In [13]:
class BENDR_BP(nn.Module):
    def __init__(self, n_times=625, n_channels_in=1, n_channels_model=20):
        super().__init__()

        self.bendr = BENDR.from_pretrained(
            "braindecode/braindecode-bendr",
            n_chans=n_channels_model,
            n_times=n_times,
            n_outputs=1
        )

        self.channel_adapter = nn.Conv1d(
            n_channels_in, n_channels_model, kernel_size=1
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.GELU(),
            nn.ConvTranspose1d(256, 128, kernel_size=4, stride=4, padding=0),
            nn.GELU(),
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=4, padding=0),
            nn.GELU(),
            nn.Conv1d(64, 1, kernel_size=3, padding=1)
        )

        self.output_mapping = nn.LazyLinear(n_times)

    def forward(self, x):
        x = self.channel_adapter(x)
        features = self.bendr.encoder(x)
        waveform = self.decoder(features).squeeze(1)
        return self.output_mapping(waveform)

In [14]:
def evaluate_model(model, loader, scaler, device):
    model.eval()
    all_preds, all_trues = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            pred = model(xb).cpu().numpy()
            all_preds.append(pred)
            all_trues.append(yb.numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_trues = np.concatenate(all_trues, axis=0)

    orig_shape = all_preds.shape

    preds_orig = scaler.inverse_transform(
        all_preds.reshape(-1, 1)
    ).reshape(orig_shape)
    trues_orig = scaler.inverse_transform(
        all_trues.reshape(-1, 1)
    ).reshape(orig_shape)

    return trues_orig, preds_orig


In [15]:
# =========================
# Train / Validation helper functions
# =========================

def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()

        pred = model(xb)

        if pred.shape != yb.shape:
            pred = pred.view_as(yb)

        loss = criterion(pred, yb)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(train_loader.dataset)


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            pred = model(xb)

            if pred.shape != yb.shape:
                pred = pred.view_as(yb)

            loss = criterion(pred, yb)
            total_loss += loss.item() * xb.size(0)

    return total_loss / len(val_loader.dataset)

In [16]:
# =========================
# Experiment: BENDR Full FT + Discriminative LR 3e-5
# BENDR lr = 3e-5, Head lr = 1e-4
# =========================

import torch
import torch.nn as nn
import pandas as pd
import json
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_disclr3 = "full_ft_discriminative_lr_3e5"

best_model_path_disclr3 = "bendr_full_ft_discriminative_lr_3e5_m4.pth"
history_path_disclr3 = "history_full_ft_discriminative_lr_3e5_m4.csv"
metrics_path_disclr3 = "metrics_full_ft_discriminative_lr_3e5_m4.json"

model_disclr3 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

# Initialize LazyLinear / lazy params
model_disclr3.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device)
    _ = model_disclr3(x_dummy)

# Full fine-tuning: all BENDR trainable
for param in model_disclr3.bendr.parameters():
    param.requires_grad = True

# Task-specific head trainable
for module in [model_disclr3.channel_adapter, model_disclr3.decoder, model_disclr3.output_mapping]:
    for param in module.parameters():
        param.requires_grad = True

trainable_params_disclr3 = [p for p in model_disclr3.parameters() if p.requires_grad]

total_params_disclr3 = sum(p.numel() for p in model_disclr3.parameters())
trainable_count_disclr3 = sum(p.numel() for p in trainable_params_disclr3)
frozen_count_disclr3 = total_params_disclr3 - trainable_count_disclr3

print("Total parameters:", total_params_disclr3)
print("Trainable parameters:", trainable_count_disclr3)
print("Frozen parameters:", frozen_count_disclr3)
print("Trainable percentage:", round(100 * trainable_count_disclr3 / total_params_disclr3, 4), "%")

Device: cuda


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/629M [00:00<?, ?B/s]

Total parameters: 157970996
Trainable parameters: 157970996
Frozen parameters: 0
Trainable percentage: 100.0 %


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  return F.dropout2d(input, self.p, self.training, self.inplace)


In [17]:
# =========================
# AdamW Optimizer: BENDR lr = 3e-5, Head lr = 1e-4
# =========================

def create_discriminative_lr_optimizer_3e5(
    model,
    bendr_lr=3e-5,
    head_lr=1e-4,
    weight_decay=1e-2
):
    bendr_decay = []
    bendr_no_decay = []

    head_decay = []
    head_no_decay = []

    no_decay_keywords = ["bias", "norm", "bn", "layernorm", "layer_norm"]

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_low = name.lower()
        is_no_decay = any(k in name_low for k in no_decay_keywords)

        is_head = (
            "channel_adapter" in name
            or "decoder" in name
            or "output_mapping" in name
        )

        if is_head:
            if is_no_decay:
                head_no_decay.append(param)
            else:
                head_decay.append(param)
        else:
            if is_no_decay:
                bendr_no_decay.append(param)
            else:
                bendr_decay.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": bendr_decay, "lr": bendr_lr, "weight_decay": weight_decay},
            {"params": bendr_no_decay, "lr": bendr_lr, "weight_decay": 0.0},
            {"params": head_decay, "lr": head_lr, "weight_decay": weight_decay},
            {"params": head_no_decay, "lr": head_lr, "weight_decay": 0.0},
        ]
    )

    print("BENDR decay params:", sum(p.numel() for p in bendr_decay))
    print("BENDR no-decay params:", sum(p.numel() for p in bendr_no_decay))
    print("Head decay params:", sum(p.numel() for p in head_decay))
    print("Head no-decay params:", sum(p.numel() for p in head_no_decay))

    return optimizer


criterion_disclr3 = nn.MSELoss()

optimizer_disclr3 = create_discriminative_lr_optimizer_3e5(
    model_disclr3,
    bendr_lr=3e-5,
    head_lr=1e-4,
    weight_decay=1e-2
)

BENDR decay params: 157043225
BENDR no-decay params: 98337
Head decay params: 828340
Head no-decay params: 1094


In [18]:
# =========================
# Train Full FT + Discriminative LR 3e-5
# =========================

num_epochs_disclr3 = 30
best_val_loss_disclr3 = float("inf")

history_disclr3 = []

for epoch in range(num_epochs_disclr3):

    train_loss_disclr3 = train_one_epoch(
        model_disclr3,
        train_loader,
        optimizer_disclr3,
        criterion_disclr3,
        device
    )

    val_loss_disclr3 = validate_one_epoch(
        model_disclr3,
        val_loader,
        criterion_disclr3,
        device
    )

    history_disclr3.append({
        "method": tuning_method_disclr3,
        "epoch": epoch + 1,
        "train_loss": float(train_loss_disclr3),
        "val_loss": float(val_loss_disclr3),
        "trainable_parameters": int(trainable_count_disclr3),
        "total_parameters": int(total_params_disclr3),
        "frozen_parameters": int(frozen_count_disclr3),
        "trainable_percentage": float(100 * trainable_count_disclr3 / total_params_disclr3),
        "bendr_lr": 3e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW"
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs_disclr3}] | "
        f"Train Loss: {train_loss_disclr3:.6f} | "
        f"Val Loss: {val_loss_disclr3:.6f}"
    )

    if val_loss_disclr3 < best_val_loss_disclr3:
        best_val_loss_disclr3 = val_loss_disclr3

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_disclr3,
            "model_state_dict": model_disclr3.state_dict(),
            "optimizer_state_dict": optimizer_disclr3.state_dict(),
            "best_val_loss": float(best_val_loss_disclr3),
            "trainable_parameters": int(trainable_count_disclr3),
            "total_parameters": int(total_params_disclr3),
            "frozen_parameters": int(frozen_count_disclr3),
            "trainable_percentage": float(100 * trainable_count_disclr3 / total_params_disclr3),
            "bendr_lr": 3e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW"
        }, best_model_path_disclr3)

        print(f"Saved best discriminative LR 3e-5 model: {best_model_path_disclr3}")

pd.DataFrame(history_disclr3).to_csv(history_path_disclr3, index=False)

Epoch [1/30] | Train Loss: 0.701028 | Val Loss: 0.507479
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [2/30] | Train Loss: 0.449201 | Val Loss: 0.396287
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [3/30] | Train Loss: 0.356084 | Val Loss: 0.319861
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [4/30] | Train Loss: 0.290554 | Val Loss: 0.274603
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [5/30] | Train Loss: 0.244129 | Val Loss: 0.226324
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [6/30] | Train Loss: 0.211738 | Val Loss: 0.224498
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [7/30] | Train Loss: 0.190985 | Val Loss: 0.196696
Saved best discriminative LR 3e-5 model: bendr_full_ft_discriminative_lr_3e5_m4.pth
Epoch [8/30] 

In [19]:
# =========================
# Load best discriminative LR 3e-5 model
# =========================

checkpoint_disclr3 = torch.load(best_model_path_disclr3, map_location=device)

model_disclr3.load_state_dict(checkpoint_disclr3["model_state_dict"])
model_disclr3.eval()

best_epoch_disclr3 = checkpoint_disclr3["epoch"]
best_val_loss_disclr3 = checkpoint_disclr3["best_val_loss"]

print("Loaded best discriminative LR 3e-5 model")
print("Best epoch:", best_epoch_disclr3)
print("Best val loss:", best_val_loss_disclr3)

Loaded best discriminative LR 3e-5 model
Best epoch: 26
Best val loss: 0.13642742523614992


In [20]:
# =========================
# Evaluate Full FT + Discriminative LR 3e-5
# =========================

y_true_disclr3, y_pred_disclr3 = evaluate_model(
    model_disclr3,
    test_loader,
    y_scaler,
    device
)

print("y_true_disclr3 shape:", y_true_disclr3.shape)
print("y_pred_disclr3 shape:", y_pred_disclr3.shape)

print("y_true min/max:", y_true_disclr3.min(), y_true_disclr3.max())
print("y_pred min/max:", y_pred_disclr3.min(), y_pred_disclr3.max())

y_true_disclr3 shape: (5064, 625)
y_pred_disclr3 shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 46.93488 201.12581


In [21]:
# =========================
# SBP / DBP metrics for Full FT + Discriminative LR 3e-5
# =========================

from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_disclr3, dbp_true_disclr3 = [], []
sbp_pred_disclr3, dbp_pred_disclr3 = [], []

for true_sig, pred_sig in zip(y_true_disclr3, y_pred_disclr3):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_disclr3.append(sbp_t)
    dbp_true_disclr3.append(dbp_t)
    sbp_pred_disclr3.append(sbp_p)
    dbp_pred_disclr3.append(dbp_p)

sbp_true_disclr3 = np.array(sbp_true_disclr3)
dbp_true_disclr3 = np.array(dbp_true_disclr3)
sbp_pred_disclr3 = np.array(sbp_pred_disclr3)
dbp_pred_disclr3 = np.array(dbp_pred_disclr3)

sbp_mae_disclr3 = mean_absolute_error(sbp_true_disclr3, sbp_pred_disclr3)
dbp_mae_disclr3 = mean_absolute_error(dbp_true_disclr3, dbp_pred_disclr3)

sbp_rmse_disclr3 = np.sqrt(mean_squared_error(sbp_true_disclr3, sbp_pred_disclr3))
dbp_rmse_disclr3 = np.sqrt(mean_squared_error(dbp_true_disclr3, dbp_pred_disclr3))

sbp_r2_disclr3 = r2_score(sbp_true_disclr3, sbp_pred_disclr3)
dbp_r2_disclr3 = r2_score(dbp_true_disclr3, dbp_pred_disclr3)

avg_mae_disclr3 = (sbp_mae_disclr3 + dbp_mae_disclr3) / 2

print("Full FT + Discriminative LR 3e-5 Results — Real mmHg Scale")
print("==========================================================")

print("SBP")
print(f"MAE  : {sbp_mae_disclr3:.3f}")
print(f"RMSE : {sbp_rmse_disclr3:.3f}")
print(f"R2   : {sbp_r2_disclr3:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_disclr3:.3f}")
print(f"RMSE : {dbp_rmse_disclr3:.3f}")
print(f"R2   : {dbp_r2_disclr3:.4f}")

print("\nTable format:")
print(f"{sbp_mae_disclr3:.2f}/{dbp_mae_disclr3:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_disclr3:.3f}")

Full FT + Discriminative LR 3e-5 Results — Real mmHg Scale
SBP
MAE  : 5.650
RMSE : 7.907
R2   : 0.6913

DBP
MAE  : 4.744
RMSE : 6.392
R2   : 0.4566

Table format:
5.65/4.74

Average MAE:
5.197


In [25]:
# =========================
# Save Full FT + Discriminative LR 3e-5 Metrics
# =========================

disclr3_metrics = {
    "method": tuning_method_disclr3,
    "best_val_loss": float(best_val_loss_disclr3),
    "best_epoch": int(best_epoch_disclr3),

    "SBP_MAE": float(sbp_mae_disclr3),
    "DBP_MAE": float(dbp_mae_disclr3),
    "AVG_MAE": float(avg_mae_disclr3),

    "SBP_RMSE": float(sbp_rmse_disclr3),
    "DBP_RMSE": float(dbp_rmse_disclr3),
    "SBP_R2": float(sbp_r2_disclr3),
    "DBP_R2": float(dbp_r2_disclr3),

    "table_format": f"{sbp_mae_disclr3:.2f}/{dbp_mae_disclr3:.2f}",
    "best_model_path": best_model_path_disclr3,

    "trainable_parameters": int(checkpoint_disclr3["trainable_parameters"]),
    "total_parameters": int(checkpoint_disclr3["total_parameters"]),
    "frozen_parameters": int(checkpoint_disclr3["frozen_parameters"]),
    "trainable_percentage": float(checkpoint_disclr3["trainable_percentage"]),

    "bendr_lr": float(checkpoint_disclr3["bendr_lr"]),
    "head_lr": float(checkpoint_disclr3["head_lr"]),
    "optimizer": checkpoint_disclr3["optimizer"]
}

with open(metrics_path_disclr3, "w") as f:
    json.dump(disclr3_metrics, f, indent=4)

pd.DataFrame([disclr3_metrics]).to_csv("metrics_full_ft_discriminative_lr_3e5_m4.csv", index=False)

disclr3_metrics

{'method': 'full_ft_discriminative_lr_3e5',
 'best_val_loss': 0.13642742523614992,
 'best_epoch': 26,
 'SBP_MAE': 5.649753570556641,
 'DBP_MAE': 4.743524551391602,
 'AVG_MAE': 5.196639060974121,
 'SBP_RMSE': 7.906907849592239,
 'DBP_RMSE': 6.391688922841674,
 'SBP_R2': 0.6913479566574097,
 'DBP_R2': 0.456606924533844,
 'table_format': '5.65/4.74',
 'best_model_path': 'bendr_full_ft_discriminative_lr_3e5_m4.pth',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'frozen_parameters': 0,
 'trainable_percentage': 100.0,
 'bendr_lr': 3e-05,
 'head_lr': 0.0001,
 'optimizer': 'AdamW'}

In [ ]:
№

# Full finetuning

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_full = "full_finetuning_lr_1e-4"

best_model_path_full = "bendr_full_finetuning_lr_1e-4_m4_seed42.pth"
history_path_full = "history_bendr_full_finetuning_lr_1e-4_m4_seed42.csv"
metrics_path_full = "metrics_bendr_full_finetuning_lr_1e-4_m4_seed42.json"

model_full = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_full.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_full(x_dummy)

criterion_full = nn.MSELoss()

print("Full fine-tuning setup ready")

Device: cuda
Full fine-tuning setup ready


In [27]:
for param in model_full.parameters():
    param.requires_grad = True

total_params_full = sum(p.numel() for p in model_full.parameters())
trainable_params_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)

print("Total parameters:", total_params_full)
print("Trainable parameters:", trainable_params_full)
print("Trainable percentage:", 100 * trainable_params_full / total_params_full)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [28]:
optimizer_full = torch.optim.AdamW(
    [p for p in model_full.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)

history_full = []

print("Optimizer ready")

Optimizer ready


In [30]:
import time
import pandas as pd
import torch

num_epochs_full = 30
best_val_loss_full = float("inf")

start_time_full = time.time()

for epoch in range(num_epochs_full):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_full,
        train_loader,
        optimizer_full,
        criterion_full,
        device
    )

    val_loss = validate_one_epoch(
        model_full,
        val_loader,
        criterion_full,
        device
    )

    history_full.append({
        "method": tuning_method_full,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "learning_rate": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_full),
        "total_parameters": int(total_params_full),
        "seed": SEED
    })

    if val_loss < best_val_loss_full:
        best_val_loss_full = val_loss
        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_full,
            "model_state_dict": model_full.state_dict(),
            "optimizer_state_dict": optimizer_full.state_dict(),
            "best_val_loss": float(best_val_loss_full),
            "learning_rate": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_full),
            "total_parameters": int(total_params_full),
            "seed": SEED
        }, best_model_path_full)
        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_full}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_full).to_csv(history_path_full, index=False)

print("Best val loss:", best_val_loss_full)
print("Total training time:", time.time() - start_time_full)

Epoch [1/30] | Train Loss: 0.688384 | Val Loss: 0.501967 | Time: 14.13s saved
Epoch [2/30] | Train Loss: 0.416747 | Val Loss: 0.344354 | Time: 14.88s saved
Epoch [3/30] | Train Loss: 0.311096 | Val Loss: 0.293689 | Time: 15.11s saved
Epoch [4/30] | Train Loss: 0.256292 | Val Loss: 0.249864 | Time: 15.40s saved
Epoch [5/30] | Train Loss: 0.219234 | Val Loss: 0.229021 | Time: 15.55s saved
Epoch [6/30] | Train Loss: 0.198524 | Val Loss: 0.213406 | Time: 15.53s saved
Epoch [7/30] | Train Loss: 0.181550 | Val Loss: 0.208264 | Time: 15.32s saved
Epoch [8/30] | Train Loss: 0.168374 | Val Loss: 0.193551 | Time: 15.15s saved
Epoch [9/30] | Train Loss: 0.156570 | Val Loss: 0.193727 | Time: 13.87s 
Epoch [10/30] | Train Loss: 0.148910 | Val Loss: 0.186352 | Time: 15.31s saved
Epoch [11/30] | Train Loss: 0.142193 | Val Loss: 0.165712 | Time: 15.27s saved
Epoch [12/30] | Train Loss: 0.135691 | Val Loss: 0.167361 | Time: 13.91s 
Epoch [13/30] | Train Loss: 0.130111 | Val Loss: 0.163050 | Time: 15.23

In [31]:
import json
import numpy as np
import torch
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

checkpoint_full = torch.load(best_model_path_full, map_location=device)

model_full.load_state_dict(checkpoint_full["model_state_dict"])
model_full = model_full.to(device)
model_full.eval()

print("Loaded best full fine-tuning model")
print("Best epoch:", checkpoint_full["epoch"])
print("Best val loss:", checkpoint_full["best_val_loss"])

Loaded best full fine-tuning model
Best epoch: 29
Best val loss: 0.13687082226851405


In [32]:
y_true_final = []
y_pred_final = []

model_full.eval()

with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device).float()

        pred = model_full(bx)

        pred_np = pred.detach().cpu().numpy()
        true_np = by.detach().cpu().numpy()

        pred_unscaled = y_scaler.inverse_transform(
            pred_np.reshape(-1, 1)
        ).reshape(pred_np.shape)

        true_unscaled = y_scaler.inverse_transform(
            true_np.reshape(-1, 1)
        ).reshape(true_np.shape)

        y_pred_final.append(pred_unscaled)
        y_true_final.append(true_unscaled)

y_pred_final = np.vstack(y_pred_final)
y_true_final = np.vstack(y_true_final)

print("y_true_final:", y_true_final.shape)
print("y_pred_final:", y_pred_final.shape)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  return F.dropout2d(input, self.p, self.training, self.inplace)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to 

y_true_final: (5064, 625)
y_pred_final: (5064, 625)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  return F.dropout2d(input, self.p, self.training, self.inplace)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to 

In [33]:
def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


def compute_metrics(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(name)
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.4f}")

    return mae, rmse, r2

In [34]:
sbp_true, dbp_true = [], []
sbp_pred, dbp_pred = [], []

for true_sig, pred_sig in zip(y_true_final, y_pred_final):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true.append(sbp_t)
    dbp_true.append(dbp_t)
    sbp_pred.append(sbp_p)
    dbp_pred.append(dbp_p)

sbp_true = np.array(sbp_true)
dbp_true = np.array(dbp_true)
sbp_pred = np.array(sbp_pred)
dbp_pred = np.array(dbp_pred)

mask_sbp = ~np.isnan(sbp_true) & ~np.isnan(sbp_pred)
mask_dbp = ~np.isnan(dbp_true) & ~np.isnan(dbp_pred)

sbp_true = sbp_true[mask_sbp]
sbp_pred = sbp_pred[mask_sbp]

dbp_true = dbp_true[mask_dbp]
dbp_pred = dbp_pred[mask_dbp]

print("SBP samples:", len(sbp_true))
print("DBP samples:", len(dbp_true))

SBP samples: 5064
DBP samples: 5064


In [35]:
print("Full Fine-Tuning Results — Real mmHg Scale")
print("==========================================")

sbp_mae, sbp_rmse, sbp_r2 = compute_metrics(sbp_true, sbp_pred, "SBP")
print()
dbp_mae, dbp_rmse, dbp_r2 = compute_metrics(dbp_true, dbp_pred, "DBP")

avg_mae = (sbp_mae + dbp_mae) / 2

print()
print("Table format:")
print(f"{sbp_mae:.2f}/{dbp_mae:.2f}")

print()
print("Average MAE:")
print(f"{avg_mae:.3f}")

Full Fine-Tuning Results — Real mmHg Scale
SBP
MAE  : 6.019
RMSE : 8.311
R2   : 0.6590

DBP
MAE  : 4.725
RMSE : 6.291
R2   : 0.4736

Table format:
6.02/4.73

Average MAE:
5.372


In [36]:
metrics_full = {
    "model": "BENDR",
    "method": tuning_method_full,
    "best_epoch": int(checkpoint_full["epoch"]),
    "best_val_loss": float(checkpoint_full["best_val_loss"]),
    "SBP_MAE": float(sbp_mae),
    "DBP_MAE": float(dbp_mae),
    "AVG_MAE": float(avg_mae),
    "SBP_RMSE": float(sbp_rmse),
    "DBP_RMSE": float(dbp_rmse),
    "SBP_R2": float(sbp_r2),
    "DBP_R2": float(dbp_r2),
    "table_format": f"{sbp_mae:.2f}/{dbp_mae:.2f}",
    "learning_rate": float(checkpoint_full["learning_rate"]),
    "optimizer": checkpoint_full["optimizer"],
    "trainable_parameters": int(checkpoint_full["trainable_parameters"]),
    "total_parameters": int(checkpoint_full["total_parameters"]),
    "seed": int(checkpoint_full["seed"])
}

with open(metrics_path_full, "w") as f:
    json.dump(metrics_full, f, indent=4)

metrics_full

{'model': 'BENDR',
 'method': 'full_finetuning_lr_1e-4',
 'best_epoch': 29,
 'best_val_loss': 0.13687082226851405,
 'SBP_MAE': 6.019430160522461,
 'DBP_MAE': 4.725004196166992,
 'AVG_MAE': 5.372217178344727,
 'SBP_RMSE': 8.31095240821656,
 'DBP_RMSE': 6.290996524245297,
 'SBP_R2': 0.658997654914856,
 'DBP_R2': 0.4735928773880005,
 'table_format': '6.02/4.73',
 'learning_rate': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# Full FT + discr LR 2e-5.

In [37]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_discr_2e5 = "full_ft_discriminative_lr_2e-5"

best_model_path_discr_2e5 = "bendr_full_ft_discriminative_lr_2e-5_m4_seed42.pth"
history_path_discr_2e5 = "history_bendr_full_ft_discriminative_lr_2e-5_m4_seed42.csv"
metrics_path_discr_2e5 = "metrics_bendr_full_ft_discriminative_lr_2e-5_m4_seed42.json"

model_discr_2e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_discr_2e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_discr_2e5(x_dummy)

criterion_discr_2e5 = nn.MSELoss()

print("Discriminative LR 2e-5 setup ready")

Device: cuda
Discriminative LR 2e-5 setup ready


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  return F.dropout2d(input, self.p, self.training, self.inplace)


In [38]:
for param in model_discr_2e5.parameters():
    param.requires_grad = True

total_params_discr_2e5 = sum(p.numel() for p in model_discr_2e5.parameters())
trainable_params_discr_2e5 = sum(p.numel() for p in model_discr_2e5.parameters() if p.requires_grad)

print("Total parameters:", total_params_discr_2e5)
print("Trainable parameters:", trainable_params_discr_2e5)
print("Trainable percentage:", 100 * trainable_params_discr_2e5 / total_params_discr_2e5)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [39]:
backbone_params = []
head_params = []

for name, param in model_discr_2e5.named_parameters():
    if not param.requires_grad:
        continue

    if "head" in name.lower() or "regressor" in name.lower() or "regression" in name.lower() or "final" in name.lower() or "output" in name.lower():
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_discr_2e5 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 1e-4, "weight_decay": 0.01}
    ]
)

history_discr_2e5 = []

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Optimizer ready")

Backbone params: 157042402
Head params: 928594
Optimizer ready


In [40]:
import time
import pandas as pd
import torch

num_epochs_discr_2e5 = 30
best_val_loss_discr_2e5 = float("inf")

start_time_discr_2e5 = time.time()

for epoch in range(num_epochs_discr_2e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_discr_2e5,
        train_loader,
        optimizer_discr_2e5,
        criterion_discr_2e5,
        device
    )

    val_loss = validate_one_epoch(
        model_discr_2e5,
        val_loader,
        criterion_discr_2e5,
        device
    )

    history_discr_2e5.append({
        "method": tuning_method_discr_2e5,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_discr_2e5),
        "total_parameters": int(total_params_discr_2e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_discr_2e5:
        best_val_loss_discr_2e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_discr_2e5,
            "model_state_dict": model_discr_2e5.state_dict(),
            "optimizer_state_dict": optimizer_discr_2e5.state_dict(),
            "best_val_loss": float(best_val_loss_discr_2e5),
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_discr_2e5),
            "total_parameters": int(total_params_discr_2e5),
            "seed": SEED
        }, best_model_path_discr_2e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_discr_2e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_discr_2e5).to_csv(history_path_discr_2e5, index=False)

print("Best val loss:", best_val_loss_discr_2e5)
print("Total training time:", time.time() - start_time_discr_2e5)

Epoch [1/30] | Train Loss: 0.763024 | Val Loss: 0.568251 | Time: 14.26s saved
Epoch [2/30] | Train Loss: 0.484876 | Val Loss: 0.440295 | Time: 14.81s saved
Epoch [3/30] | Train Loss: 0.397318 | Val Loss: 0.418047 | Time: 14.95s saved
Epoch [4/30] | Train Loss: 0.330921 | Val Loss: 0.309612 | Time: 15.37s saved
Epoch [5/30] | Train Loss: 0.282219 | Val Loss: 0.274271 | Time: 15.55s saved
Epoch [6/30] | Train Loss: 0.248511 | Val Loss: 0.243747 | Time: 15.05s saved
Epoch [7/30] | Train Loss: 0.221917 | Val Loss: 0.231476 | Time: 15.00s saved
Epoch [8/30] | Train Loss: 0.199435 | Val Loss: 0.211916 | Time: 15.19s saved
Epoch [9/30] | Train Loss: 0.184693 | Val Loss: 0.201533 | Time: 15.27s saved
Epoch [10/30] | Train Loss: 0.172519 | Val Loss: 0.187533 | Time: 15.17s saved
Epoch [11/30] | Train Loss: 0.160488 | Val Loss: 0.178965 | Time: 15.13s saved
Epoch [12/30] | Train Loss: 0.152621 | Val Loss: 0.175101 | Time: 15.09s saved
Epoch [13/30] | Train Loss: 0.145813 | Val Loss: 0.166633 | T

In [41]:
checkpoint_discr_2e5 = torch.load(best_model_path_discr_2e5, map_location=device)

model_discr_2e5.load_state_dict(checkpoint_discr_2e5["model_state_dict"])
model_discr_2e5 = model_discr_2e5.to(device)
model_discr_2e5.eval()

print("Loaded best discriminative LR 2e-5 model")
print("Best epoch:", checkpoint_discr_2e5["epoch"])
print("Best val loss:", checkpoint_discr_2e5["best_val_loss"])

Loaded best discriminative LR 2e-5 model
Best epoch: 30
Best val loss: 0.1379485985627883


In [46]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="dropout2d: Received a 3D input to dropout2d*"
)

In [47]:
y_true_discr_2e5 = []
y_pred_discr_2e5 = []

model_discr_2e5.eval()

with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device).float()

        pred = model_discr_2e5(bx)

        pred_np = pred.detach().cpu().numpy()
        true_np = by.detach().cpu().numpy()

        pred_unscaled = y_scaler.inverse_transform(
            pred_np.reshape(-1, 1)
        ).reshape(pred_np.shape)

        true_unscaled = y_scaler.inverse_transform(
            true_np.reshape(-1, 1)
        ).reshape(true_np.shape)

        y_pred_discr_2e5.append(pred_unscaled)
        y_true_discr_2e5.append(true_unscaled)

y_pred_discr_2e5 = np.vstack(y_pred_discr_2e5)
y_true_discr_2e5 = np.vstack(y_true_discr_2e5)

print("y_true_discr_2e5:", y_true_discr_2e5.shape)
print("y_pred_discr_2e5:", y_pred_discr_2e5.shape)

y_true_discr_2e5: (5064, 625)
y_pred_discr_2e5: (5064, 625)


In [48]:
sbp_true_discr_2e5, dbp_true_discr_2e5 = [], []
sbp_pred_discr_2e5, dbp_pred_discr_2e5 = [], []

for true_sig, pred_sig in zip(y_true_discr_2e5, y_pred_discr_2e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_discr_2e5.append(sbp_t)
    dbp_true_discr_2e5.append(dbp_t)
    sbp_pred_discr_2e5.append(sbp_p)
    dbp_pred_discr_2e5.append(dbp_p)

sbp_true_discr_2e5 = np.array(sbp_true_discr_2e5)
dbp_true_discr_2e5 = np.array(dbp_true_discr_2e5)
sbp_pred_discr_2e5 = np.array(sbp_pred_discr_2e5)
dbp_pred_discr_2e5 = np.array(dbp_pred_discr_2e5)

mask_sbp = ~np.isnan(sbp_true_discr_2e5) & ~np.isnan(sbp_pred_discr_2e5)
mask_dbp = ~np.isnan(dbp_true_discr_2e5) & ~np.isnan(dbp_pred_discr_2e5)

sbp_true_discr_2e5 = sbp_true_discr_2e5[mask_sbp]
sbp_pred_discr_2e5 = sbp_pred_discr_2e5[mask_sbp]

dbp_true_discr_2e5 = dbp_true_discr_2e5[mask_dbp]
dbp_pred_discr_2e5 = dbp_pred_discr_2e5[mask_dbp]

print("Full FT + Discriminative LR 2e-5 Results — Real mmHg Scale")
print("==========================================================")

sbp_mae_discr_2e5, sbp_rmse_discr_2e5, sbp_r2_discr_2e5 = compute_metrics(
    sbp_true_discr_2e5,
    sbp_pred_discr_2e5,
    "SBP"
)

print()

dbp_mae_discr_2e5, dbp_rmse_discr_2e5, dbp_r2_discr_2e5 = compute_metrics(
    dbp_true_discr_2e5,
    dbp_pred_discr_2e5,
    "DBP"
)

avg_mae_discr_2e5 = (sbp_mae_discr_2e5 + dbp_mae_discr_2e5) / 2

print()
print("Table format:")
print(f"{sbp_mae_discr_2e5:.2f}/{dbp_mae_discr_2e5:.2f}")

print()
print("Average MAE:")
print(f"{avg_mae_discr_2e5:.3f}")

Full FT + Discriminative LR 2e-5 Results — Real mmHg Scale
SBP
MAE  : 5.498
RMSE : 7.808
R2   : 0.6990

DBP
MAE  : 4.791
RMSE : 6.402
R2   : 0.4548

Table format:
5.50/4.79

Average MAE:
5.144


In [49]:
metrics_discr_2e5 = {
    "model": "BENDR",
    "method": tuning_method_discr_2e5,
    "best_epoch": int(checkpoint_discr_2e5["epoch"]),
    "best_val_loss": float(checkpoint_discr_2e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_discr_2e5),
    "DBP_MAE": float(dbp_mae_discr_2e5),
    "AVG_MAE": float(avg_mae_discr_2e5),
    "SBP_RMSE": float(sbp_rmse_discr_2e5),
    "DBP_RMSE": float(dbp_rmse_discr_2e5),
    "SBP_R2": float(sbp_r2_discr_2e5),
    "DBP_R2": float(dbp_r2_discr_2e5),
    "table_format": f"{sbp_mae_discr_2e5:.2f}/{dbp_mae_discr_2e5:.2f}",
    "backbone_lr": float(checkpoint_discr_2e5["backbone_lr"]),
    "head_lr": float(checkpoint_discr_2e5["head_lr"]),
    "optimizer": checkpoint_discr_2e5["optimizer"],
    "trainable_parameters": int(checkpoint_discr_2e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_discr_2e5["total_parameters"]),
    "seed": int(checkpoint_discr_2e5["seed"])
}

with open(metrics_path_discr_2e5, "w") as f:
    json.dump(metrics_discr_2e5, f, indent=4)

metrics_discr_2e5

{'model': 'BENDR',
 'method': 'full_ft_discriminative_lr_2e-5',
 'best_epoch': 30,
 'best_val_loss': 0.1379485985627883,
 'SBP_MAE': 5.4976806640625,
 'DBP_MAE': 4.790627479553223,
 'AVG_MAE': 5.144154071807861,
 'SBP_RMSE': 7.8083294532247844,
 'DBP_RMSE': 6.40207889886519,
 'SBP_R2': 0.6989961862564087,
 'DBP_R2': 0.4548388719558716,
 'table_format': '5.50/4.79',
 'backbone_lr': 2e-05,
 'head_lr': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# Full FT + discr LR 1e-5

In [50]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_discr_1e5 = "full_ft_discriminative_lr_1e-5"

best_model_path_discr_1e5 = "bendr_full_ft_discriminative_lr_1e-5_m4_seed42.pth"
history_path_discr_1e5 = "history_bendr_full_ft_discriminative_lr_1e-5_m4_seed42.csv"
metrics_path_discr_1e5 = "metrics_bendr_full_ft_discriminative_lr_1e-5_m4_seed42.json"

model_discr_1e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_discr_1e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_discr_1e5(x_dummy)

criterion_discr_1e5 = nn.MSELoss()

print("Discriminative LR 1e-5 setup ready")

Device: cuda
Discriminative LR 1e-5 setup ready


In [51]:
for param in model_discr_1e5.parameters():
    param.requires_grad = True

total_params_discr_1e5 = sum(p.numel() for p in model_discr_1e5.parameters())
trainable_params_discr_1e5 = sum(p.numel() for p in model_discr_1e5.parameters() if p.requires_grad)

print("Total parameters:", total_params_discr_1e5)
print("Trainable parameters:", trainable_params_discr_1e5)
print("Trainable percentage:", 100 * trainable_params_discr_1e5 / total_params_discr_1e5)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [52]:
backbone_params = []
head_params = []

for name, param in model_discr_1e5.named_parameters():
    if not param.requires_grad:
        continue

    if "head" in name.lower() or "regressor" in name.lower() or "regression" in name.lower() or "final" in name.lower() or "output" in name.lower():
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_discr_1e5 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 1e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 1e-4, "weight_decay": 0.01}
    ]
)

history_discr_1e5 = []

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Optimizer ready")

Backbone params: 157042402
Head params: 928594
Optimizer ready


In [53]:
import time
import pandas as pd
import torch

num_epochs_discr_1e5 = 30
best_val_loss_discr_1e5 = float("inf")

start_time_discr_1e5 = time.time()

for epoch in range(num_epochs_discr_1e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_discr_1e5,
        train_loader,
        optimizer_discr_1e5,
        criterion_discr_1e5,
        device
    )

    val_loss = validate_one_epoch(
        model_discr_1e5,
        val_loader,
        criterion_discr_1e5,
        device
    )

    history_discr_1e5.append({
        "method": tuning_method_discr_1e5,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 1e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_discr_1e5),
        "total_parameters": int(total_params_discr_1e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_discr_1e5:
        best_val_loss_discr_1e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_discr_1e5,
            "model_state_dict": model_discr_1e5.state_dict(),
            "optimizer_state_dict": optimizer_discr_1e5.state_dict(),
            "best_val_loss": float(best_val_loss_discr_1e5),
            "backbone_lr": 1e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_discr_1e5),
            "total_parameters": int(total_params_discr_1e5),
            "seed": SEED
        }, best_model_path_discr_1e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_discr_1e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_discr_1e5).to_csv(history_path_discr_1e5, index=False)

print("Best val loss:", best_val_loss_discr_1e5)
print("Total training time:", time.time() - start_time_discr_1e5)

Epoch [1/30] | Train Loss: 0.823816 | Val Loss: 0.626137 | Time: 14.20s saved
Epoch [2/30] | Train Loss: 0.536129 | Val Loss: 0.485148 | Time: 14.92s saved
Epoch [3/30] | Train Loss: 0.449406 | Val Loss: 0.418971 | Time: 15.09s saved
Epoch [4/30] | Train Loss: 0.388899 | Val Loss: 0.362028 | Time: 15.43s saved
Epoch [5/30] | Train Loss: 0.336067 | Val Loss: 0.328282 | Time: 15.44s saved
Epoch [6/30] | Train Loss: 0.295217 | Val Loss: 0.290682 | Time: 15.33s saved
Epoch [7/30] | Train Loss: 0.262827 | Val Loss: 0.262821 | Time: 15.17s saved
Epoch [8/30] | Train Loss: 0.237862 | Val Loss: 0.236755 | Time: 15.26s saved
Epoch [9/30] | Train Loss: 0.215995 | Val Loss: 0.218228 | Time: 15.10s saved
Epoch [10/30] | Train Loss: 0.197165 | Val Loss: 0.217274 | Time: 15.28s saved
Epoch [11/30] | Train Loss: 0.184525 | Val Loss: 0.195899 | Time: 14.95s saved
Epoch [12/30] | Train Loss: 0.173774 | Val Loss: 0.196816 | Time: 13.92s 
Epoch [13/30] | Train Loss: 0.165632 | Val Loss: 0.183533 | Time: 

In [54]:
checkpoint_discr_1e5 = torch.load(best_model_path_discr_1e5, map_location=device)

model_discr_1e5.load_state_dict(checkpoint_discr_1e5["model_state_dict"])
model_discr_1e5 = model_discr_1e5.to(device)
model_discr_1e5.eval()

print("Loaded best discriminative LR 1e-5 model")
print("Best epoch:", checkpoint_discr_1e5["epoch"])
print("Best val loss:", checkpoint_discr_1e5["best_val_loss"])

Loaded best discriminative LR 1e-5 model
Best epoch: 25
Best val loss: 0.1465833192841561


In [55]:
y_true_discr_1e5, y_pred_discr_1e5 = evaluate_model(
    model_discr_1e5,
    test_loader,
    y_scaler,
    device
)

print("y_true_discr_1e5 shape:", y_true_discr_1e5.shape)
print("y_pred_discr_1e5 shape:", y_pred_discr_1e5.shape)

print("y_true min/max:", y_true_discr_1e5.min(), y_true_discr_1e5.max())
print("y_pred min/max:", y_pred_discr_1e5.min(), y_pred_discr_1e5.max())

y_true_discr_1e5 shape: (5064, 625)
y_pred_discr_1e5 shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 35.009678 185.86719


In [56]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_discr_1e5, dbp_true_discr_1e5 = [], []
sbp_pred_discr_1e5, dbp_pred_discr_1e5 = [], []

for true_sig, pred_sig in zip(y_true_discr_1e5, y_pred_discr_1e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_discr_1e5.append(sbp_t)
    dbp_true_discr_1e5.append(dbp_t)
    sbp_pred_discr_1e5.append(sbp_p)
    dbp_pred_discr_1e5.append(dbp_p)

sbp_true_discr_1e5 = np.array(sbp_true_discr_1e5)
dbp_true_discr_1e5 = np.array(dbp_true_discr_1e5)
sbp_pred_discr_1e5 = np.array(sbp_pred_discr_1e5)
dbp_pred_discr_1e5 = np.array(dbp_pred_discr_1e5)

sbp_mae_discr_1e5 = mean_absolute_error(sbp_true_discr_1e5, sbp_pred_discr_1e5)
dbp_mae_discr_1e5 = mean_absolute_error(dbp_true_discr_1e5, dbp_pred_discr_1e5)

sbp_rmse_discr_1e5 = np.sqrt(mean_squared_error(sbp_true_discr_1e5, sbp_pred_discr_1e5))
dbp_rmse_discr_1e5 = np.sqrt(mean_squared_error(dbp_true_discr_1e5, dbp_pred_discr_1e5))

sbp_r2_discr_1e5 = r2_score(sbp_true_discr_1e5, sbp_pred_discr_1e5)
dbp_r2_discr_1e5 = r2_score(dbp_true_discr_1e5, dbp_pred_discr_1e5)

avg_mae_discr_1e5 = (sbp_mae_discr_1e5 + dbp_mae_discr_1e5) / 2

print("BENDR Full FT + Discriminative LR 1e-5 Results — Real mmHg Scale")
print("================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_discr_1e5:.3f}")
print(f"RMSE : {sbp_rmse_discr_1e5:.3f}")
print(f"R2   : {sbp_r2_discr_1e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_discr_1e5:.3f}")
print(f"RMSE : {dbp_rmse_discr_1e5:.3f}")
print(f"R2   : {dbp_r2_discr_1e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_discr_1e5:.2f}/{dbp_mae_discr_1e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_discr_1e5:.3f}")

BENDR Full FT + Discriminative LR 1e-5 Results — Real mmHg Scale
SBP
MAE  : 5.925
RMSE : 8.004
R2   : 0.6837

DBP
MAE  : 4.534
RMSE : 6.109
R2   : 0.5036

Table format:
5.93/4.53

Average MAE:
5.230


In [57]:
metrics_discr_1e5 = {
    "model": "BENDR",
    "method": tuning_method_discr_1e5,
    "best_epoch": int(checkpoint_discr_1e5["epoch"]),
    "best_val_loss": float(checkpoint_discr_1e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_discr_1e5),
    "DBP_MAE": float(dbp_mae_discr_1e5),
    "AVG_MAE": float(avg_mae_discr_1e5),
    "SBP_RMSE": float(sbp_rmse_discr_1e5),
    "DBP_RMSE": float(dbp_rmse_discr_1e5),
    "SBP_R2": float(sbp_r2_discr_1e5),
    "DBP_R2": float(dbp_r2_discr_1e5),
    "table_format": f"{sbp_mae_discr_1e5:.2f}/{dbp_mae_discr_1e5:.2f}",
    "backbone_lr": float(checkpoint_discr_1e5["backbone_lr"]),
    "head_lr": float(checkpoint_discr_1e5["head_lr"]),
    "optimizer": checkpoint_discr_1e5["optimizer"],
    "trainable_parameters": int(checkpoint_discr_1e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_discr_1e5["total_parameters"]),
    "seed": int(checkpoint_discr_1e5["seed"])
}

with open(metrics_path_discr_1e5, "w") as f:
    json.dump(metrics_discr_1e5, f, indent=4)

metrics_discr_1e5

{'model': 'BENDR',
 'method': 'full_ft_discriminative_lr_1e-5',
 'best_epoch': 25,
 'best_val_loss': 0.1465833192841561,
 'SBP_MAE': 5.925294399261475,
 'DBP_MAE': 4.5338239669799805,
 'AVG_MAE': 5.2295591831207275,
 'SBP_RMSE': 8.004043987498356,
 'DBP_RMSE': 6.109243250057973,
 'SBP_R2': 0.6837178468704224,
 'DBP_R2': 0.5035704374313354,
 'table_format': '5.93/4.53',
 'backbone_lr': 1e-05,
 'head_lr': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# Full FT + discr LR
Backbone LR = 2e-5
Head LR = 5e-5

In [58]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_discr_2e5_head5e5 = "full_ft_discriminative_lr_backbone_2e-5_head_5e-5"

best_model_path_discr_2e5_head5e5 = "bendr_full_ft_discriminative_lr_backbone_2e-5_head_5e-5_m4_seed42.pth"
history_path_discr_2e5_head5e5 = "history_bendr_full_ft_discriminative_lr_backbone_2e-5_head_5e-5_m4_seed42.csv"
metrics_path_discr_2e5_head5e5 = "metrics_bendr_full_ft_discriminative_lr_backbone_2e-5_head_5e-5_m4_seed42.json"

model_discr_2e5_head5e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_discr_2e5_head5e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_discr_2e5_head5e5(x_dummy)

criterion_discr_2e5_head5e5 = nn.MSELoss()

print("Setup ready")

Device: cuda
Setup ready


In [59]:
for param in model_discr_2e5_head5e5.parameters():
    param.requires_grad = True

total_params_discr_2e5_head5e5 = sum(p.numel() for p in model_discr_2e5_head5e5.parameters())
trainable_params_discr_2e5_head5e5 = sum(p.numel() for p in model_discr_2e5_head5e5.parameters() if p.requires_grad)

print("Total parameters:", total_params_discr_2e5_head5e5)
print("Trainable parameters:", trainable_params_discr_2e5_head5e5)
print("Trainable percentage:", 100 * trainable_params_discr_2e5_head5e5 / total_params_discr_2e5_head5e5)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [60]:
backbone_params = []
head_params = []

for name, param in model_discr_2e5_head5e5.named_parameters():
    if not param.requires_grad:
        continue

    if "head" in name.lower() or "regressor" in name.lower() or "regression" in name.lower() or "final" in name.lower() or "output" in name.lower():
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_discr_2e5_head5e5 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 5e-5, "weight_decay": 0.01}
    ]
)

history_discr_2e5_head5e5 = []

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Optimizer ready")

Backbone params: 157042402
Head params: 928594
Optimizer ready


In [61]:
import time
import pandas as pd
import torch

num_epochs_discr_2e5_head5e5 = 30
best_val_loss_discr_2e5_head5e5 = float("inf")

start_time_discr_2e5_head5e5 = time.time()

for epoch in range(num_epochs_discr_2e5_head5e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_discr_2e5_head5e5,
        train_loader,
        optimizer_discr_2e5_head5e5,
        criterion_discr_2e5_head5e5,
        device
    )

    val_loss = validate_one_epoch(
        model_discr_2e5_head5e5,
        val_loader,
        criterion_discr_2e5_head5e5,
        device
    )

    history_discr_2e5_head5e5.append({
        "method": tuning_method_discr_2e5_head5e5,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_discr_2e5_head5e5),
        "total_parameters": int(total_params_discr_2e5_head5e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_discr_2e5_head5e5:
        best_val_loss_discr_2e5_head5e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_discr_2e5_head5e5,
            "model_state_dict": model_discr_2e5_head5e5.state_dict(),
            "optimizer_state_dict": optimizer_discr_2e5_head5e5.state_dict(),
            "best_val_loss": float(best_val_loss_discr_2e5_head5e5),
            "backbone_lr": 2e-5,
            "head_lr": 5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_discr_2e5_head5e5),
            "total_parameters": int(total_params_discr_2e5_head5e5),
            "seed": SEED
        }, best_model_path_discr_2e5_head5e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_discr_2e5_head5e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_discr_2e5_head5e5).to_csv(history_path_discr_2e5_head5e5, index=False)

print("Best val loss:", best_val_loss_discr_2e5_head5e5)
print("Total training time:", time.time() - start_time_discr_2e5_head5e5)

Epoch [1/30] | Train Loss: 0.828155 | Val Loss: 0.658851 | Time: 14.25s saved
Epoch [2/30] | Train Loss: 0.567782 | Val Loss: 0.514583 | Time: 14.84s saved
Epoch [3/30] | Train Loss: 0.469142 | Val Loss: 0.441164 | Time: 15.06s saved
Epoch [4/30] | Train Loss: 0.407009 | Val Loss: 0.389608 | Time: 15.35s saved
Epoch [5/30] | Train Loss: 0.353369 | Val Loss: 0.338679 | Time: 15.36s saved
Epoch [6/30] | Train Loss: 0.313562 | Val Loss: 0.307889 | Time: 14.98s saved
Epoch [7/30] | Train Loss: 0.278854 | Val Loss: 0.273900 | Time: 15.01s saved
Epoch [8/30] | Train Loss: 0.251155 | Val Loss: 0.255737 | Time: 15.08s saved
Epoch [9/30] | Train Loss: 0.229224 | Val Loss: 0.233316 | Time: 14.93s saved
Epoch [10/30] | Train Loss: 0.209702 | Val Loss: 0.214384 | Time: 15.18s saved
Epoch [11/30] | Train Loss: 0.193612 | Val Loss: 0.210780 | Time: 15.18s saved
Epoch [12/30] | Train Loss: 0.182290 | Val Loss: 0.213027 | Time: 13.95s 
Epoch [13/30] | Train Loss: 0.172727 | Val Loss: 0.192123 | Time: 

In [62]:
checkpoint_discr_2e5_head5e5 = torch.load(best_model_path_discr_2e5_head5e5, map_location=device)

model_discr_2e5_head5e5.load_state_dict(checkpoint_discr_2e5_head5e5["model_state_dict"])
model_discr_2e5_head5e5 = model_discr_2e5_head5e5.to(device)
model_discr_2e5_head5e5.eval()

print("Loaded best model")
print("Best epoch:", checkpoint_discr_2e5_head5e5["epoch"])
print("Best val loss:", checkpoint_discr_2e5_head5e5["best_val_loss"])

Loaded best model
Best epoch: 28
Best val loss: 0.15089301508581218


In [63]:
y_true_discr_2e5_head5e5, y_pred_discr_2e5_head5e5 = evaluate_model(
    model_discr_2e5_head5e5,
    test_loader,
    y_scaler,
    device
)

print("y_true shape:", y_true_discr_2e5_head5e5.shape)
print("y_pred shape:", y_pred_discr_2e5_head5e5.shape)

print("y_true min/max:", y_true_discr_2e5_head5e5.min(), y_true_discr_2e5_head5e5.max())
print("y_pred min/max:", y_pred_discr_2e5_head5e5.min(), y_pred_discr_2e5_head5e5.max())

y_true shape: (5064, 625)
y_pred shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 28.719973 188.23521


In [64]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_discr_2e5_head5e5, dbp_true_discr_2e5_head5e5 = [], []
sbp_pred_discr_2e5_head5e5, dbp_pred_discr_2e5_head5e5 = [], []

for true_sig, pred_sig in zip(y_true_discr_2e5_head5e5, y_pred_discr_2e5_head5e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_discr_2e5_head5e5.append(sbp_t)
    dbp_true_discr_2e5_head5e5.append(dbp_t)
    sbp_pred_discr_2e5_head5e5.append(sbp_p)
    dbp_pred_discr_2e5_head5e5.append(dbp_p)

sbp_true_discr_2e5_head5e5 = np.array(sbp_true_discr_2e5_head5e5)
dbp_true_discr_2e5_head5e5 = np.array(dbp_true_discr_2e5_head5e5)
sbp_pred_discr_2e5_head5e5 = np.array(sbp_pred_discr_2e5_head5e5)
dbp_pred_discr_2e5_head5e5 = np.array(dbp_pred_discr_2e5_head5e5)

sbp_mae_discr_2e5_head5e5 = mean_absolute_error(sbp_true_discr_2e5_head5e5, sbp_pred_discr_2e5_head5e5)
dbp_mae_discr_2e5_head5e5 = mean_absolute_error(dbp_true_discr_2e5_head5e5, dbp_pred_discr_2e5_head5e5)

sbp_rmse_discr_2e5_head5e5 = np.sqrt(mean_squared_error(sbp_true_discr_2e5_head5e5, sbp_pred_discr_2e5_head5e5))
dbp_rmse_discr_2e5_head5e5 = np.sqrt(mean_squared_error(dbp_true_discr_2e5_head5e5, dbp_pred_discr_2e5_head5e5))

sbp_r2_discr_2e5_head5e5 = r2_score(sbp_true_discr_2e5_head5e5, sbp_pred_discr_2e5_head5e5)
dbp_r2_discr_2e5_head5e5 = r2_score(dbp_true_discr_2e5_head5e5, dbp_pred_discr_2e5_head5e5)

avg_mae_discr_2e5_head5e5 = (sbp_mae_discr_2e5_head5e5 + dbp_mae_discr_2e5_head5e5) / 2

print("BENDR Full FT + Discriminative LR Backbone 2e-5 Head 5e-5 Results — Real mmHg Scale")
print("===================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_discr_2e5_head5e5:.3f}")
print(f"RMSE : {sbp_rmse_discr_2e5_head5e5:.3f}")
print(f"R2   : {sbp_r2_discr_2e5_head5e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_discr_2e5_head5e5:.3f}")
print(f"RMSE : {dbp_rmse_discr_2e5_head5e5:.3f}")
print(f"R2   : {dbp_r2_discr_2e5_head5e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_discr_2e5_head5e5:.2f}/{dbp_mae_discr_2e5_head5e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_discr_2e5_head5e5:.3f}")

BENDR Full FT + Discriminative LR Backbone 2e-5 Head 5e-5 Results — Real mmHg Scale
SBP
MAE  : 5.677
RMSE : 7.834
R2   : 0.6970

DBP
MAE  : 4.504
RMSE : 6.050
R2   : 0.5132

Table format:
5.68/4.50

Average MAE:
5.091


In [65]:
metrics_discr_2e5_head5e5 = {
    "model": "BENDR",
    "method": tuning_method_discr_2e5_head5e5,
    "best_epoch": int(checkpoint_discr_2e5_head5e5["epoch"]),
    "best_val_loss": float(checkpoint_discr_2e5_head5e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_discr_2e5_head5e5),
    "DBP_MAE": float(dbp_mae_discr_2e5_head5e5),
    "AVG_MAE": float(avg_mae_discr_2e5_head5e5),
    "SBP_RMSE": float(sbp_rmse_discr_2e5_head5e5),
    "DBP_RMSE": float(dbp_rmse_discr_2e5_head5e5),
    "SBP_R2": float(sbp_r2_discr_2e5_head5e5),
    "DBP_R2": float(dbp_r2_discr_2e5_head5e5),
    "table_format": f"{sbp_mae_discr_2e5_head5e5:.2f}/{dbp_mae_discr_2e5_head5e5:.2f}",
    "backbone_lr": float(checkpoint_discr_2e5_head5e5["backbone_lr"]),
    "head_lr": float(checkpoint_discr_2e5_head5e5["head_lr"]),
    "optimizer": checkpoint_discr_2e5_head5e5["optimizer"],
    "trainable_parameters": int(checkpoint_discr_2e5_head5e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_discr_2e5_head5e5["total_parameters"]),
    "seed": int(checkpoint_discr_2e5_head5e5["seed"])
}

with open(metrics_path_discr_2e5_head5e5, "w") as f:
    json.dump(metrics_discr_2e5_head5e5, f, indent=4)

metrics_discr_2e5_head5e5

{'model': 'BENDR',
 'method': 'full_ft_discriminative_lr_backbone_2e-5_head_5e-5',
 'best_epoch': 28,
 'best_val_loss': 0.15089301508581218,
 'SBP_MAE': 5.6773481369018555,
 'DBP_MAE': 4.504167556762695,
 'AVG_MAE': 5.090757846832275,
 'SBP_RMSE': 7.834196191975615,
 'DBP_RMSE': 6.049972962878636,
 'SBP_R2': 0.6969985961914062,
 'DBP_R2': 0.5131561160087585,
 'table_format': '5.68/4.50',
 'backbone_lr': 2e-05,
 'head_lr': 5e-05,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# Head LR = 7.5e-5 Backbone LR = 2e-5


In [66]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_discr_2e5_head75e5 = "full_ft_discriminative_lr_backbone_2e-5_head_7.5e-5"

best_model_path_discr_2e5_head75e5 = "bendr_full_ft_discriminative_lr_backbone_2e-5_head_7.5e-5_m4_seed42.pth"
history_path_discr_2e5_head75e5 = "history_bendr_full_ft_discriminative_lr_backbone_2e-5_head_7.5e-5_m4_seed42.csv"
metrics_path_discr_2e5_head75e5 = "metrics_bendr_full_ft_discriminative_lr_backbone_2e-5_head_7.5e-5_m4_seed42.json"

model_discr_2e5_head75e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_discr_2e5_head75e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_discr_2e5_head75e5(x_dummy)

criterion_discr_2e5_head75e5 = nn.MSELoss()

print("Setup ready")

Device: cuda
Setup ready


In [67]:
for param in model_discr_2e5_head75e5.parameters():
    param.requires_grad = True

total_params_discr_2e5_head75e5 = sum(p.numel() for p in model_discr_2e5_head75e5.parameters())
trainable_params_discr_2e5_head75e5 = sum(p.numel() for p in model_discr_2e5_head75e5.parameters() if p.requires_grad)

print("Total parameters:", total_params_discr_2e5_head75e5)
print("Trainable parameters:", trainable_params_discr_2e5_head75e5)
print("Trainable percentage:", 100 * trainable_params_discr_2e5_head75e5 / total_params_discr_2e5_head75e5)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [68]:
backbone_params = []
head_params = []

for name, param in model_discr_2e5_head75e5.named_parameters():
    if not param.requires_grad:
        continue

    if "head" in name.lower() or "regressor" in name.lower() or "regression" in name.lower() or "final" in name.lower() or "output" in name.lower():
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_discr_2e5_head75e5 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 7.5e-5, "weight_decay": 0.01}
    ]
)

history_discr_2e5_head75e5 = []

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Optimizer ready")

Backbone params: 157042402
Head params: 928594
Optimizer ready


In [69]:
import time
import pandas as pd
import torch

num_epochs_discr_2e5_head75e5 = 30
best_val_loss_discr_2e5_head75e5 = float("inf")

start_time_discr_2e5_head75e5 = time.time()

for epoch in range(num_epochs_discr_2e5_head75e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_discr_2e5_head75e5,
        train_loader,
        optimizer_discr_2e5_head75e5,
        criterion_discr_2e5_head75e5,
        device
    )

    val_loss = validate_one_epoch(
        model_discr_2e5_head75e5,
        val_loader,
        criterion_discr_2e5_head75e5,
        device
    )

    history_discr_2e5_head75e5.append({
        "method": tuning_method_discr_2e5_head75e5,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 7.5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_discr_2e5_head75e5),
        "total_parameters": int(total_params_discr_2e5_head75e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_discr_2e5_head75e5:
        best_val_loss_discr_2e5_head75e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_discr_2e5_head75e5,
            "model_state_dict": model_discr_2e5_head75e5.state_dict(),
            "optimizer_state_dict": optimizer_discr_2e5_head75e5.state_dict(),
            "best_val_loss": float(best_val_loss_discr_2e5_head75e5),
            "backbone_lr": 2e-5,
            "head_lr": 7.5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_discr_2e5_head75e5),
            "total_parameters": int(total_params_discr_2e5_head75e5),
            "seed": SEED
        }, best_model_path_discr_2e5_head75e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_discr_2e5_head75e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_discr_2e5_head75e5).to_csv(history_path_discr_2e5_head75e5, index=False)

print("Best val loss:", best_val_loss_discr_2e5_head75e5)
print("Total training time:", time.time() - start_time_discr_2e5_head75e5)

Epoch [1/30] | Train Loss: 0.799031 | Val Loss: 0.596259 | Time: 14.20s saved
Epoch [2/30] | Train Loss: 0.514558 | Val Loss: 0.458503 | Time: 14.75s saved
Epoch [3/30] | Train Loss: 0.418333 | Val Loss: 0.387807 | Time: 15.10s saved
Epoch [4/30] | Train Loss: 0.355331 | Val Loss: 0.345035 | Time: 15.23s saved
Epoch [5/30] | Train Loss: 0.307100 | Val Loss: 0.295541 | Time: 15.35s saved
Epoch [6/30] | Train Loss: 0.266508 | Val Loss: 0.267839 | Time: 15.00s saved
Epoch [7/30] | Train Loss: 0.236555 | Val Loss: 0.238807 | Time: 15.02s saved
Epoch [8/30] | Train Loss: 0.211881 | Val Loss: 0.242149 | Time: 13.80s 
Epoch [9/30] | Train Loss: 0.195219 | Val Loss: 0.210905 | Time: 14.97s saved
Epoch [10/30] | Train Loss: 0.181299 | Val Loss: 0.200607 | Time: 15.07s saved
Epoch [11/30] | Train Loss: 0.169939 | Val Loss: 0.190588 | Time: 15.18s saved
Epoch [12/30] | Train Loss: 0.160025 | Val Loss: 0.191449 | Time: 13.99s 
Epoch [13/30] | Train Loss: 0.152700 | Val Loss: 0.176114 | Time: 15.19

In [70]:
checkpoint_discr_2e5_head75e5 = torch.load(best_model_path_discr_2e5_head75e5, map_location=device)

model_discr_2e5_head75e5.load_state_dict(checkpoint_discr_2e5_head75e5["model_state_dict"])
model_discr_2e5_head75e5 = model_discr_2e5_head75e5.to(device)
model_discr_2e5_head75e5.eval()

print("Loaded best model")
print("Best epoch:", checkpoint_discr_2e5_head75e5["epoch"])
print("Best val loss:", checkpoint_discr_2e5_head75e5["best_val_loss"])

Loaded best model
Best epoch: 27
Best val loss: 0.1468197045426138


In [71]:
y_true_discr_2e5_head75e5, y_pred_discr_2e5_head75e5 = evaluate_model(
    model_discr_2e5_head75e5,
    test_loader,
    y_scaler,
    device
)

print("y_true shape:", y_true_discr_2e5_head75e5.shape)
print("y_pred shape:", y_pred_discr_2e5_head75e5.shape)

print("y_true min/max:", y_true_discr_2e5_head75e5.min(), y_true_discr_2e5_head75e5.max())
print("y_pred min/max:", y_pred_discr_2e5_head75e5.min(), y_pred_discr_2e5_head75e5.max())

y_true shape: (5064, 625)
y_pred shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 36.64436 185.30357


In [72]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_discr_2e5_head75e5, dbp_true_discr_2e5_head75e5 = [], []
sbp_pred_discr_2e5_head75e5, dbp_pred_discr_2e5_head75e5 = [], []

for true_sig, pred_sig in zip(y_true_discr_2e5_head75e5, y_pred_discr_2e5_head75e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_discr_2e5_head75e5.append(sbp_t)
    dbp_true_discr_2e5_head75e5.append(dbp_t)
    sbp_pred_discr_2e5_head75e5.append(sbp_p)
    dbp_pred_discr_2e5_head75e5.append(dbp_p)

sbp_true_discr_2e5_head75e5 = np.array(sbp_true_discr_2e5_head75e5)
dbp_true_discr_2e5_head75e5 = np.array(dbp_true_discr_2e5_head75e5)
sbp_pred_discr_2e5_head75e5 = np.array(sbp_pred_discr_2e5_head75e5)
dbp_pred_discr_2e5_head75e5 = np.array(dbp_pred_discr_2e5_head75e5)

sbp_mae_discr_2e5_head75e5 = mean_absolute_error(sbp_true_discr_2e5_head75e5, sbp_pred_discr_2e5_head75e5)
dbp_mae_discr_2e5_head75e5 = mean_absolute_error(dbp_true_discr_2e5_head75e5, dbp_pred_discr_2e5_head75e5)

sbp_rmse_discr_2e5_head75e5 = np.sqrt(mean_squared_error(sbp_true_discr_2e5_head75e5, sbp_pred_discr_2e5_head75e5))
dbp_rmse_discr_2e5_head75e5 = np.sqrt(mean_squared_error(dbp_true_discr_2e5_head75e5, dbp_pred_discr_2e5_head75e5))

sbp_r2_discr_2e5_head75e5 = r2_score(sbp_true_discr_2e5_head75e5, sbp_pred_discr_2e5_head75e5)
dbp_r2_discr_2e5_head75e5 = r2_score(dbp_true_discr_2e5_head75e5, dbp_pred_discr_2e5_head75e5)

avg_mae_discr_2e5_head75e5 = (sbp_mae_discr_2e5_head75e5 + dbp_mae_discr_2e5_head75e5) / 2

print("BENDR Full FT + Discriminative LR Backbone 2e-5 Head 7.5e-5 Results — Real mmHg Scale")
print("=====================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_discr_2e5_head75e5:.3f}")
print(f"RMSE : {sbp_rmse_discr_2e5_head75e5:.3f}")
print(f"R2   : {sbp_r2_discr_2e5_head75e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_discr_2e5_head75e5:.3f}")
print(f"RMSE : {dbp_rmse_discr_2e5_head75e5:.3f}")
print(f"R2   : {dbp_r2_discr_2e5_head75e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_discr_2e5_head75e5:.2f}/{dbp_mae_discr_2e5_head75e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_discr_2e5_head75e5:.3f}")

BENDR Full FT + Discriminative LR Backbone 2e-5 Head 7.5e-5 Results — Real mmHg Scale
SBP
MAE  : 6.235
RMSE : 8.576
R2   : 0.6369

DBP
MAE  : 4.893
RMSE : 6.519
R2   : 0.4347

Table format:
6.24/4.89

Average MAE:
5.564


In [ ]:
metrics_discr_2e5_head75e5 = {
    "model": "BENDR",
    "method": tuning_method_discr_2e5_head75e5,
    "best_epoch": int(checkpoint_discr_2e5_head75e5["epoch"]),
    "best_val_loss": float(checkpoint_discr_2e5_head75e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_discr_2e5_head75e5),
    "DBP_MAE": float(dbp_mae_discr_2e5_head75e5),
    "AVG_MAE": float(avg_mae_discr_2e5_head75e5),
    "SBP_RMSE": float(sbp_rmse_discr_2e5_head75e5),
    "DBP_RMSE": float(dbp_rmse_discr_2e5_head75e5),
    "SBP_R2": float(sbp_r2_discr_2e5_head75e5),
    "DBP_R2": float(dbp_r2_discr_2e5_head75e5),
    "table_format": f"{sbp_mae_discr_2e5_head75e5:.2f}/{dbp_mae_discr_2e5_head75e5:.2f}",
    "backbone_lr": float(checkpoint_discr_2e5_head75e5["backbone_lr"]),
    "head_lr": float(checkpoint_discr_2e5_head75e5["head_lr"]),
    "optimizer": checkpoint_discr_2e5_head75e5["optimizer"],
    "trainable_parameters": int(checkpoint_discr_2e5_head75e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_discr_2e5_head75e5["total_parameters"]),
    "seed": int(checkpoint_discr_2e5_head75e5["seed"])
}

with open(metrics_path_discr_2e5_head75e5, "w") as f:
    json.dump(metrics_discr_2e5_head75e5, f, indent=4)

metrics_discr_2e5_head75e5

Backbone LR = 2e-5
# Head LR = 2e-4

In [73]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_discr_2e5_head2e4 = "full_ft_discriminative_lr_backbone_2e-5_head_2e-4"

best_model_path_discr_2e5_head2e4 = "bendr_full_ft_discriminative_lr_backbone_2e-5_head_2e-4_m4_seed42.pth"
history_path_discr_2e5_head2e4 = "history_bendr_full_ft_discriminative_lr_backbone_2e-5_head_2e-4_m4_seed42.csv"
metrics_path_discr_2e5_head2e4 = "metrics_bendr_full_ft_discriminative_lr_backbone_2e-5_head_2e-4_m4_seed42.json"

model_discr_2e5_head2e4 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_discr_2e5_head2e4.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_discr_2e5_head2e4(x_dummy)

criterion_discr_2e5_head2e4 = nn.MSELoss()

print("Setup ready")

Device: cuda
Setup ready


In [74]:
for param in model_discr_2e5_head2e4.parameters():
    param.requires_grad = True

total_params_discr_2e5_head2e4 = sum(p.numel() for p in model_discr_2e5_head2e4.parameters())
trainable_params_discr_2e5_head2e4 = sum(p.numel() for p in model_discr_2e5_head2e4.parameters() if p.requires_grad)

print("Total parameters:", total_params_discr_2e5_head2e4)
print("Trainable parameters:", trainable_params_discr_2e5_head2e4)
print("Trainable percentage:", 100 * trainable_params_discr_2e5_head2e4 / total_params_discr_2e5_head2e4)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [75]:
backbone_params = []
head_params = []

for name, param in model_discr_2e5_head2e4.named_parameters():
    if not param.requires_grad:
        continue

    if "head" in name.lower() or "regressor" in name.lower() or "regression" in name.lower() or "final" in name.lower() or "output" in name.lower():
        head_params.append(param)
    else:
        backbone_params.append(param)

optimizer_discr_2e5_head2e4 = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-5, "weight_decay": 0.01},
        {"params": head_params, "lr": 2e-4, "weight_decay": 0.01}
    ]
)

history_discr_2e5_head2e4 = []

print("Backbone params:", sum(p.numel() for p in backbone_params))
print("Head params:", sum(p.numel() for p in head_params))
print("Optimizer ready")

Backbone params: 157042402
Head params: 928594
Optimizer ready


In [76]:
import time
import pandas as pd
import torch

num_epochs_discr_2e5_head2e4 = 30
best_val_loss_discr_2e5_head2e4 = float("inf")

start_time_discr_2e5_head2e4 = time.time()

for epoch in range(num_epochs_discr_2e5_head2e4):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_discr_2e5_head2e4,
        train_loader,
        optimizer_discr_2e5_head2e4,
        criterion_discr_2e5_head2e4,
        device
    )

    val_loss = validate_one_epoch(
        model_discr_2e5_head2e4,
        val_loader,
        criterion_discr_2e5_head2e4,
        device
    )

    history_discr_2e5_head2e4.append({
        "method": tuning_method_discr_2e5_head2e4,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 2e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_discr_2e5_head2e4),
        "total_parameters": int(total_params_discr_2e5_head2e4),
        "seed": SEED
    })

    if val_loss < best_val_loss_discr_2e5_head2e4:
        best_val_loss_discr_2e5_head2e4 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_discr_2e5_head2e4,
            "model_state_dict": model_discr_2e5_head2e4.state_dict(),
            "optimizer_state_dict": optimizer_discr_2e5_head2e4.state_dict(),
            "best_val_loss": float(best_val_loss_discr_2e5_head2e4),
            "backbone_lr": 2e-5,
            "head_lr": 2e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_discr_2e5_head2e4),
            "total_parameters": int(total_params_discr_2e5_head2e4),
            "seed": SEED
        }, best_model_path_discr_2e5_head2e4)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_discr_2e5_head2e4}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_discr_2e5_head2e4).to_csv(history_path_discr_2e5_head2e4, index=False)

print("Best val loss:", best_val_loss_discr_2e5_head2e4)
print("Total training time:", time.time() - start_time_discr_2e5_head2e4)

Epoch [1/30] | Train Loss: 0.722520 | Val Loss: 0.527684 | Time: 14.28s saved
Epoch [2/30] | Train Loss: 0.449538 | Val Loss: 0.400214 | Time: 14.89s saved
Epoch [3/30] | Train Loss: 0.362948 | Val Loss: 0.331945 | Time: 15.09s saved
Epoch [4/30] | Train Loss: 0.310171 | Val Loss: 0.293646 | Time: 15.37s saved
Epoch [5/30] | Train Loss: 0.265306 | Val Loss: 0.253085 | Time: 15.42s saved
Epoch [6/30] | Train Loss: 0.229782 | Val Loss: 0.227381 | Time: 15.17s saved
Epoch [7/30] | Train Loss: 0.205319 | Val Loss: 0.211303 | Time: 14.77s saved
Epoch [8/30] | Train Loss: 0.188424 | Val Loss: 0.207619 | Time: 14.96s saved
Epoch [9/30] | Train Loss: 0.174794 | Val Loss: 0.190785 | Time: 15.03s saved
Epoch [10/30] | Train Loss: 0.162094 | Val Loss: 0.178967 | Time: 15.13s saved
Epoch [11/30] | Train Loss: 0.152675 | Val Loss: 0.180307 | Time: 13.96s 
Epoch [12/30] | Train Loss: 0.147237 | Val Loss: 0.177777 | Time: 15.14s saved
Epoch [13/30] | Train Loss: 0.141419 | Val Loss: 0.167940 | Time: 

In [77]:
checkpoint_discr_2e5_head2e4 = torch.load(best_model_path_discr_2e5_head2e4, map_location=device)

model_discr_2e5_head2e4.load_state_dict(checkpoint_discr_2e5_head2e4["model_state_dict"])
model_discr_2e5_head2e4 = model_discr_2e5_head2e4.to(device)
model_discr_2e5_head2e4.eval()

print("Loaded best model")
print("Best epoch:", checkpoint_discr_2e5_head2e4["epoch"])
print("Best val loss:", checkpoint_discr_2e5_head2e4["best_val_loss"])

Loaded best model
Best epoch: 26
Best val loss: 0.1409334822977893


In [78]:
y_true_discr_2e5_head2e4, y_pred_discr_2e5_head2e4 = evaluate_model(
    model_discr_2e5_head2e4,
    test_loader,
    y_scaler,
    device
)

print("y_true shape:", y_true_discr_2e5_head2e4.shape)
print("y_pred shape:", y_pred_discr_2e5_head2e4.shape)

print("y_true min/max:", y_true_discr_2e5_head2e4.min(), y_true_discr_2e5_head2e4.max())
print("y_pred min/max:", y_pred_discr_2e5_head2e4.min(), y_pred_discr_2e5_head2e4.max())

y_true shape: (5064, 625)
y_pred shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 40.416965 190.2308


In [79]:
sbp_true_discr_2e5_head2e4, dbp_true_discr_2e5_head2e4 = [], []
sbp_pred_discr_2e5_head2e4, dbp_pred_discr_2e5_head2e4 = [], []

for true_sig, pred_sig in zip(y_true_discr_2e5_head2e4, y_pred_discr_2e5_head2e4):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_discr_2e5_head2e4.append(sbp_t)
    dbp_true_discr_2e5_head2e4.append(dbp_t)
    sbp_pred_discr_2e5_head2e4.append(sbp_p)
    dbp_pred_discr_2e5_head2e4.append(dbp_p)

sbp_true_discr_2e5_head2e4 = np.array(sbp_true_discr_2e5_head2e4)
dbp_true_discr_2e5_head2e4 = np.array(dbp_true_discr_2e5_head2e4)
sbp_pred_discr_2e5_head2e4 = np.array(sbp_pred_discr_2e5_head2e4)
dbp_pred_discr_2e5_head2e4 = np.array(dbp_pred_discr_2e5_head2e4)

sbp_mae_discr_2e5_head2e4 = mean_absolute_error(sbp_true_discr_2e5_head2e4, sbp_pred_discr_2e5_head2e4)
dbp_mae_discr_2e5_head2e4 = mean_absolute_error(dbp_true_discr_2e5_head2e4, dbp_pred_discr_2e5_head2e4)

sbp_rmse_discr_2e5_head2e4 = np.sqrt(mean_squared_error(sbp_true_discr_2e5_head2e4, sbp_pred_discr_2e5_head2e4))
dbp_rmse_discr_2e5_head2e4 = np.sqrt(mean_squared_error(dbp_true_discr_2e5_head2e4, dbp_pred_discr_2e5_head2e4))

sbp_r2_discr_2e5_head2e4 = r2_score(sbp_true_discr_2e5_head2e4, sbp_pred_discr_2e5_head2e4)
dbp_r2_discr_2e5_head2e4 = r2_score(dbp_true_discr_2e5_head2e4, dbp_pred_discr_2e5_head2e4)

avg_mae_discr_2e5_head2e4 = (sbp_mae_discr_2e5_head2e4 + dbp_mae_discr_2e5_head2e4) / 2

print("BENDR Full FT + Discriminative LR Backbone 2e-5 Head 2e-4 Results — Real mmHg Scale")
print("===================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_discr_2e5_head2e4:.3f}")
print(f"RMSE : {sbp_rmse_discr_2e5_head2e4:.3f}")
print(f"R2   : {sbp_r2_discr_2e5_head2e4:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_discr_2e5_head2e4:.3f}")
print(f"RMSE : {dbp_rmse_discr_2e5_head2e4:.3f}")
print(f"R2   : {dbp_r2_discr_2e5_head2e4:.4f}")

print("\nTable format:")
print(f"{sbp_mae_discr_2e5_head2e4:.2f}/{dbp_mae_discr_2e5_head2e4:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_discr_2e5_head2e4:.3f}")

BENDR Full FT + Discriminative LR Backbone 2e-5 Head 2e-4 Results — Real mmHg Scale
SBP
MAE  : 5.785
RMSE : 8.002
R2   : 0.6839

DBP
MAE  : 5.185
RMSE : 7.010
R2   : 0.3465

Table format:
5.79/5.19

Average MAE:
5.485


# layer-wise LR decay

In [80]:
# =========================
# Train / Validation helper functions
# =========================

def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()

        pred = model(xb)

        if pred.shape != yb.shape:
            pred = pred.view_as(yb)

        loss = criterion(pred, yb)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * xb.size(0)

    return total_loss / len(train_loader.dataset)


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            pred = model(xb)

            if pred.shape != yb.shape:
                pred = pred.view_as(yb)

            loss = criterion(pred, yb)
            total_loss += loss.item() * xb.size(0)

    return total_loss / len(val_loader.dataset)

In [81]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_lwlr = "layerwise_lr_decay_early1e-5_middle1.5e-5_late2e-5_head5e-5"

best_model_path_lwlr = "bendr_layerwise_lr_decay_early1e-5_middle1.5e-5_late2e-5_head5e-5_m4_seed42.pth"
history_path_lwlr = "history_bendr_layerwise_lr_decay_early1e-5_middle1.5e-5_late2e-5_head5e-5_m4_seed42.csv"
metrics_path_lwlr = "metrics_bendr_layerwise_lr_decay_early1e-5_middle1.5e-5_late2e-5_head5e-5_m4_seed42.json"

model_lwlr = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_lwlr.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_lwlr(x_dummy)

criterion_lwlr = nn.MSELoss()

print("Layer-wise LR decay setup ready")

Device: cuda
Layer-wise LR decay setup ready


In [82]:
for param in model_lwlr.parameters():
    param.requires_grad = True

total_params_lwlr = sum(p.numel() for p in model_lwlr.parameters())
trainable_params_lwlr = sum(p.numel() for p in model_lwlr.parameters() if p.requires_grad)

print("Total parameters:", total_params_lwlr)
print("Trainable parameters:", trainable_params_lwlr)
print("Trainable percentage:", 100 * trainable_params_lwlr / total_params_lwlr)

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0


In [83]:
def create_layerwise_lr_decay_optimizer(
    model,
    early_lr=1e-5,
    middle_lr=1.5e-5,
    late_lr=2e-5,
    head_lr=5e-5,
    weight_decay=1e-2
):
    no_decay_keywords = ["bias", "norm", "bn", "layernorm", "layer_norm"]

    early_decay, early_no_decay = [], []
    middle_decay, middle_no_decay = [], []
    late_decay, late_no_decay = [], []
    head_decay, head_no_decay = [], []

    transformer_layers = model.bendr.contextualizer.transformer_layers
    n_layers = len(transformer_layers)

    early_cutoff = n_layers // 3
    middle_cutoff = 2 * n_layers // 3

    print("Total transformer layers:", n_layers)
    print("Early layers: 0 to", early_cutoff - 1)
    print("Middle layers:", early_cutoff, "to", middle_cutoff - 1)
    print("Late layers:", middle_cutoff, "to", n_layers - 1)

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_low = name.lower()
        is_no_decay = any(k in name_low for k in no_decay_keywords)

        is_head = not name.startswith("bendr.")

        if is_head:
            if is_no_decay:
                head_no_decay.append(param)
            else:
                head_decay.append(param)
            continue

        assigned = False

        if "bendr.contextualizer.transformer_layers" in name:
            parts = name.split(".")
            layer_idx = None

            for i, part in enumerate(parts):
                if part == "transformer_layers" and i + 1 < len(parts):
                    try:
                        layer_idx = int(parts[i + 1])
                    except:
                        layer_idx = None
                    break

            if layer_idx is not None:
                if layer_idx < early_cutoff:
                    if is_no_decay:
                        early_no_decay.append(param)
                    else:
                        early_decay.append(param)
                elif layer_idx < middle_cutoff:
                    if is_no_decay:
                        middle_no_decay.append(param)
                    else:
                        middle_decay.append(param)
                else:
                    if is_no_decay:
                        late_no_decay.append(param)
                    else:
                        late_decay.append(param)

                assigned = True

        if not assigned:
            if is_no_decay:
                early_no_decay.append(param)
            else:
                early_decay.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": early_decay, "lr": early_lr, "weight_decay": weight_decay},
            {"params": early_no_decay, "lr": early_lr, "weight_decay": 0.0},
            {"params": middle_decay, "lr": middle_lr, "weight_decay": weight_decay},
            {"params": middle_no_decay, "lr": middle_lr, "weight_decay": 0.0},
            {"params": late_decay, "lr": late_lr, "weight_decay": weight_decay},
            {"params": late_no_decay, "lr": late_lr, "weight_decay": 0.0},
            {"params": head_decay, "lr": head_lr, "weight_decay": weight_decay},
            {"params": head_no_decay, "lr": head_lr, "weight_decay": 0.0},
        ]
    )

    print("\nParameter groups:")
    print("Early decay:", sum(p.numel() for p in early_decay))
    print("Early no-decay:", sum(p.numel() for p in early_no_decay))
    print("Middle decay:", sum(p.numel() for p in middle_decay))
    print("Middle no-decay:", sum(p.numel() for p in middle_no_decay))
    print("Late decay:", sum(p.numel() for p in late_decay))
    print("Late no-decay:", sum(p.numel() for p in late_no_decay))
    print("Head decay:", sum(p.numel() for p in head_decay))
    print("Head no-decay:", sum(p.numel() for p in head_no_decay))

    return optimizer


optimizer_lwlr = create_layerwise_lr_decay_optimizer(
    model_lwlr,
    early_lr=1e-5,
    middle_lr=1.5e-5,
    late_lr=2e-5,
    head_lr=5e-5,
    weight_decay=1e-2
)

history_lwlr = []

Total transformer layers: 8
Early layers: 0 to 1
Middle layers: 2 to 4
Late layers: 5 to 7

Parameter groups:
Early decay: 43723289
Early no-decay: 33801
Middle decay: 56659968
Middle no-decay: 32268
Late decay: 56659968
Late no-decay: 32268
Head decay: 828340
Head no-decay: 1094


In [84]:
import time
import pandas as pd
import torch

num_epochs_lwlr = 30
best_val_loss_lwlr = float("inf")

start_time_lwlr = time.time()

for epoch in range(num_epochs_lwlr):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_lwlr,
        train_loader,
        optimizer_lwlr,
        criterion_lwlr,
        device
    )

    val_loss = validate_one_epoch(
        model_lwlr,
        val_loader,
        criterion_lwlr,
        device
    )

    history_lwlr.append({
        "method": tuning_method_lwlr,
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "early_lr": 1e-5,
        "middle_lr": 1.5e-5,
        "late_lr": 2e-5,
        "head_lr": 5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_lwlr),
        "total_parameters": int(total_params_lwlr),
        "seed": SEED
    })

    if val_loss < best_val_loss_lwlr:
        best_val_loss_lwlr = val_loss

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_lwlr,
            "model_state_dict": model_lwlr.state_dict(),
            "optimizer_state_dict": optimizer_lwlr.state_dict(),
            "best_val_loss": float(best_val_loss_lwlr),
            "early_lr": 1e-5,
            "middle_lr": 1.5e-5,
            "late_lr": 2e-5,
            "head_lr": 5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_lwlr),
            "total_parameters": int(total_params_lwlr),
            "seed": SEED
        }, best_model_path_lwlr)

        status = "saved"
    else:
        status = ""

    print(
        f"Epoch [{epoch+1}/{num_epochs_lwlr}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_lwlr).to_csv(history_path_lwlr, index=False)

print("Best val loss:", best_val_loss_lwlr)
print("Total training time:", time.time() - start_time_lwlr)

Epoch [1/30] | Train Loss: 0.804752 | Val Loss: 0.641462 | Time: 14.10s saved
Epoch [2/30] | Train Loss: 0.562464 | Val Loss: 0.511765 | Time: 14.64s saved
Epoch [3/30] | Train Loss: 0.470129 | Val Loss: 0.460380 | Time: 14.87s saved
Epoch [4/30] | Train Loss: 0.409745 | Val Loss: 0.384570 | Time: 15.14s saved
Epoch [5/30] | Train Loss: 0.360095 | Val Loss: 0.342719 | Time: 15.40s saved
Epoch [6/30] | Train Loss: 0.315008 | Val Loss: 0.301454 | Time: 15.16s saved
Epoch [7/30] | Train Loss: 0.276991 | Val Loss: 0.267497 | Time: 14.98s saved
Epoch [8/30] | Train Loss: 0.248979 | Val Loss: 0.247525 | Time: 14.96s saved
Epoch [9/30] | Train Loss: 0.224832 | Val Loss: 0.231761 | Time: 14.96s saved
Epoch [10/30] | Train Loss: 0.204760 | Val Loss: 0.214164 | Time: 15.11s saved
Epoch [11/30] | Train Loss: 0.190240 | Val Loss: 0.212894 | Time: 15.12s saved
Epoch [12/30] | Train Loss: 0.178947 | Val Loss: 0.199224 | Time: 15.18s saved
Epoch [13/30] | Train Loss: 0.167802 | Val Loss: 0.195443 | T

In [85]:
checkpoint_lwlr = torch.load(best_model_path_lwlr, map_location=device)

model_lwlr.load_state_dict(checkpoint_lwlr["model_state_dict"])
model_lwlr = model_lwlr.to(device)
model_lwlr.eval()

print("Loaded best layer-wise LR decay model")
print("Best epoch:", checkpoint_lwlr["epoch"])
print("Best val loss:", checkpoint_lwlr["best_val_loss"])

Loaded best layer-wise LR decay model
Best epoch: 28
Best val loss: 0.14943655283619992


In [86]:
y_true_lwlr, y_pred_lwlr = evaluate_model(
    model_lwlr,
    test_loader,
    y_scaler,
    device
)

print("y_true_lwlr shape:", y_true_lwlr.shape)
print("y_pred_lwlr shape:", y_pred_lwlr.shape)

print("y_true min/max:", y_true_lwlr.min(), y_true_lwlr.max())
print("y_pred min/max:", y_pred_lwlr.min(), y_pred_lwlr.max())

y_true_lwlr shape: (5064, 625)
y_pred_lwlr shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 29.929632 188.74118


In [87]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_lwlr, dbp_true_lwlr = [], []
sbp_pred_lwlr, dbp_pred_lwlr = [], []

for true_sig, pred_sig in zip(y_true_lwlr, y_pred_lwlr):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_lwlr.append(sbp_t)
    dbp_true_lwlr.append(dbp_t)
    sbp_pred_lwlr.append(sbp_p)
    dbp_pred_lwlr.append(dbp_p)

sbp_true_lwlr = np.array(sbp_true_lwlr)
dbp_true_lwlr = np.array(dbp_true_lwlr)
sbp_pred_lwlr = np.array(sbp_pred_lwlr)
dbp_pred_lwlr = np.array(dbp_pred_lwlr)

sbp_mae_lwlr = mean_absolute_error(sbp_true_lwlr, sbp_pred_lwlr)
dbp_mae_lwlr = mean_absolute_error(dbp_true_lwlr, dbp_pred_lwlr)

sbp_rmse_lwlr = np.sqrt(mean_squared_error(sbp_true_lwlr, sbp_pred_lwlr))
dbp_rmse_lwlr = np.sqrt(mean_squared_error(dbp_true_lwlr, dbp_pred_lwlr))

sbp_r2_lwlr = r2_score(sbp_true_lwlr, sbp_pred_lwlr)
dbp_r2_lwlr = r2_score(dbp_true_lwlr, dbp_pred_lwlr)

avg_mae_lwlr = (sbp_mae_lwlr + dbp_mae_lwlr) / 2

print("BENDR Layer-wise LR Decay Results — Real mmHg Scale")
print("===================================================")

print("SBP")
print(f"MAE  : {sbp_mae_lwlr:.3f}")
print(f"RMSE : {sbp_rmse_lwlr:.3f}")
print(f"R2   : {sbp_r2_lwlr:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_lwlr:.3f}")
print(f"RMSE : {dbp_rmse_lwlr:.3f}")
print(f"R2   : {dbp_r2_lwlr:.4f}")

print("\nTable format:")
print(f"{sbp_mae_lwlr:.2f}/{dbp_mae_lwlr:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_lwlr:.3f}")

BENDR Layer-wise LR Decay Results — Real mmHg Scale
SBP
MAE  : 5.952
RMSE : 8.215
R2   : 0.6668

DBP
MAE  : 4.570
RMSE : 6.251
R2   : 0.4802

Table format:
5.95/4.57

Average MAE:
5.261


In [88]:
metrics_lwlr = {
    "model": "BENDR",
    "method": tuning_method_lwlr,
    "best_epoch": int(checkpoint_lwlr["epoch"]),
    "best_val_loss": float(checkpoint_lwlr["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_lwlr),
    "DBP_MAE": float(dbp_mae_lwlr),
    "AVG_MAE": float(avg_mae_lwlr),
    "SBP_RMSE": float(sbp_rmse_lwlr),
    "DBP_RMSE": float(dbp_rmse_lwlr),
    "SBP_R2": float(sbp_r2_lwlr),
    "DBP_R2": float(dbp_r2_lwlr),
    "table_format": f"{sbp_mae_lwlr:.2f}/{dbp_mae_lwlr:.2f}",
    "early_lr": float(checkpoint_lwlr["early_lr"]),
    "middle_lr": float(checkpoint_lwlr["middle_lr"]),
    "late_lr": float(checkpoint_lwlr["late_lr"]),
    "head_lr": float(checkpoint_lwlr["head_lr"]),
    "optimizer": checkpoint_lwlr["optimizer"],
    "trainable_parameters": int(checkpoint_lwlr["trainable_parameters"]),
    "total_parameters": int(checkpoint_lwlr["total_parameters"]),
    "seed": int(checkpoint_lwlr["seed"])
}

with open(metrics_path_lwlr, "w") as f:
    json.dump(metrics_lwlr, f, indent=4)

metrics_lwlr

{'model': 'BENDR',
 'method': 'layerwise_lr_decay_early1e-5_middle1.5e-5_late2e-5_head5e-5',
 'best_epoch': 28,
 'best_val_loss': 0.14943655283619992,
 'SBP_MAE': 5.951727867126465,
 'DBP_MAE': 4.570346355438232,
 'AVG_MAE': 5.261037111282349,
 'SBP_RMSE': 8.215479442980023,
 'DBP_RMSE': 6.25143355141474,
 'SBP_R2': 0.6667872667312622,
 'DBP_R2': 0.4801930785179138,
 'table_format': '5.95/4.57',
 'early_lr': 1e-05,
 'middle_lr': 1.5e-05,
 'late_lr': 2e-05,
 'head_lr': 5e-05,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# L2-SP regularized full fine-tuning

In [ ]:
# =========================
# L2-SP helper functions
# =========================

import torch
import torch.nn as nn
import pandas as pd
import json
import numpy as np

def save_pretrained_bendr_weights(model):
    """
    Save initial pretrained BENDR weights.
    These are W_pretrained in L2-SP:
    loss = MSE + lambda * ||W - W_pretrained||^2
    """
    pretrained_weights = {}

    for name, param in model.bendr.named_parameters():
        if param.requires_grad:
            pretrained_weights[name] = param.detach().clone()

    return pretrained_weights


def compute_l2sp_penalty(model, pretrained_weights):
    """
    Compute L2-SP penalty only for BENDR backbone parameters.
    """
    l2sp_loss = 0.0

    for name, param in model.bendr.named_parameters():
        if param.requires_grad and name in pretrained_weights:
            l2sp_loss = l2sp_loss + torch.sum((param - pretrained_weights[name].to(param.device)) ** 2)

    return l2sp_loss

In [ ]:
# =========================
# Train / Validation with L2-SP
# =========================

def train_one_epoch_l2sp(
    model,
    train_loader,
    optimizer,
    criterion,
    device,
    pretrained_bendr_weights,
    lambda_l2sp=1e-5
):
    model.train()
    total_loss = 0.0
    total_mse_loss = 0.0
    total_l2sp_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()

        pred = model(xb)

        if pred.shape != yb.shape:
            pred = pred.view_as(yb)

        mse_loss = criterion(pred, yb)
        l2sp_penalty = compute_l2sp_penalty(model, pretrained_bendr_weights)

        loss = mse_loss + lambda_l2sp * l2sp_penalty

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        batch_size = xb.size(0)

        total_loss += loss.item() * batch_size
        total_mse_loss += mse_loss.item() * batch_size
        total_l2sp_loss += l2sp_penalty.item() * batch_size

    return (
        total_loss / len(train_loader.dataset),
        total_mse_loss / len(train_loader.dataset),
        total_l2sp_loss / len(train_loader.dataset)
    )


def validate_one_epoch_l2sp(
    model,
    val_loader,
    criterion,
    device,
    pretrained_bendr_weights,
    lambda_l2sp=1e-5
):
    model.eval()
    total_loss = 0.0
    total_mse_loss = 0.0
    total_l2sp_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            pred = model(xb)

            if pred.shape != yb.shape:
                pred = pred.view_as(yb)

            mse_loss = criterion(pred, yb)
            l2sp_penalty = compute_l2sp_penalty(model, pretrained_bendr_weights)

            loss = mse_loss + lambda_l2sp * l2sp_penalty

            batch_size = xb.size(0)

            total_loss += loss.item() * batch_size
            total_mse_loss += mse_loss.item() * batch_size
            total_l2sp_loss += l2sp_penalty.item() * batch_size

    return (
        total_loss / len(val_loader.dataset),
        total_mse_loss / len(val_loader.dataset),
        total_l2sp_loss / len(val_loader.dataset)
    )

In [ ]:
# =========================
# Experiment: Full FT + Discriminative LR 2e-5 + L2-SP
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_l2sp = "full_ft_discriminative_lr_2e5_l2sp"

best_model_path_l2sp = "bendr_full_ft_discriminative_lr_2e5_l2sp_m4.pth"
history_path_l2sp = "history_full_ft_discriminative_lr_2e5_l2sp_m4.csv"
metrics_path_l2sp = "metrics_full_ft_discriminative_lr_2e5_l2sp_m4.json"

lambda_l2sp = 1e-5

model_l2sp = BENDR_BP(n_times=625, n_channels_in=1).to(device)

# Initialize LazyLinear / lazy layers
model_l2sp.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_l2sp(x_dummy)

# Full fine-tuning: all parameters trainable
for p in model_l2sp.parameters():
    p.requires_grad = True

# Save initial pretrained BENDR weights BEFORE training
pretrained_bendr_weights_l2sp = save_pretrained_bendr_weights(model_l2sp)

total_params_l2sp = sum(p.numel() for p in model_l2sp.parameters())
trainable_count_l2sp = sum(p.numel() for p in model_l2sp.parameters() if p.requires_grad)
frozen_count_l2sp = total_params_l2sp - trainable_count_l2sp

print("Total parameters:", total_params_l2sp)
print("Trainable parameters:", trainable_count_l2sp)
print("Frozen parameters:", frozen_count_l2sp)
print("Trainable percentage:", round(100 * trainable_count_l2sp / total_params_l2sp, 4), "%")
print("L2-SP lambda:", lambda_l2sp)

In [ ]:
# =========================
# AdamW optimizer for L2-SP experiment
# BENDR lr = 2e-5
# Head lr = 1e-4
# =========================

def create_discriminative_lr_optimizer_l2sp(
    model,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
):
    bendr_decay = []
    bendr_no_decay = []

    head_decay = []
    head_no_decay = []

    no_decay_keywords = ["bias", "norm", "bn", "layernorm", "layer_norm"]

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_low = name.lower()
        is_no_decay = any(k in name_low for k in no_decay_keywords)

        is_head = (
            "channel_adapter" in name
            or "decoder" in name
            or "output_mapping" in name
        )

        if is_head:
            if is_no_decay:
                head_no_decay.append(param)
            else:
                head_decay.append(param)
        else:
            if is_no_decay:
                bendr_no_decay.append(param)
            else:
                bendr_decay.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": bendr_decay, "lr": bendr_lr, "weight_decay": weight_decay},
            {"params": bendr_no_decay, "lr": bendr_lr, "weight_decay": 0.0},

            {"params": head_decay, "lr": head_lr, "weight_decay": weight_decay},
            {"params": head_no_decay, "lr": head_lr, "weight_decay": 0.0},
        ]
    )

    print("BENDR decay params:", sum(p.numel() for p in bendr_decay))
    print("BENDR no-decay params:", sum(p.numel() for p in bendr_no_decay))
    print("Head decay params:", sum(p.numel() for p in head_decay))
    print("Head no-decay params:", sum(p.numel() for p in head_no_decay))

    return optimizer


criterion_l2sp = nn.MSELoss()

optimizer_l2sp = create_discriminative_lr_optimizer_l2sp(
    model_l2sp,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
# =========================
# Train Full FT + Discriminative LR 2e-5 + L2-SP
# =========================

num_epochs_l2sp = 30
best_val_loss_l2sp = float("inf")

history_l2sp = []

for epoch in range(num_epochs_l2sp):

    train_loss_l2sp, train_mse_l2sp, train_l2sp_penalty = train_one_epoch_l2sp(
        model_l2sp,
        train_loader,
        optimizer_l2sp,
        criterion_l2sp,
        device,
        pretrained_bendr_weights_l2sp,
        lambda_l2sp=lambda_l2sp
    )

    val_loss_l2sp, val_mse_l2sp, val_l2sp_penalty = validate_one_epoch_l2sp(
        model_l2sp,
        val_loader,
        criterion_l2sp,
        device,
        pretrained_bendr_weights_l2sp,
        lambda_l2sp=lambda_l2sp
    )

    history_l2sp.append({
        "method": tuning_method_l2sp,
        "epoch": epoch + 1,

        "train_total_loss": float(train_loss_l2sp),
        "train_mse_loss": float(train_mse_l2sp),
        "train_l2sp_penalty_raw": float(train_l2sp_penalty),

        "val_total_loss": float(val_loss_l2sp),
        "val_mse_loss": float(val_mse_l2sp),
        "val_l2sp_penalty_raw": float(val_l2sp_penalty),

        "lambda_l2sp": float(lambda_l2sp),

        "trainable_parameters": int(trainable_count_l2sp),
        "total_parameters": int(total_params_l2sp),
        "frozen_parameters": int(frozen_count_l2sp),
        "trainable_percentage": float(100 * trainable_count_l2sp / total_params_l2sp),

        "bendr_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW"
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs_l2sp}] | "
        f"Train Total: {train_loss_l2sp:.6f} | "
        f"Train MSE: {train_mse_l2sp:.6f} | "
        f"Val Total: {val_loss_l2sp:.6f} | "
        f"Val MSE: {val_mse_l2sp:.6f} | "
        f"L2SP raw: {val_l2sp_penalty:.2f}"
    )

    # IMPORTANT:
    # Save best model based on val MSE, not total loss.
    # Because final task metric depends on prediction error, not regularization term.
    if val_mse_l2sp < best_val_loss_l2sp:
        best_val_loss_l2sp = val_mse_l2sp

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_l2sp,
            "model_state_dict": model_l2sp.state_dict(),
            "optimizer_state_dict": optimizer_l2sp.state_dict(),

            "best_val_mse_loss": float(best_val_loss_l2sp),
            "lambda_l2sp": float(lambda_l2sp),

            "trainable_parameters": int(trainable_count_l2sp),
            "total_parameters": int(total_params_l2sp),
            "frozen_parameters": int(frozen_count_l2sp),
            "trainable_percentage": float(100 * trainable_count_l2sp / total_params_l2sp),

            "bendr_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW"
        }, best_model_path_l2sp)

        print(f"Saved best L2-SP model: {best_model_path_l2sp}")

pd.DataFrame(history_l2sp).to_csv(history_path_l2sp, index=False)

print("Best val MSE loss:", best_val_loss_l2sp)

In [ ]:
# =========================
# Load best L2-SP model
# =========================

checkpoint_l2sp = torch.load(best_model_path_l2sp, map_location=device)

model_l2sp.load_state_dict(checkpoint_l2sp["model_state_dict"])
model_l2sp.eval()

best_epoch_l2sp = checkpoint_l2sp["epoch"]
best_val_loss_l2sp = checkpoint_l2sp["best_val_mse_loss"]

print("Loaded best L2-SP model")
print("Best epoch:", best_epoch_l2sp)
print("Best val MSE loss:", best_val_loss_l2sp)
print("Lambda L2-SP:", checkpoint_l2sp["lambda_l2sp"])

In [ ]:
# =========================
# Evaluate L2-SP model in real mmHg scale
# =========================

y_true_l2sp, y_pred_l2sp = evaluate_model(
    model_l2sp,
    test_loader,
    y_scaler,
    device
)

print("y_true_l2sp shape:", y_true_l2sp.shape)
print("y_pred_l2sp shape:", y_pred_l2sp.shape)

print("y_true min/max:", y_true_l2sp.min(), y_true_l2sp.max())
print("y_pred min/max:", y_pred_l2sp.min(), y_pred_l2sp.max())

In [ ]:
# =========================
# SBP / DBP metrics for L2-SP
# =========================

from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_l2sp, dbp_true_l2sp = [], []
sbp_pred_l2sp, dbp_pred_l2sp = [], []

for true_sig, pred_sig in zip(y_true_l2sp, y_pred_l2sp):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_l2sp.append(sbp_t)
    dbp_true_l2sp.append(dbp_t)
    sbp_pred_l2sp.append(sbp_p)
    dbp_pred_l2sp.append(dbp_p)

sbp_true_l2sp = np.array(sbp_true_l2sp)
dbp_true_l2sp = np.array(dbp_true_l2sp)
sbp_pred_l2sp = np.array(sbp_pred_l2sp)
dbp_pred_l2sp = np.array(dbp_pred_l2sp)

sbp_mae_l2sp = mean_absolute_error(sbp_true_l2sp, sbp_pred_l2sp)
dbp_mae_l2sp = mean_absolute_error(dbp_true_l2sp, dbp_pred_l2sp)

sbp_rmse_l2sp = np.sqrt(mean_squared_error(sbp_true_l2sp, sbp_pred_l2sp))
dbp_rmse_l2sp = np.sqrt(mean_squared_error(dbp_true_l2sp, dbp_pred_l2sp))

sbp_r2_l2sp = r2_score(sbp_true_l2sp, sbp_pred_l2sp)
dbp_r2_l2sp = r2_score(dbp_true_l2sp, dbp_pred_l2sp)

avg_mae_l2sp = (sbp_mae_l2sp + dbp_mae_l2sp) / 2

print("Full FT + Discriminative LR 2e-5 + L2-SP Results — Real mmHg Scale")
print("====================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_l2sp:.3f}")
print(f"RMSE : {sbp_rmse_l2sp:.3f}")
print(f"R2   : {sbp_r2_l2sp:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_l2sp:.3f}")
print(f"RMSE : {dbp_rmse_l2sp:.3f}")
print(f"R2   : {dbp_r2_l2sp:.4f}")

print("\nTable format:")
print(f"{sbp_mae_l2sp:.2f}/{dbp_mae_l2sp:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_l2sp:.3f}")

In [ ]:
# =========================
# Save L2-SP metrics
# =========================

l2sp_metrics = {
    "method": tuning_method_l2sp,
    "best_val_mse_loss": float(best_val_loss_l2sp),
    "best_epoch": int(best_epoch_l2sp),

    "SBP_MAE": float(sbp_mae_l2sp),
    "DBP_MAE": float(dbp_mae_l2sp),
    "AVG_MAE": float(avg_mae_l2sp),

    "SBP_RMSE": float(sbp_rmse_l2sp),
    "DBP_RMSE": float(dbp_rmse_l2sp),
    "SBP_R2": float(sbp_r2_l2sp),
    "DBP_R2": float(dbp_r2_l2sp),

    "table_format": f"{sbp_mae_l2sp:.2f}/{dbp_mae_l2sp:.2f}",
    "best_model_path": best_model_path_l2sp,

    "trainable_parameters": int(checkpoint_l2sp["trainable_parameters"]),
    "total_parameters": int(checkpoint_l2sp["total_parameters"]),
    "frozen_parameters": int(checkpoint_l2sp["frozen_parameters"]),
    "trainable_percentage": float(checkpoint_l2sp["trainable_percentage"]),

    "bendr_lr": float(checkpoint_l2sp["bendr_lr"]),
    "head_lr": float(checkpoint_l2sp["head_lr"]),
    "lambda_l2sp": float(checkpoint_l2sp["lambda_l2sp"]),
    "optimizer": checkpoint_l2sp["optimizer"]
}

with open(metrics_path_l2sp, "w") as f:
    json.dump(l2sp_metrics, f, indent=4)

pd.DataFrame([l2sp_metrics]).to_csv("metrics_full_ft_discriminative_lr_2e5_l2sp_m4.csv", index=False)

l2sp_metrics

lambda_l2sp = 1e-6

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import json
import numpy as np

def save_pretrained_bendr_weights(model):
    pretrained_weights = {}

    for name, param in model.bendr.named_parameters():
        if param.requires_grad:
            pretrained_weights[name] = param.detach().clone()

    return pretrained_weights


def compute_l2sp_penalty(model, pretrained_weights):
    l2sp_loss = 0.0

    for name, param in model.bendr.named_parameters():
        if param.requires_grad and name in pretrained_weights:
            l2sp_loss = l2sp_loss + torch.sum((param - pretrained_weights[name].to(param.device)) ** 2)

    return l2sp_loss

In [ ]:
def train_one_epoch_l2sp(
    model,
    train_loader,
    optimizer,
    criterion,
    device,
    pretrained_bendr_weights,
    lambda_l2sp=1e-6
):
    model.train()
    total_loss = 0.0
    total_mse_loss = 0.0
    total_l2sp_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device).float()
        yb = yb.to(device).float()

        optimizer.zero_grad()

        pred = model(xb)

        if pred.shape != yb.shape:
            pred = pred.view_as(yb)

        mse_loss = criterion(pred, yb)
        l2sp_penalty = compute_l2sp_penalty(model, pretrained_bendr_weights)

        loss = mse_loss + lambda_l2sp * l2sp_penalty

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        batch_size = xb.size(0)

        total_loss += loss.item() * batch_size
        total_mse_loss += mse_loss.item() * batch_size
        total_l2sp_loss += l2sp_penalty.item() * batch_size

    return (
        total_loss / len(train_loader.dataset),
        total_mse_loss / len(train_loader.dataset),
        total_l2sp_loss / len(train_loader.dataset)
    )


def validate_one_epoch_l2sp(
    model,
    val_loader,
    criterion,
    device,
    pretrained_bendr_weights,
    lambda_l2sp=1e-6
):
    model.eval()
    total_loss = 0.0
    total_mse_loss = 0.0
    total_l2sp_loss = 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device).float()
            yb = yb.to(device).float()

            pred = model(xb)

            if pred.shape != yb.shape:
                pred = pred.view_as(yb)

            mse_loss = criterion(pred, yb)
            l2sp_penalty = compute_l2sp_penalty(model, pretrained_bendr_weights)

            loss = mse_loss + lambda_l2sp * l2sp_penalty

            batch_size = xb.size(0)

            total_loss += loss.item() * batch_size
            total_mse_loss += mse_loss.item() * batch_size
            total_l2sp_loss += l2sp_penalty.item() * batch_size

    return (
        total_loss / len(val_loader.dataset),
        total_mse_loss / len(val_loader.dataset),
        total_l2sp_loss / len(val_loader.dataset)
    )

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_l2sp_1e6 = "full_ft_discriminative_lr_2e5_l2sp_1e6"

best_model_path_l2sp_1e6 = "bendr_full_ft_discriminative_lr_2e5_l2sp_1e6_m4.pth"
history_path_l2sp_1e6 = "history_full_ft_discriminative_lr_2e5_l2sp_1e6_m4.csv"
metrics_path_l2sp_1e6 = "metrics_full_ft_discriminative_lr_2e5_l2sp_1e6_m4.json"

lambda_l2sp_1e6 = 1e-6

model_l2sp_1e6 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_l2sp_1e6.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_l2sp_1e6(x_dummy)

for p in model_l2sp_1e6.parameters():
    p.requires_grad = True

pretrained_bendr_weights_l2sp_1e6 = save_pretrained_bendr_weights(model_l2sp_1e6)

total_params_l2sp_1e6 = sum(p.numel() for p in model_l2sp_1e6.parameters())
trainable_count_l2sp_1e6 = sum(p.numel() for p in model_l2sp_1e6.parameters() if p.requires_grad)
frozen_count_l2sp_1e6 = total_params_l2sp_1e6 - trainable_count_l2sp_1e6

print("Total parameters:", total_params_l2sp_1e6)
print("Trainable parameters:", trainable_count_l2sp_1e6)
print("Frozen parameters:", frozen_count_l2sp_1e6)
print("Trainable percentage:", round(100 * trainable_count_l2sp_1e6 / total_params_l2sp_1e6, 4), "%")
print("L2-SP lambda:", lambda_l2sp_1e6)

In [ ]:
def create_discriminative_lr_optimizer_l2sp_1e6(
    model,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
):
    bendr_decay = []
    bendr_no_decay = []

    head_decay = []
    head_no_decay = []

    no_decay_keywords = ["bias", "norm", "bn", "layernorm", "layer_norm"]

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_low = name.lower()
        is_no_decay = any(k in name_low for k in no_decay_keywords)

        is_head = (
            "channel_adapter" in name
            or "decoder" in name
            or "output_mapping" in name
        )

        if is_head:
            if is_no_decay:
                head_no_decay.append(param)
            else:
                head_decay.append(param)
        else:
            if is_no_decay:
                bendr_no_decay.append(param)
            else:
                bendr_decay.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": bendr_decay, "lr": bendr_lr, "weight_decay": weight_decay},
            {"params": bendr_no_decay, "lr": bendr_lr, "weight_decay": 0.0},
            {"params": head_decay, "lr": head_lr, "weight_decay": weight_decay},
            {"params": head_no_decay, "lr": head_lr, "weight_decay": 0.0},
        ]
    )

    print("BENDR decay params:", sum(p.numel() for p in bendr_decay))
    print("BENDR no-decay params:", sum(p.numel() for p in bendr_no_decay))
    print("Head decay params:", sum(p.numel() for p in head_decay))
    print("Head no-decay params:", sum(p.numel() for p in head_no_decay))

    return optimizer


criterion_l2sp_1e6 = nn.MSELoss()

optimizer_l2sp_1e6 = create_discriminative_lr_optimizer_l2sp_1e6(
    model_l2sp_1e6,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
num_epochs_l2sp_1e6 = 30
best_val_loss_l2sp_1e6 = float("inf")

history_l2sp_1e6 = []

for epoch in range(num_epochs_l2sp_1e6):

    train_loss_l2sp_1e6, train_mse_l2sp_1e6, train_l2sp_penalty_1e6 = train_one_epoch_l2sp(
        model_l2sp_1e6,
        train_loader,
        optimizer_l2sp_1e6,
        criterion_l2sp_1e6,
        device,
        pretrained_bendr_weights_l2sp_1e6,
        lambda_l2sp=lambda_l2sp_1e6
    )

    val_loss_l2sp_1e6, val_mse_l2sp_1e6, val_l2sp_penalty_1e6 = validate_one_epoch_l2sp(
        model_l2sp_1e6,
        val_loader,
        criterion_l2sp_1e6,
        device,
        pretrained_bendr_weights_l2sp_1e6,
        lambda_l2sp=lambda_l2sp_1e6
    )

    history_l2sp_1e6.append({
        "method": tuning_method_l2sp_1e6,
        "epoch": epoch + 1,
        "train_total_loss": float(train_loss_l2sp_1e6),
        "train_mse_loss": float(train_mse_l2sp_1e6),
        "train_l2sp_penalty_raw": float(train_l2sp_penalty_1e6),
        "val_total_loss": float(val_loss_l2sp_1e6),
        "val_mse_loss": float(val_mse_l2sp_1e6),
        "val_l2sp_penalty_raw": float(val_l2sp_penalty_1e6),
        "lambda_l2sp": float(lambda_l2sp_1e6),
        "trainable_parameters": int(trainable_count_l2sp_1e6),
        "total_parameters": int(total_params_l2sp_1e6),
        "frozen_parameters": int(frozen_count_l2sp_1e6),
        "trainable_percentage": float(100 * trainable_count_l2sp_1e6 / total_params_l2sp_1e6),
        "bendr_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW"
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs_l2sp_1e6}] | "
        f"Train Total: {train_loss_l2sp_1e6:.6f} | "
        f"Train MSE: {train_mse_l2sp_1e6:.6f} | "
        f"Val Total: {val_loss_l2sp_1e6:.6f} | "
        f"Val MSE: {val_mse_l2sp_1e6:.6f} | "
        f"L2SP raw: {val_l2sp_penalty_1e6:.2f}"
    )

    if val_mse_l2sp_1e6 < best_val_loss_l2sp_1e6:
        best_val_loss_l2sp_1e6 = val_mse_l2sp_1e6

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_l2sp_1e6,
            "model_state_dict": model_l2sp_1e6.state_dict(),
            "optimizer_state_dict": optimizer_l2sp_1e6.state_dict(),
            "best_val_mse_loss": float(best_val_loss_l2sp_1e6),
            "lambda_l2sp": float(lambda_l2sp_1e6),
            "trainable_parameters": int(trainable_count_l2sp_1e6),
            "total_parameters": int(total_params_l2sp_1e6),
            "frozen_parameters": int(frozen_count_l2sp_1e6),
            "trainable_percentage": float(100 * trainable_count_l2sp_1e6 / total_params_l2sp_1e6),
            "bendr_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW"
        }, best_model_path_l2sp_1e6)

        print(f"Saved best L2-SP 1e-6 model: {best_model_path_l2sp_1e6}")

pd.DataFrame(history_l2sp_1e6).to_csv(history_path_l2sp_1e6, index=False)

print("Best val MSE loss:", best_val_loss_l2sp_1e6)

In [ ]:
checkpoint_l2sp_1e6 = torch.load(best_model_path_l2sp_1e6, map_location=device)

model_l2sp_1e6.load_state_dict(checkpoint_l2sp_1e6["model_state_dict"])
model_l2sp_1e6.eval()

best_epoch_l2sp_1e6 = checkpoint_l2sp_1e6["epoch"]
best_val_loss_l2sp_1e6 = checkpoint_l2sp_1e6["best_val_mse_loss"]

print("Loaded best L2-SP 1e-6 model")
print("Best epoch:", best_epoch_l2sp_1e6)
print("Best val MSE loss:", best_val_loss_l2sp_1e6)
print("Lambda L2-SP:", checkpoint_l2sp_1e6["lambda_l2sp"])

In [ ]:
y_true_l2sp_1e6, y_pred_l2sp_1e6 = evaluate_model(
    model_l2sp_1e6,
    test_loader,
    y_scaler,
    device
)

print("y_true_l2sp_1e6 shape:", y_true_l2sp_1e6.shape)
print("y_pred_l2sp_1e6 shape:", y_pred_l2sp_1e6.shape)

print("y_true min/max:", y_true_l2sp_1e6.min(), y_true_l2sp_1e6.max())
print("y_pred min/max:", y_pred_l2sp_1e6.min(), y_pred_l2sp_1e6.max())

In [ ]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_l2sp_1e6, dbp_true_l2sp_1e6 = [], []
sbp_pred_l2sp_1e6, dbp_pred_l2sp_1e6 = [], []

for true_sig, pred_sig in zip(y_true_l2sp_1e6, y_pred_l2sp_1e6):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_l2sp_1e6.append(sbp_t)
    dbp_true_l2sp_1e6.append(dbp_t)
    sbp_pred_l2sp_1e6.append(sbp_p)
    dbp_pred_l2sp_1e6.append(dbp_p)

sbp_true_l2sp_1e6 = np.array(sbp_true_l2sp_1e6)
dbp_true_l2sp_1e6 = np.array(dbp_true_l2sp_1e6)
sbp_pred_l2sp_1e6 = np.array(sbp_pred_l2sp_1e6)
dbp_pred_l2sp_1e6 = np.array(dbp_pred_l2sp_1e6)

sbp_mae_l2sp_1e6 = mean_absolute_error(sbp_true_l2sp_1e6, sbp_pred_l2sp_1e6)
dbp_mae_l2sp_1e6 = mean_absolute_error(dbp_true_l2sp_1e6, dbp_pred_l2sp_1e6)

sbp_rmse_l2sp_1e6 = np.sqrt(mean_squared_error(sbp_true_l2sp_1e6, sbp_pred_l2sp_1e6))
dbp_rmse_l2sp_1e6 = np.sqrt(mean_squared_error(dbp_true_l2sp_1e6, dbp_pred_l2sp_1e6))

sbp_r2_l2sp_1e6 = r2_score(sbp_true_l2sp_1e6, sbp_pred_l2sp_1e6)
dbp_r2_l2sp_1e6 = r2_score(dbp_true_l2sp_1e6, dbp_pred_l2sp_1e6)

avg_mae_l2sp_1e6 = (sbp_mae_l2sp_1e6 + dbp_mae_l2sp_1e6) / 2

print("Full FT + Discriminative LR 2e-5 + L2-SP 1e-6 Results — Real mmHg Scale")
print("=========================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_l2sp_1e6:.3f}")
print(f"RMSE : {sbp_rmse_l2sp_1e6:.3f}")
print(f"R2   : {sbp_r2_l2sp_1e6:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_l2sp_1e6:.3f}")
print(f"RMSE : {dbp_rmse_l2sp_1e6:.3f}")
print(f"R2   : {dbp_r2_l2sp_1e6:.4f}")

print("\nTable format:")
print(f"{sbp_mae_l2sp_1e6:.2f}/{dbp_mae_l2sp_1e6:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_l2sp_1e6:.3f}")

In [ ]:
l2sp_1e6_metrics = {
    "method": tuning_method_l2sp_1e6,
    "best_val_mse_loss": float(best_val_loss_l2sp_1e6),
    "best_epoch": int(best_epoch_l2sp_1e6),

    "SBP_MAE": float(sbp_mae_l2sp_1e6),
    "DBP_MAE": float(dbp_mae_l2sp_1e6),
    "AVG_MAE": float(avg_mae_l2sp_1e6),

    "SBP_RMSE": float(sbp_rmse_l2sp_1e6),
    "DBP_RMSE": float(dbp_rmse_l2sp_1e6),
    "SBP_R2": float(sbp_r2_l2sp_1e6),
    "DBP_R2": float(dbp_r2_l2sp_1e6),

    "table_format": f"{sbp_mae_l2sp_1e6:.2f}/{dbp_mae_l2sp_1e6:.2f}",
    "best_model_path": best_model_path_l2sp_1e6,

    "trainable_parameters": int(checkpoint_l2sp_1e6["trainable_parameters"]),
    "total_parameters": int(checkpoint_l2sp_1e6["total_parameters"]),
    "frozen_parameters": int(checkpoint_l2sp_1e6["frozen_parameters"]),
    "trainable_percentage": float(checkpoint_l2sp_1e6["trainable_percentage"]),

    "bendr_lr": float(checkpoint_l2sp_1e6["bendr_lr"]),
    "head_lr": float(checkpoint_l2sp_1e6["head_lr"]),
    "lambda_l2sp": float(checkpoint_l2sp_1e6["lambda_l2sp"]),
    "optimizer": checkpoint_l2sp_1e6["optimizer"]
}

with open(metrics_path_l2sp_1e6, "w") as f:
    json.dump(l2sp_1e6_metrics, f, indent=4)

pd.DataFrame([l2sp_1e6_metrics]).to_csv("metrics_full_ft_discriminative_lr_2e5_l2sp_1e6_m4.csv", index=False)

l2sp_1e6_metrics

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import json
import numpy as np

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_disclr2_50 = "full_ft_discriminative_lr_2e5_50ep_earlystop"

best_model_path_disclr2_50 = "bendr_full_ft_discriminative_lr_2e5_50ep_earlystop_m4.pth"
history_path_disclr2_50 = "history_full_ft_discriminative_lr_2e5_50ep_earlystop_m4.csv"
metrics_path_disclr2_50 = "metrics_full_ft_discriminative_lr_2e5_50ep_earlystop_m4.json"

model_disclr2_50 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_disclr2_50.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_disclr2_50(x_dummy)

for p in model_disclr2_50.parameters():
    p.requires_grad = True

total_params_disclr2_50 = sum(p.numel() for p in model_disclr2_50.parameters())
trainable_count_disclr2_50 = sum(p.numel() for p in model_disclr2_50.parameters() if p.requires_grad)
frozen_count_disclr2_50 = total_params_disclr2_50 - trainable_count_disclr2_50

print("Total parameters:", total_params_disclr2_50)
print("Trainable parameters:", trainable_count_disclr2_50)
print("Frozen parameters:", frozen_count_disclr2_50)
print("Trainable percentage:", round(100 * trainable_count_disclr2_50 / total_params_disclr2_50, 4), "%")

In [ ]:
def create_discriminative_lr_optimizer_2e5_50(
    model,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
):
    bendr_decay = []
    bendr_no_decay = []

    head_decay = []
    head_no_decay = []

    no_decay_keywords = ["bias", "norm", "bn", "layernorm", "layer_norm"]

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        name_low = name.lower()
        is_no_decay = any(k in name_low for k in no_decay_keywords)

        is_head = (
            "channel_adapter" in name
            or "decoder" in name
            or "output_mapping" in name
        )

        if is_head:
            if is_no_decay:
                head_no_decay.append(param)
            else:
                head_decay.append(param)
        else:
            if is_no_decay:
                bendr_no_decay.append(param)
            else:
                bendr_decay.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": bendr_decay, "lr": bendr_lr, "weight_decay": weight_decay},
            {"params": bendr_no_decay, "lr": bendr_lr, "weight_decay": 0.0},
            {"params": head_decay, "lr": head_lr, "weight_decay": weight_decay},
            {"params": head_no_decay, "lr": head_lr, "weight_decay": 0.0},
        ]
    )

    print("BENDR decay params:", sum(p.numel() for p in bendr_decay))
    print("BENDR no-decay params:", sum(p.numel() for p in bendr_no_decay))
    print("Head decay params:", sum(p.numel() for p in head_decay))
    print("Head no-decay params:", sum(p.numel() for p in head_no_decay))

    return optimizer


criterion_disclr2_50 = nn.MSELoss()

optimizer_disclr2_50 = create_discriminative_lr_optimizer_2e5_50(
    model_disclr2_50,
    bendr_lr=2e-5,
    head_lr=1e-4,
    weight_decay=1e-2
)

In [ ]:
num_epochs_disclr2_50 = 50
patience_disclr2_50 = 7
min_delta_disclr2_50 = 1e-5

best_val_loss_disclr2_50 = float("inf")
epochs_without_improvement_disclr2_50 = 0

history_disclr2_50 = []

for epoch in range(num_epochs_disclr2_50):

    train_loss_disclr2_50 = train_one_epoch(
        model_disclr2_50,
        train_loader,
        optimizer_disclr2_50,
        criterion_disclr2_50,
        device
    )

    val_loss_disclr2_50 = validate_one_epoch(
        model_disclr2_50,
        val_loader,
        criterion_disclr2_50,
        device
    )

    history_disclr2_50.append({
        "method": tuning_method_disclr2_50,
        "epoch": epoch + 1,
        "train_loss": float(train_loss_disclr2_50),
        "val_loss": float(val_loss_disclr2_50),
        "trainable_parameters": int(trainable_count_disclr2_50),
        "total_parameters": int(total_params_disclr2_50),
        "frozen_parameters": int(frozen_count_disclr2_50),
        "trainable_percentage": float(100 * trainable_count_disclr2_50 / total_params_disclr2_50),
        "bendr_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW"
    })

    print(
        f"Epoch [{epoch+1}/{num_epochs_disclr2_50}] | "
        f"Train Loss: {train_loss_disclr2_50:.6f} | "
        f"Val Loss: {val_loss_disclr2_50:.6f}"
    )

    if val_loss_disclr2_50 < best_val_loss_disclr2_50 - min_delta_disclr2_50:
        best_val_loss_disclr2_50 = val_loss_disclr2_50
        epochs_without_improvement_disclr2_50 = 0

        torch.save({
            "epoch": epoch + 1,
            "method": tuning_method_disclr2_50,
            "model_state_dict": model_disclr2_50.state_dict(),
            "optimizer_state_dict": optimizer_disclr2_50.state_dict(),
            "best_val_loss": float(best_val_loss_disclr2_50),
            "trainable_parameters": int(trainable_count_disclr2_50),
            "total_parameters": int(total_params_disclr2_50),
            "frozen_parameters": int(frozen_count_disclr2_50),
            "trainable_percentage": float(100 * trainable_count_disclr2_50 / total_params_disclr2_50),
            "bendr_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "patience": patience_disclr2_50,
            "min_delta": min_delta_disclr2_50
        }, best_model_path_disclr2_50)

        print(f"Saved best model: {best_model_path_disclr2_50}")

    else:
        epochs_without_improvement_disclr2_50 += 1
        print(f"No improvement count: {epochs_without_improvement_disclr2_50}/{patience_disclr2_50}")

    if epochs_without_improvement_disclr2_50 >= patience_disclr2_50:
        print(f"Early stopping at epoch {epoch + 1}")
        break

pd.DataFrame(history_disclr2_50).to_csv(history_path_disclr2_50, index=False)

print("Best val loss:", best_val_loss_disclr2_50)

In [ ]:
checkpoint_disclr2_50 = torch.load(best_model_path_disclr2_50, map_location=device)

model_disclr2_50.load_state_dict(checkpoint_disclr2_50["model_state_dict"])
model_disclr2_50.eval()

best_epoch_disclr2_50 = checkpoint_disclr2_50["epoch"]
best_val_loss_disclr2_50 = checkpoint_disclr2_50["best_val_loss"]

print("Loaded best model")
print("Best epoch:", best_epoch_disclr2_50)
print("Best val loss:", best_val_loss_disclr2_50)

In [ ]:
y_true_disclr2_50, y_pred_disclr2_50 = evaluate_model(
    model_disclr2_50,
    test_loader,
    y_scaler,
    device
)

print("y_true_disclr2_50 shape:", y_true_disclr2_50.shape)
print("y_pred_disclr2_50 shape:", y_pred_disclr2_50.shape)

print("y_true min/max:", y_true_disclr2_50.min(), y_true_disclr2_50.max())
print("y_pred min/max:", y_pred_disclr2_50.min(), y_pred_disclr2_50.max())

In [ ]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_disclr2_50, dbp_true_disclr2_50 = [], []
sbp_pred_disclr2_50, dbp_pred_disclr2_50 = [], []

for true_sig, pred_sig in zip(y_true_disclr2_50, y_pred_disclr2_50):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_disclr2_50.append(sbp_t)
    dbp_true_disclr2_50.append(dbp_t)
    sbp_pred_disclr2_50.append(sbp_p)
    dbp_pred_disclr2_50.append(dbp_p)

sbp_true_disclr2_50 = np.array(sbp_true_disclr2_50)
dbp_true_disclr2_50 = np.array(dbp_true_disclr2_50)
sbp_pred_disclr2_50 = np.array(sbp_pred_disclr2_50)
dbp_pred_disclr2_50 = np.array(dbp_pred_disclr2_50)

sbp_mae_disclr2_50 = mean_absolute_error(sbp_true_disclr2_50, sbp_pred_disclr2_50)
dbp_mae_disclr2_50 = mean_absolute_error(dbp_true_disclr2_50, dbp_pred_disclr2_50)

sbp_rmse_disclr2_50 = np.sqrt(mean_squared_error(sbp_true_disclr2_50, sbp_pred_disclr2_50))
dbp_rmse_disclr2_50 = np.sqrt(mean_squared_error(dbp_true_disclr2_50, dbp_pred_disclr2_50))

sbp_r2_disclr2_50 = r2_score(sbp_true_disclr2_50, sbp_pred_disclr2_50)
dbp_r2_disclr2_50 = r2_score(dbp_true_disclr2_50, dbp_pred_disclr2_50)

avg_mae_disclr2_50 = (sbp_mae_disclr2_50 + dbp_mae_disclr2_50) / 2

print("Full FT + Discriminative LR 2e-5, 50 Epochs + Early Stopping Results — Real mmHg Scale")
print("=======================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_disclr2_50:.3f}")
print(f"RMSE : {sbp_rmse_disclr2_50:.3f}")
print(f"R2   : {sbp_r2_disclr2_50:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_disclr2_50:.3f}")
print(f"RMSE : {dbp_rmse_disclr2_50:.3f}")
print(f"R2   : {dbp_r2_disclr2_50:.4f}")

print("\nTable format:")
print(f"{sbp_mae_disclr2_50:.2f}/{dbp_mae_disclr2_50:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_disclr2_50:.3f}")

# Two stage fine tuning

In [89]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage = "two_stage_head5e-5_then_full_ft_backbone2e-5_head5e-5"

best_model_path_twostage = "bendr_two_stage_head5e-5_then_full_ft_backbone2e-5_head5e-5_m4_seed42.pth"
history_path_twostage = "history_bendr_two_stage_head5e-5_then_full_ft_backbone2e-5_head5e-5_m4_seed42.csv"
metrics_path_twostage = "metrics_bendr_two_stage_head5e-5_then_full_ft_backbone2e-5_head5e-5_m4_seed42.json"

model_twostage = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage(x_dummy)

criterion_twostage = nn.MSELoss()

print("Two-stage setup ready")

Device: cuda
Two-stage setup ready


In [90]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        if is_head_parameter(name):
            param.requires_grad = True
        else:
            param.requires_grad = False


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [91]:
set_stage1_head_only(model_twostage)

total_params_stage1, trainable_params_stage1 = count_trainable(model_twostage)

head_params_stage1 = []

for name, param in model_twostage.named_parameters():
    if param.requires_grad:
        head_params_stage1.append(param)

optimizer_stage1 = torch.optim.AdamW(
    head_params_stage1,
    lr=5e-5,
    weight_decay=1e-2
)

history_twostage = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [92]:
import time
import pandas as pd
import torch

num_epochs_stage1 = 10
best_val_loss_twostage = float("inf")

start_time_twostage = time.time()

for epoch in range(num_epochs_stage1):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage,
        train_loader,
        optimizer_stage1,
        criterion_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage,
        val_loader,
        criterion_twostage,
        device
    )

    history_twostage.append({
        "method": tuning_method_twostage,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1),
        "total_parameters": int(total_params_stage1),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage:
        best_val_loss_twostage = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage,
            "model_state_dict": model_twostage.state_dict(),
            "optimizer_state_dict": optimizer_stage1.state_dict(),
            "best_val_loss": float(best_val_loss_twostage),
            "backbone_lr": 0.0,
            "head_lr": 5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1),
            "total_parameters": int(total_params_stage1),
            "seed": SEED
        }, best_model_path_twostage)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/10] | Train Loss: 0.974442 | Val Loss: 0.877207 | Time: 10.50s saved
Stage 1 Epoch [2/10] | Train Loss: 0.829440 | Val Loss: 0.809631 | Time: 11.27s saved
Stage 1 Epoch [3/10] | Train Loss: 0.790283 | Val Loss: 0.775979 | Time: 11.04s saved
Stage 1 Epoch [4/10] | Train Loss: 0.765465 | Val Loss: 0.742931 | Time: 11.14s saved
Stage 1 Epoch [5/10] | Train Loss: 0.743767 | Val Loss: 0.730071 | Time: 11.31s saved
Stage 1 Epoch [6/10] | Train Loss: 0.724712 | Val Loss: 0.696629 | Time: 11.45s saved
Stage 1 Epoch [7/10] | Train Loss: 0.709460 | Val Loss: 0.682820 | Time: 11.50s saved
Stage 1 Epoch [8/10] | Train Loss: 0.697772 | Val Loss: 0.667847 | Time: 11.40s saved
Stage 1 Epoch [9/10] | Train Loss: 0.686459 | Val Loss: 0.660526 | Time: 11.30s saved
Stage 1 Epoch [10/10] | Train Loss: 0.677835 | Val Loss: 0.626079 | Time: 11.18s saved


In [93]:
set_stage2_full_ft(model_twostage)

total_params_stage2, trainable_params_stage2 = count_trainable(model_twostage)

backbone_params_stage2 = []
head_params_stage2 = []

for name, param in model_twostage.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2.append(param)
    else:
        backbone_params_stage2.append(param)

optimizer_stage2 = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2, "lr": 2e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2, "lr": 5e-5, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2))
print("Head params:", sum(p.numel() for p in head_params_stage2))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [94]:
num_epochs_stage2 = 30

for epoch in range(num_epochs_stage2):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage,
        train_loader,
        optimizer_stage2,
        criterion_twostage,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage,
        val_loader,
        criterion_twostage,
        device
    )

    history_twostage.append({
        "method": tuning_method_twostage,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2),
        "total_parameters": int(total_params_stage2),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage:
        best_val_loss_twostage = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage,
            "model_state_dict": model_twostage.state_dict(),
            "optimizer_state_dict": optimizer_stage2.state_dict(),
            "best_val_loss": float(best_val_loss_twostage),
            "backbone_lr": 2e-5,
            "head_lr": 5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2),
            "total_parameters": int(total_params_stage2),
            "seed": SEED
        }, best_model_path_twostage)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage).to_csv(history_path_twostage, index=False)

print("Best val loss:", best_val_loss_twostage)
print("Total training time:", time.time() - start_time_twostage)

Stage 2 Epoch [1/30] | Train Loss: 0.455629 | Val Loss: 0.376707 | Time: 14.55s saved
Stage 2 Epoch [2/30] | Train Loss: 0.334018 | Val Loss: 0.313736 | Time: 14.74s saved
Stage 2 Epoch [3/30] | Train Loss: 0.284552 | Val Loss: 0.328305 | Time: 13.88s 
Stage 2 Epoch [4/30] | Train Loss: 0.250856 | Val Loss: 0.255242 | Time: 15.23s saved
Stage 2 Epoch [5/30] | Train Loss: 0.221163 | Val Loss: 0.224024 | Time: 15.18s saved
Stage 2 Epoch [6/30] | Train Loss: 0.202224 | Val Loss: 0.223337 | Time: 14.80s saved
Stage 2 Epoch [7/30] | Train Loss: 0.186680 | Val Loss: 0.195638 | Time: 14.92s saved
Stage 2 Epoch [8/30] | Train Loss: 0.173357 | Val Loss: 0.194748 | Time: 14.84s saved
Stage 2 Epoch [9/30] | Train Loss: 0.163605 | Val Loss: 0.193738 | Time: 14.96s saved
Stage 2 Epoch [10/30] | Train Loss: 0.154653 | Val Loss: 0.174783 | Time: 15.05s saved
Stage 2 Epoch [11/30] | Train Loss: 0.146587 | Val Loss: 0.176363 | Time: 13.92s 
Stage 2 Epoch [12/30] | Train Loss: 0.140601 | Val Loss: 0.175

In [95]:
checkpoint_twostage = torch.load(best_model_path_twostage, map_location=device)

model_twostage.load_state_dict(checkpoint_twostage["model_state_dict"])
model_twostage = model_twostage.to(device)
model_twostage.eval()

print("Loaded best two-stage model")
print("Best stage:", checkpoint_twostage["stage"])
print("Best epoch:", checkpoint_twostage["epoch"])
print("Best val loss:", checkpoint_twostage["best_val_loss"])

Loaded best two-stage model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 28
Best val loss: 0.13846667468827378


In [96]:
y_true_twostage, y_pred_twostage = evaluate_model(
    model_twostage,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage shape:", y_true_twostage.shape)
print("y_pred_twostage shape:", y_pred_twostage.shape)

print("y_true min/max:", y_true_twostage.min(), y_true_twostage.max())
print("y_pred min/max:", y_pred_twostage.min(), y_pred_twostage.max())

y_true_twostage shape: (5064, 625)
y_pred_twostage shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 43.067055 191.03566


In [97]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage, dbp_true_twostage = [], []
sbp_pred_twostage, dbp_pred_twostage = [], []

for true_sig, pred_sig in zip(y_true_twostage, y_pred_twostage):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage.append(sbp_t)
    dbp_true_twostage.append(dbp_t)
    sbp_pred_twostage.append(sbp_p)
    dbp_pred_twostage.append(dbp_p)

sbp_true_twostage = np.array(sbp_true_twostage)
dbp_true_twostage = np.array(dbp_true_twostage)
sbp_pred_twostage = np.array(sbp_pred_twostage)
dbp_pred_twostage = np.array(dbp_pred_twostage)

sbp_mae_twostage = mean_absolute_error(sbp_true_twostage, sbp_pred_twostage)
dbp_mae_twostage = mean_absolute_error(dbp_true_twostage, dbp_pred_twostage)

sbp_rmse_twostage = np.sqrt(mean_squared_error(sbp_true_twostage, sbp_pred_twostage))
dbp_rmse_twostage = np.sqrt(mean_squared_error(dbp_true_twostage, dbp_pred_twostage))

sbp_r2_twostage = r2_score(sbp_true_twostage, sbp_pred_twostage)
dbp_r2_twostage = r2_score(dbp_true_twostage, dbp_pred_twostage)

avg_mae_twostage = (sbp_mae_twostage + dbp_mae_twostage) / 2

print("BENDR Two-Stage Head 5e-5 Then Full FT Backbone 2e-5 Head 5e-5 Results — Real mmHg Scale")
print("================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage:.3f}")
print(f"RMSE : {sbp_rmse_twostage:.3f}")
print(f"R2   : {sbp_r2_twostage:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage:.3f}")
print(f"RMSE : {dbp_rmse_twostage:.3f}")
print(f"R2   : {dbp_r2_twostage:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage:.2f}/{dbp_mae_twostage:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage:.3f}")

BENDR Two-Stage Head 5e-5 Then Full FT Backbone 2e-5 Head 5e-5 Results — Real mmHg Scale
SBP
MAE  : 5.863
RMSE : 8.054
R2   : 0.6797

DBP
MAE  : 4.594
RMSE : 6.170
R2   : 0.4937

Table format:
5.86/4.59

Average MAE:
5.229


In [98]:
metrics_twostage = {
    "model": "BENDR",
    "method": tuning_method_twostage,
    "best_stage": checkpoint_twostage["stage"],
    "best_epoch": int(checkpoint_twostage["epoch"]),
    "best_val_loss": float(checkpoint_twostage["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_twostage),
    "DBP_MAE": float(dbp_mae_twostage),
    "AVG_MAE": float(avg_mae_twostage),
    "SBP_RMSE": float(sbp_rmse_twostage),
    "DBP_RMSE": float(dbp_rmse_twostage),
    "SBP_R2": float(sbp_r2_twostage),
    "DBP_R2": float(dbp_r2_twostage),
    "table_format": f"{sbp_mae_twostage:.2f}/{dbp_mae_twostage:.2f}",
    "stage1_head_lr": 5e-5,
    "stage2_backbone_lr": 2e-5,
    "stage2_head_lr": 5e-5,
    "optimizer": checkpoint_twostage["optimizer"],
    "trainable_parameters": int(checkpoint_twostage["trainable_parameters"]),
    "total_parameters": int(checkpoint_twostage["total_parameters"]),
    "seed": int(checkpoint_twostage["seed"])
}

with open(metrics_path_twostage, "w") as f:
    json.dump(metrics_twostage, f, indent=4)

metrics_twostage

{'model': 'BENDR',
 'method': 'two_stage_head5e-5_then_full_ft_backbone2e-5_head5e-5',
 'best_stage': 'stage2_full_ft_discriminative_lr',
 'best_epoch': 28,
 'best_val_loss': 0.13846667468827378,
 'SBP_MAE': 5.863358974456787,
 'DBP_MAE': 4.593940734863281,
 'AVG_MAE': 5.228649854660034,
 'SBP_RMSE': 8.054333239663409,
 'DBP_RMSE': 6.1697085670629725,
 'SBP_R2': 0.6797308921813965,
 'DBP_R2': 0.49369508028030396,
 'table_format': '5.86/4.59',
 'stage1_head_lr': 5e-05,
 'stage2_backbone_lr': 2e-05,
 'stage2_head_lr': 5e-05,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# two stage with 
Stage 1:
Backbone frozen
Head LR = 1e-4

Stage 2:
Backbone LR = 2e-5
Head LR = 1e-4

In [99]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage_sbp = "two_stage_head1e-4_then_full_ft_backbone2e-5_head1e-4"

best_model_path_twostage_sbp = "bendr_two_stage_head1e-4_then_full_ft_backbone2e-5_head1e-4_m4_seed42.pth"
history_path_twostage_sbp = "history_bendr_two_stage_head1e-4_then_full_ft_backbone2e-5_head1e-4_m4_seed42.csv"
metrics_path_twostage_sbp = "metrics_bendr_two_stage_head1e-4_then_full_ft_backbone2e-5_head1e-4_m4_seed42.json"

model_twostage_sbp = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage_sbp.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage_sbp(x_dummy)

criterion_twostage_sbp = nn.MSELoss()

print("Two-stage SBP-focused setup ready")

Device: cuda
Two-stage SBP-focused setup ready


In [100]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        param.requires_grad = is_head_parameter(name)


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [101]:
set_stage1_head_only(model_twostage_sbp)

total_params_stage1_sbp, trainable_params_stage1_sbp = count_trainable(model_twostage_sbp)

head_params_stage1_sbp = []

for name, param in model_twostage_sbp.named_parameters():
    if param.requires_grad:
        head_params_stage1_sbp.append(param)

optimizer_stage1_sbp = torch.optim.AdamW(
    head_params_stage1_sbp,
    lr=1e-4,
    weight_decay=1e-2
)

history_twostage_sbp = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1_sbp))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [102]:
import time
import pandas as pd
import torch

num_epochs_stage1_sbp = 10
best_val_loss_twostage_sbp = float("inf")

start_time_twostage_sbp = time.time()

for epoch in range(num_epochs_stage1_sbp):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp,
        train_loader,
        optimizer_stage1_sbp,
        criterion_twostage_sbp,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp,
        val_loader,
        criterion_twostage_sbp,
        device
    )

    history_twostage_sbp.append({
        "method": tuning_method_twostage_sbp,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1_sbp),
        "total_parameters": int(total_params_stage1_sbp),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp:
        best_val_loss_twostage_sbp = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage_sbp,
            "model_state_dict": model_twostage_sbp.state_dict(),
            "optimizer_state_dict": optimizer_stage1_sbp.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp),
            "backbone_lr": 0.0,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1_sbp),
            "total_parameters": int(total_params_stage1_sbp),
            "seed": SEED
        }, best_model_path_twostage_sbp)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_sbp}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/10] | Train Loss: 0.919698 | Val Loss: 0.807301 | Time: 10.46s saved
Stage 1 Epoch [2/10] | Train Loss: 0.788847 | Val Loss: 0.728094 | Time: 10.99s saved
Stage 1 Epoch [3/10] | Train Loss: 0.739586 | Val Loss: 0.684660 | Time: 11.09s saved
Stage 1 Epoch [4/10] | Train Loss: 0.705661 | Val Loss: 0.642009 | Time: 11.25s saved
Stage 1 Epoch [5/10] | Train Loss: 0.681521 | Val Loss: 0.619305 | Time: 11.43s saved
Stage 1 Epoch [6/10] | Train Loss: 0.661593 | Val Loss: 0.599392 | Time: 11.54s saved
Stage 1 Epoch [7/10] | Train Loss: 0.645494 | Val Loss: 0.579669 | Time: 11.51s saved
Stage 1 Epoch [8/10] | Train Loss: 0.630007 | Val Loss: 0.567935 | Time: 11.40s saved
Stage 1 Epoch [9/10] | Train Loss: 0.617737 | Val Loss: 0.548471 | Time: 11.29s saved
Stage 1 Epoch [10/10] | Train Loss: 0.605265 | Val Loss: 0.542690 | Time: 11.23s saved


In [103]:
set_stage2_full_ft(model_twostage_sbp)

total_params_stage2_sbp, trainable_params_stage2_sbp = count_trainable(model_twostage_sbp)

backbone_params_stage2_sbp = []
head_params_stage2_sbp = []

for name, param in model_twostage_sbp.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2_sbp.append(param)
    else:
        backbone_params_stage2_sbp.append(param)

optimizer_stage2_sbp = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2_sbp, "lr": 2e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2_sbp, "lr": 1e-4, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2_sbp))
print("Head params:", sum(p.numel() for p in head_params_stage2_sbp))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [104]:
num_epochs_stage2_sbp = 30

for epoch in range(num_epochs_stage2_sbp):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp,
        train_loader,
        optimizer_stage2_sbp,
        criterion_twostage_sbp,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp,
        val_loader,
        criterion_twostage_sbp,
        device
    )

    history_twostage_sbp.append({
        "method": tuning_method_twostage_sbp,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2_sbp),
        "total_parameters": int(total_params_stage2_sbp),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp:
        best_val_loss_twostage_sbp = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage_sbp,
            "model_state_dict": model_twostage_sbp.state_dict(),
            "optimizer_state_dict": optimizer_stage2_sbp.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp),
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2_sbp),
            "total_parameters": int(total_params_stage2_sbp),
            "seed": SEED
        }, best_model_path_twostage_sbp)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_sbp}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage_sbp).to_csv(history_path_twostage_sbp, index=False)

print("Best val loss:", best_val_loss_twostage_sbp)
print("Total training time:", time.time() - start_time_twostage_sbp)

Stage 2 Epoch [1/30] | Train Loss: 0.392067 | Val Loss: 0.332099 | Time: 14.37s saved
Stage 2 Epoch [2/30] | Train Loss: 0.280591 | Val Loss: 0.257982 | Time: 14.60s saved
Stage 2 Epoch [3/30] | Train Loss: 0.231736 | Val Loss: 0.220943 | Time: 14.82s saved
Stage 2 Epoch [4/30] | Train Loss: 0.203398 | Val Loss: 0.223856 | Time: 13.93s 
Stage 2 Epoch [5/30] | Train Loss: 0.182382 | Val Loss: 0.188556 | Time: 15.26s saved
Stage 2 Epoch [6/30] | Train Loss: 0.165225 | Val Loss: 0.186880 | Time: 15.03s saved
Stage 2 Epoch [7/30] | Train Loss: 0.155092 | Val Loss: 0.178087 | Time: 14.84s saved
Stage 2 Epoch [8/30] | Train Loss: 0.145088 | Val Loss: 0.169200 | Time: 14.84s saved
Stage 2 Epoch [9/30] | Train Loss: 0.137612 | Val Loss: 0.175335 | Time: 13.75s 
Stage 2 Epoch [10/30] | Train Loss: 0.131192 | Val Loss: 0.163204 | Time: 14.90s saved
Stage 2 Epoch [11/30] | Train Loss: 0.125969 | Val Loss: 0.152325 | Time: 14.96s saved
Stage 2 Epoch [12/30] | Train Loss: 0.119484 | Val Loss: 0.156

In [105]:
checkpoint_twostage_sbp = torch.load(best_model_path_twostage_sbp, map_location=device)

model_twostage_sbp.load_state_dict(checkpoint_twostage_sbp["model_state_dict"])
model_twostage_sbp = model_twostage_sbp.to(device)
model_twostage_sbp.eval()

print("Loaded best two-stage SBP-focused model")
print("Best stage:", checkpoint_twostage_sbp["stage"])
print("Best epoch:", checkpoint_twostage_sbp["epoch"])
print("Best val loss:", checkpoint_twostage_sbp["best_val_loss"])

Loaded best two-stage SBP-focused model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 29
Best val loss: 0.125049720959468


In [106]:
y_true_twostage_sbp, y_pred_twostage_sbp = evaluate_model(
    model_twostage_sbp,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage_sbp shape:", y_true_twostage_sbp.shape)
print("y_pred_twostage_sbp shape:", y_pred_twostage_sbp.shape)

print("y_true min/max:", y_true_twostage_sbp.min(), y_true_twostage_sbp.max())
print("y_pred min/max:", y_pred_twostage_sbp.min(), y_pred_twostage_sbp.max())

y_true_twostage_sbp shape: (5064, 625)
y_pred_twostage_sbp shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 48.033203 188.89024


In [107]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage_sbp, dbp_true_twostage_sbp = [], []
sbp_pred_twostage_sbp, dbp_pred_twostage_sbp = [], []

for true_sig, pred_sig in zip(y_true_twostage_sbp, y_pred_twostage_sbp):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage_sbp.append(sbp_t)
    dbp_true_twostage_sbp.append(dbp_t)
    sbp_pred_twostage_sbp.append(sbp_p)
    dbp_pred_twostage_sbp.append(dbp_p)

sbp_true_twostage_sbp = np.array(sbp_true_twostage_sbp)
dbp_true_twostage_sbp = np.array(dbp_true_twostage_sbp)
sbp_pred_twostage_sbp = np.array(sbp_pred_twostage_sbp)
dbp_pred_twostage_sbp = np.array(dbp_pred_twostage_sbp)

sbp_mae_twostage_sbp = mean_absolute_error(sbp_true_twostage_sbp, sbp_pred_twostage_sbp)
dbp_mae_twostage_sbp = mean_absolute_error(dbp_true_twostage_sbp, dbp_pred_twostage_sbp)

sbp_rmse_twostage_sbp = np.sqrt(mean_squared_error(sbp_true_twostage_sbp, sbp_pred_twostage_sbp))
dbp_rmse_twostage_sbp = np.sqrt(mean_squared_error(dbp_true_twostage_sbp, dbp_pred_twostage_sbp))

sbp_r2_twostage_sbp = r2_score(sbp_true_twostage_sbp, sbp_pred_twostage_sbp)
dbp_r2_twostage_sbp = r2_score(dbp_true_twostage_sbp, dbp_pred_twostage_sbp)

avg_mae_twostage_sbp = (sbp_mae_twostage_sbp + dbp_mae_twostage_sbp) / 2

print("BENDR Two-Stage Head 1e-4 Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale")
print("================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage_sbp:.3f}")
print(f"RMSE : {sbp_rmse_twostage_sbp:.3f}")
print(f"R2   : {sbp_r2_twostage_sbp:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage_sbp:.3f}")
print(f"RMSE : {dbp_rmse_twostage_sbp:.3f}")
print(f"R2   : {dbp_r2_twostage_sbp:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage_sbp:.2f}/{dbp_mae_twostage_sbp:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage_sbp:.3f}")

BENDR Two-Stage Head 1e-4 Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale
SBP
MAE  : 5.459
RMSE : 7.736
R2   : 0.7045

DBP
MAE  : 4.104
RMSE : 5.605
R2   : 0.5822

Table format:
5.46/4.10

Average MAE:
4.782


In [108]:
metrics_twostage_sbp = {
    "model": "BENDR",
    "method": tuning_method_twostage_sbp,
    "best_stage": checkpoint_twostage_sbp["stage"],
    "best_epoch": int(checkpoint_twostage_sbp["epoch"]),
    "best_val_loss": float(checkpoint_twostage_sbp["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_twostage_sbp),
    "DBP_MAE": float(dbp_mae_twostage_sbp),
    "AVG_MAE": float(avg_mae_twostage_sbp),
    "SBP_RMSE": float(sbp_rmse_twostage_sbp),
    "DBP_RMSE": float(dbp_rmse_twostage_sbp),
    "SBP_R2": float(sbp_r2_twostage_sbp),
    "DBP_R2": float(dbp_r2_twostage_sbp),
    "table_format": f"{sbp_mae_twostage_sbp:.2f}/{dbp_mae_twostage_sbp:.2f}",
    "stage1_head_lr": 1e-4,
    "stage2_backbone_lr": 2e-5,
    "stage2_head_lr": 1e-4,
    "optimizer": checkpoint_twostage_sbp["optimizer"],
    "trainable_parameters": int(checkpoint_twostage_sbp["trainable_parameters"]),
    "total_parameters": int(checkpoint_twostage_sbp["total_parameters"]),
    "seed": int(checkpoint_twostage_sbp["seed"])
}

with open(metrics_path_twostage_sbp, "w") as f:
    json.dump(metrics_twostage_sbp, f, indent=4)

metrics_twostage_sbp

{'model': 'BENDR',
 'method': 'two_stage_head1e-4_then_full_ft_backbone2e-5_head1e-4',
 'best_stage': 'stage2_full_ft_discriminative_lr',
 'best_epoch': 29,
 'best_val_loss': 0.125049720959468,
 'SBP_MAE': 5.459190368652344,
 'DBP_MAE': 4.104099273681641,
 'AVG_MAE': 4.781644821166992,
 'SBP_RMSE': 7.7362455321739905,
 'DBP_RMSE': 5.604888226690289,
 'SBP_R2': 0.7045280337333679,
 'DBP_R2': 0.5821535587310791,
 'table_format': '5.46/4.10',
 'stage1_head_lr': 0.0001,
 'stage2_backbone_lr': 2e-05,
 'stage2_head_lr': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

# Stage 1:
Backbone frozen
Head LR = 1e-4
Epochs = 5

Stage 2:
Backbone LR = 2e-5
Head LR = 1e-4
Epochs = 30
avoid overfitting

In [109]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage_sbp_5ep = "two_stage_head1e-4_5ep_then_full_ft_backbone2e-5_head1e-4"

best_model_path_twostage_sbp_5ep = "bendr_two_stage_head1e-4_5ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.pth"
history_path_twostage_sbp_5ep = "history_bendr_two_stage_head1e-4_5ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.csv"
metrics_path_twostage_sbp_5ep = "metrics_bendr_two_stage_head1e-4_5ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.json"

model_twostage_sbp_5ep = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage_sbp_5ep.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage_sbp_5ep(x_dummy)

criterion_twostage_sbp_5ep = nn.MSELoss()

print("Two-stage 5-epoch head warmup setup ready")

Device: cuda
Two-stage 5-epoch head warmup setup ready


In [110]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        param.requires_grad = is_head_parameter(name)


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [111]:
set_stage1_head_only(model_twostage_sbp_5ep)

total_params_stage1_sbp_5ep, trainable_params_stage1_sbp_5ep = count_trainable(model_twostage_sbp_5ep)

head_params_stage1_sbp_5ep = []

for name, param in model_twostage_sbp_5ep.named_parameters():
    if param.requires_grad:
        head_params_stage1_sbp_5ep.append(param)

optimizer_stage1_sbp_5ep = torch.optim.AdamW(
    head_params_stage1_sbp_5ep,
    lr=1e-4,
    weight_decay=1e-2
)

history_twostage_sbp_5ep = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1_sbp_5ep))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [112]:
import time
import pandas as pd
import torch

num_epochs_stage1_sbp_5ep = 5
best_val_loss_twostage_sbp_5ep = float("inf")

start_time_twostage_sbp_5ep = time.time()

for epoch in range(num_epochs_stage1_sbp_5ep):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_5ep,
        train_loader,
        optimizer_stage1_sbp_5ep,
        criterion_twostage_sbp_5ep,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_5ep,
        val_loader,
        criterion_twostage_sbp_5ep,
        device
    )

    history_twostage_sbp_5ep.append({
        "method": tuning_method_twostage_sbp_5ep,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1_sbp_5ep),
        "total_parameters": int(total_params_stage1_sbp_5ep),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_5ep:
        best_val_loss_twostage_sbp_5ep = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage_sbp_5ep,
            "model_state_dict": model_twostage_sbp_5ep.state_dict(),
            "optimizer_state_dict": optimizer_stage1_sbp_5ep.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_5ep),
            "backbone_lr": 0.0,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1_sbp_5ep),
            "total_parameters": int(total_params_stage1_sbp_5ep),
            "seed": SEED
        }, best_model_path_twostage_sbp_5ep)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_sbp_5ep}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/5] | Train Loss: 0.924226 | Val Loss: 0.803997 | Time: 10.46s saved
Stage 1 Epoch [2/5] | Train Loss: 0.795977 | Val Loss: 0.755143 | Time: 10.94s saved
Stage 1 Epoch [3/5] | Train Loss: 0.751315 | Val Loss: 0.697083 | Time: 11.04s saved
Stage 1 Epoch [4/5] | Train Loss: 0.723576 | Val Loss: 0.674881 | Time: 11.15s saved
Stage 1 Epoch [5/5] | Train Loss: 0.702454 | Val Loss: 0.652932 | Time: 11.35s saved


In [113]:
set_stage2_full_ft(model_twostage_sbp_5ep)

total_params_stage2_sbp_5ep, trainable_params_stage2_sbp_5ep = count_trainable(model_twostage_sbp_5ep)

backbone_params_stage2_sbp_5ep = []
head_params_stage2_sbp_5ep = []

for name, param in model_twostage_sbp_5ep.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2_sbp_5ep.append(param)
    else:
        backbone_params_stage2_sbp_5ep.append(param)

optimizer_stage2_sbp_5ep = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2_sbp_5ep, "lr": 2e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2_sbp_5ep, "lr": 1e-4, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2_sbp_5ep))
print("Head params:", sum(p.numel() for p in head_params_stage2_sbp_5ep))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [114]:
num_epochs_stage2_sbp_5ep = 30

for epoch in range(num_epochs_stage2_sbp_5ep):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_5ep,
        train_loader,
        optimizer_stage2_sbp_5ep,
        criterion_twostage_sbp_5ep,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_5ep,
        val_loader,
        criterion_twostage_sbp_5ep,
        device
    )

    history_twostage_sbp_5ep.append({
        "method": tuning_method_twostage_sbp_5ep,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2_sbp_5ep),
        "total_parameters": int(total_params_stage2_sbp_5ep),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_5ep:
        best_val_loss_twostage_sbp_5ep = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage_sbp_5ep,
            "model_state_dict": model_twostage_sbp_5ep.state_dict(),
            "optimizer_state_dict": optimizer_stage2_sbp_5ep.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_5ep),
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2_sbp_5ep),
            "total_parameters": int(total_params_stage2_sbp_5ep),
            "seed": SEED
        }, best_model_path_twostage_sbp_5ep)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_sbp_5ep}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage_sbp_5ep).to_csv(history_path_twostage_sbp_5ep, index=False)

print("Best val loss:", best_val_loss_twostage_sbp_5ep)
print("Total training time:", time.time() - start_time_twostage_sbp_5ep)

Stage 2 Epoch [1/30] | Train Loss: 0.440802 | Val Loss: 0.363042 | Time: 14.92s saved
Stage 2 Epoch [2/30] | Train Loss: 0.313514 | Val Loss: 0.281748 | Time: 15.17s saved
Stage 2 Epoch [3/30] | Train Loss: 0.255598 | Val Loss: 0.245632 | Time: 15.04s saved
Stage 2 Epoch [4/30] | Train Loss: 0.221199 | Val Loss: 0.239888 | Time: 14.94s saved
Stage 2 Epoch [5/30] | Train Loss: 0.196237 | Val Loss: 0.203454 | Time: 14.82s saved
Stage 2 Epoch [6/30] | Train Loss: 0.178355 | Val Loss: 0.191540 | Time: 14.77s saved
Stage 2 Epoch [7/30] | Train Loss: 0.164644 | Val Loss: 0.181329 | Time: 14.82s saved
Stage 2 Epoch [8/30] | Train Loss: 0.155096 | Val Loss: 0.175126 | Time: 15.00s saved
Stage 2 Epoch [9/30] | Train Loss: 0.145539 | Val Loss: 0.185371 | Time: 13.89s 
Stage 2 Epoch [10/30] | Train Loss: 0.138631 | Val Loss: 0.164111 | Time: 14.97s saved
Stage 2 Epoch [11/30] | Train Loss: 0.131995 | Val Loss: 0.157275 | Time: 14.91s saved
Stage 2 Epoch [12/30] | Train Loss: 0.128115 | Val Loss: 

In [119]:
import gc
import torch

for var_name in [
    "model_twostage_sbp",
    "model_twostage",
    "model_lwlr",
    "model_discr_2e5_head2e4",
    "model_discr_2e5_head75e5",
    "model_discr_2e5_head5e5",
    "optimizer_stage1_sbp",
    "optimizer_stage2_sbp",
    "optimizer_lwlr"
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 3            |        cudaMalloc retries: 4         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  13147 MiB |  13952 MiB |  52422 GiB |  52409 GiB |
|       from large pool |  13045 MiB |  13839 MiB |  48185 GiB |  48172 GiB |
|       from small pool |    102 MiB |    120 MiB |   4237 GiB |   4237 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  13147 MiB |  13952 MiB |  52422 GiB |  52409 GiB |
|       from large pool |  13045 MiB |  13839 MiB |  48185 GiB |

In [120]:
checkpoint_twostage_sbp_5ep = torch.load(
    best_model_path_twostage_sbp_5ep,
    map_location="cpu"
)

model_twostage_sbp_5ep = BENDR_BP(n_times=625, n_channels_in=1)

model_twostage_sbp_5ep.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].float()
    _ = model_twostage_sbp_5ep(x_dummy)

model_twostage_sbp_5ep.load_state_dict(checkpoint_twostage_sbp_5ep["model_state_dict"])

model_twostage_sbp_5ep = model_twostage_sbp_5ep.to(device)
model_twostage_sbp_5ep.eval()

print("Loaded best two-stage 5-epoch head warmup model")
print("Best stage:", checkpoint_twostage_sbp_5ep["stage"])
print("Best epoch:", checkpoint_twostage_sbp_5ep["epoch"])
print("Best val loss:", checkpoint_twostage_sbp_5ep["best_val_loss"])

Loaded best two-stage 5-epoch head warmup model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 26
Best val loss: 0.12943077995229196


In [121]:
y_true_twostage_sbp_5ep, y_pred_twostage_sbp_5ep = evaluate_model(
    model_twostage_sbp_5ep,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage_sbp_5ep shape:", y_true_twostage_sbp_5ep.shape)
print("y_pred_twostage_sbp_5ep shape:", y_pred_twostage_sbp_5ep.shape)

print("y_true min/max:", y_true_twostage_sbp_5ep.min(), y_true_twostage_sbp_5ep.max())
print("y_pred min/max:", y_pred_twostage_sbp_5ep.min(), y_pred_twostage_sbp_5ep.max())

y_true_twostage_sbp_5ep shape: (5064, 625)
y_pred_twostage_sbp_5ep shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 48.891975 188.89346


In [122]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage_sbp_5ep, dbp_true_twostage_sbp_5ep = [], []
sbp_pred_twostage_sbp_5ep, dbp_pred_twostage_sbp_5ep = [], []

for true_sig, pred_sig in zip(y_true_twostage_sbp_5ep, y_pred_twostage_sbp_5ep):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage_sbp_5ep.append(sbp_t)
    dbp_true_twostage_sbp_5ep.append(dbp_t)
    sbp_pred_twostage_sbp_5ep.append(sbp_p)
    dbp_pred_twostage_sbp_5ep.append(dbp_p)

sbp_true_twostage_sbp_5ep = np.array(sbp_true_twostage_sbp_5ep)
dbp_true_twostage_sbp_5ep = np.array(dbp_true_twostage_sbp_5ep)
sbp_pred_twostage_sbp_5ep = np.array(sbp_pred_twostage_sbp_5ep)
dbp_pred_twostage_sbp_5ep = np.array(dbp_pred_twostage_sbp_5ep)

sbp_mae_twostage_sbp_5ep = mean_absolute_error(sbp_true_twostage_sbp_5ep, sbp_pred_twostage_sbp_5ep)
dbp_mae_twostage_sbp_5ep = mean_absolute_error(dbp_true_twostage_sbp_5ep, dbp_pred_twostage_sbp_5ep)

sbp_rmse_twostage_sbp_5ep = np.sqrt(mean_squared_error(sbp_true_twostage_sbp_5ep, sbp_pred_twostage_sbp_5ep))
dbp_rmse_twostage_sbp_5ep = np.sqrt(mean_squared_error(dbp_true_twostage_sbp_5ep, dbp_pred_twostage_sbp_5ep))

sbp_r2_twostage_sbp_5ep = r2_score(sbp_true_twostage_sbp_5ep, sbp_pred_twostage_sbp_5ep)
dbp_r2_twostage_sbp_5ep = r2_score(dbp_true_twostage_sbp_5ep, dbp_pred_twostage_sbp_5ep)

avg_mae_twostage_sbp_5ep = (sbp_mae_twostage_sbp_5ep + dbp_mae_twostage_sbp_5ep) / 2

print("BENDR Two-Stage Head 1e-4 5ep Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale")
print("=====================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage_sbp_5ep:.3f}")
print(f"RMSE : {sbp_rmse_twostage_sbp_5ep:.3f}")
print(f"R2   : {sbp_r2_twostage_sbp_5ep:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage_sbp_5ep:.3f}")
print(f"RMSE : {dbp_rmse_twostage_sbp_5ep:.3f}")
print(f"R2   : {dbp_r2_twostage_sbp_5ep:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage_sbp_5ep:.2f}/{dbp_mae_twostage_sbp_5ep:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage_sbp_5ep:.3f}")

BENDR Two-Stage Head 1e-4 5ep Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale
SBP
MAE  : 5.676
RMSE : 7.943
R2   : 0.6885

DBP
MAE  : 4.203
RMSE : 5.781
R2   : 0.5554

Table format:
5.68/4.20

Average MAE:
4.940


In [124]:
import gc
import torch

for var_name in list(globals().keys()):
    if (
        var_name.startswith("model_")
        or var_name.startswith("optimizer_")
        or var_name.startswith("checkpoint_")
    ):
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared")
print(torch.cuda.memory_summary())

GPU memory cleared
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 4            |        cudaMalloc retries: 5         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   3313 MiB |  13952 MiB |  52437 GiB |  52434 GiB |
|       from large pool |   3298 MiB |  13839 MiB |  48198 GiB |  48195 GiB |
|       from small pool |     15 MiB |    120 MiB |   4238 GiB |   4238 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   3313 MiB |  13952 MiB |  52437 GiB |  52434 GiB |
|       from large pool |   3298 MiB |  13839

In [125]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage_sbp_15ep = "two_stage_head1e-4_15ep_then_full_ft_backbone2e-5_head1e-4"

best_model_path_twostage_sbp_15ep = "bendr_two_stage_head1e-4_15ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.pth"
history_path_twostage_sbp_15ep = "history_bendr_two_stage_head1e-4_15ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.csv"
metrics_path_twostage_sbp_15ep = "metrics_bendr_two_stage_head1e-4_15ep_then_full_ft_backbone2e-5_head1e-4_m4_seed42.json"

model_twostage_sbp_15ep = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage_sbp_15ep.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage_sbp_15ep(x_dummy)

criterion_twostage_sbp_15ep = nn.MSELoss()

print("Two-stage 15-epoch head warmup setup ready")

Device: cuda
Two-stage 15-epoch head warmup setup ready


In [126]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        param.requires_grad = is_head_parameter(name)


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [127]:
set_stage1_head_only(model_twostage_sbp_15ep)

total_params_stage1_sbp_15ep, trainable_params_stage1_sbp_15ep = count_trainable(model_twostage_sbp_15ep)

head_params_stage1_sbp_15ep = []

for name, param in model_twostage_sbp_15ep.named_parameters():
    if param.requires_grad:
        head_params_stage1_sbp_15ep.append(param)

optimizer_stage1_sbp_15ep = torch.optim.AdamW(
    head_params_stage1_sbp_15ep,
    lr=1e-4,
    weight_decay=1e-2
)

history_twostage_sbp_15ep = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1_sbp_15ep))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [128]:
import time
import pandas as pd
import torch

num_epochs_stage1_sbp_15ep = 15
best_val_loss_twostage_sbp_15ep = float("inf")

start_time_twostage_sbp_15ep = time.time()

for epoch in range(num_epochs_stage1_sbp_15ep):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_15ep,
        train_loader,
        optimizer_stage1_sbp_15ep,
        criterion_twostage_sbp_15ep,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_15ep,
        val_loader,
        criterion_twostage_sbp_15ep,
        device
    )

    history_twostage_sbp_15ep.append({
        "method": tuning_method_twostage_sbp_15ep,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1_sbp_15ep),
        "total_parameters": int(total_params_stage1_sbp_15ep),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_15ep:
        best_val_loss_twostage_sbp_15ep = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage_sbp_15ep,
            "model_state_dict": model_twostage_sbp_15ep.state_dict(),
            "optimizer_state_dict": optimizer_stage1_sbp_15ep.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_15ep),
            "backbone_lr": 0.0,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1_sbp_15ep),
            "total_parameters": int(total_params_stage1_sbp_15ep),
            "seed": SEED
        }, best_model_path_twostage_sbp_15ep)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_sbp_15ep}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/15] | Train Loss: 0.914295 | Val Loss: 0.805011 | Time: 10.42s saved
Stage 1 Epoch [2/15] | Train Loss: 0.780185 | Val Loss: 0.723306 | Time: 10.87s saved
Stage 1 Epoch [3/15] | Train Loss: 0.728489 | Val Loss: 0.666050 | Time: 11.02s saved
Stage 1 Epoch [4/15] | Train Loss: 0.692849 | Val Loss: 0.638804 | Time: 11.13s saved
Stage 1 Epoch [5/15] | Train Loss: 0.670034 | Val Loss: 0.613961 | Time: 11.28s saved
Stage 1 Epoch [6/15] | Train Loss: 0.652871 | Val Loss: 0.594835 | Time: 11.46s saved
Stage 1 Epoch [7/15] | Train Loss: 0.638980 | Val Loss: 0.581023 | Time: 11.50s saved
Stage 1 Epoch [8/15] | Train Loss: 0.626470 | Val Loss: 0.572718 | Time: 11.40s saved
Stage 1 Epoch [9/15] | Train Loss: 0.615757 | Val Loss: 0.557433 | Time: 11.32s saved
Stage 1 Epoch [10/15] | Train Loss: 0.605496 | Val Loss: 0.553727 | Time: 11.31s saved
Stage 1 Epoch [11/15] | Train Loss: 0.597188 | Val Loss: 0.538976 | Time: 11.24s saved
Stage 1 Epoch [12/15] | Train Loss: 0.586441 | Val L

In [129]:
set_stage2_full_ft(model_twostage_sbp_15ep)

total_params_stage2_sbp_15ep, trainable_params_stage2_sbp_15ep = count_trainable(model_twostage_sbp_15ep)

backbone_params_stage2_sbp_15ep = []
head_params_stage2_sbp_15ep = []

for name, param in model_twostage_sbp_15ep.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2_sbp_15ep.append(param)
    else:
        backbone_params_stage2_sbp_15ep.append(param)

optimizer_stage2_sbp_15ep = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2_sbp_15ep, "lr": 2e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2_sbp_15ep, "lr": 1e-4, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2_sbp_15ep))
print("Head params:", sum(p.numel() for p in head_params_stage2_sbp_15ep))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [130]:
num_epochs_stage2_sbp_15ep = 30

for epoch in range(num_epochs_stage2_sbp_15ep):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_15ep,
        train_loader,
        optimizer_stage2_sbp_15ep,
        criterion_twostage_sbp_15ep,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_15ep,
        val_loader,
        criterion_twostage_sbp_15ep,
        device
    )

    history_twostage_sbp_15ep.append({
        "method": tuning_method_twostage_sbp_15ep,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 2e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2_sbp_15ep),
        "total_parameters": int(total_params_stage2_sbp_15ep),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_15ep:
        best_val_loss_twostage_sbp_15ep = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage_sbp_15ep,
            "model_state_dict": model_twostage_sbp_15ep.state_dict(),
            "optimizer_state_dict": optimizer_stage2_sbp_15ep.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_15ep),
            "backbone_lr": 2e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2_sbp_15ep),
            "total_parameters": int(total_params_stage2_sbp_15ep),
            "seed": SEED
        }, best_model_path_twostage_sbp_15ep)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_sbp_15ep}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage_sbp_15ep).to_csv(history_path_twostage_sbp_15ep, index=False)

print("Best val loss:", best_val_loss_twostage_sbp_15ep)
print("Total training time:", time.time() - start_time_twostage_sbp_15ep)

Stage 2 Epoch [1/30] | Train Loss: 0.369266 | Val Loss: 0.300705 | Time: 15.12s saved
Stage 2 Epoch [2/30] | Train Loss: 0.267711 | Val Loss: 0.253599 | Time: 14.92s saved
Stage 2 Epoch [3/30] | Train Loss: 0.223280 | Val Loss: 0.232942 | Time: 14.93s saved
Stage 2 Epoch [4/30] | Train Loss: 0.196761 | Val Loss: 0.198260 | Time: 15.04s saved
Stage 2 Epoch [5/30] | Train Loss: 0.177844 | Val Loss: 0.187991 | Time: 15.12s saved
Stage 2 Epoch [6/30] | Train Loss: 0.163808 | Val Loss: 0.187451 | Time: 15.11s saved
Stage 2 Epoch [7/30] | Train Loss: 0.152866 | Val Loss: 0.172567 | Time: 15.02s saved
Stage 2 Epoch [8/30] | Train Loss: 0.143563 | Val Loss: 0.166585 | Time: 15.07s saved
Stage 2 Epoch [9/30] | Train Loss: 0.136168 | Val Loss: 0.163213 | Time: 15.07s saved
Stage 2 Epoch [10/30] | Train Loss: 0.130420 | Val Loss: 0.168281 | Time: 13.86s 
Stage 2 Epoch [11/30] | Train Loss: 0.123562 | Val Loss: 0.155249 | Time: 15.11s saved
Stage 2 Epoch [12/30] | Train Loss: 0.118221 | Val Loss: 

In [131]:
checkpoint_twostage_sbp_15ep = torch.load(
    best_model_path_twostage_sbp_15ep,
    map_location="cpu"
)

model_twostage_sbp_15ep = BENDR_BP(n_times=625, n_channels_in=1)

model_twostage_sbp_15ep.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].float()
    _ = model_twostage_sbp_15ep(x_dummy)

model_twostage_sbp_15ep.load_state_dict(checkpoint_twostage_sbp_15ep["model_state_dict"])

model_twostage_sbp_15ep = model_twostage_sbp_15ep.to(device)
model_twostage_sbp_15ep.eval()

print("Loaded best two-stage 15-epoch head warmup model")
print("Best stage:", checkpoint_twostage_sbp_15ep["stage"])
print("Best epoch:", checkpoint_twostage_sbp_15ep["epoch"])
print("Best val loss:", checkpoint_twostage_sbp_15ep["best_val_loss"])

Loaded best two-stage 15-epoch head warmup model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 30
Best val loss: 0.12590843802159346


In [132]:
y_true_twostage_sbp_15ep, y_pred_twostage_sbp_15ep = evaluate_model(
    model_twostage_sbp_15ep,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage_sbp_15ep shape:", y_true_twostage_sbp_15ep.shape)
print("y_pred_twostage_sbp_15ep shape:", y_pred_twostage_sbp_15ep.shape)

print("y_true min/max:", y_true_twostage_sbp_15ep.min(), y_true_twostage_sbp_15ep.max())
print("y_pred min/max:", y_pred_twostage_sbp_15ep.min(), y_pred_twostage_sbp_15ep.max())

y_true_twostage_sbp_15ep shape: (5064, 625)
y_pred_twostage_sbp_15ep shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 48.54063 184.62556


In [133]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage_sbp_15ep, dbp_true_twostage_sbp_15ep = [], []
sbp_pred_twostage_sbp_15ep, dbp_pred_twostage_sbp_15ep = [], []

for true_sig, pred_sig in zip(y_true_twostage_sbp_15ep, y_pred_twostage_sbp_15ep):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage_sbp_15ep.append(sbp_t)
    dbp_true_twostage_sbp_15ep.append(dbp_t)
    sbp_pred_twostage_sbp_15ep.append(sbp_p)
    dbp_pred_twostage_sbp_15ep.append(dbp_p)

sbp_true_twostage_sbp_15ep = np.array(sbp_true_twostage_sbp_15ep)
dbp_true_twostage_sbp_15ep = np.array(dbp_true_twostage_sbp_15ep)
sbp_pred_twostage_sbp_15ep = np.array(sbp_pred_twostage_sbp_15ep)
dbp_pred_twostage_sbp_15ep = np.array(dbp_pred_twostage_sbp_15ep)

sbp_mae_twostage_sbp_15ep = mean_absolute_error(sbp_true_twostage_sbp_15ep, sbp_pred_twostage_sbp_15ep)
dbp_mae_twostage_sbp_15ep = mean_absolute_error(dbp_true_twostage_sbp_15ep, dbp_pred_twostage_sbp_15ep)

sbp_rmse_twostage_sbp_15ep = np.sqrt(mean_squared_error(sbp_true_twostage_sbp_15ep, sbp_pred_twostage_sbp_15ep))
dbp_rmse_twostage_sbp_15ep = np.sqrt(mean_squared_error(dbp_true_twostage_sbp_15ep, dbp_pred_twostage_sbp_15ep))

sbp_r2_twostage_sbp_15ep = r2_score(sbp_true_twostage_sbp_15ep, sbp_pred_twostage_sbp_15ep)
dbp_r2_twostage_sbp_15ep = r2_score(dbp_true_twostage_sbp_15ep, dbp_pred_twostage_sbp_15ep)

avg_mae_twostage_sbp_15ep = (sbp_mae_twostage_sbp_15ep + dbp_mae_twostage_sbp_15ep) / 2

print("BENDR Two-Stage Head 1e-4 15ep Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale")
print("======================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage_sbp_15ep:.3f}")
print(f"RMSE : {sbp_rmse_twostage_sbp_15ep:.3f}")
print(f"R2   : {sbp_r2_twostage_sbp_15ep:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage_sbp_15ep:.3f}")
print(f"RMSE : {dbp_rmse_twostage_sbp_15ep:.3f}")
print(f"R2   : {dbp_r2_twostage_sbp_15ep:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage_sbp_15ep:.2f}/{dbp_mae_twostage_sbp_15ep:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage_sbp_15ep:.3f}")

BENDR Two-Stage Head 1e-4 15ep Then Full FT Backbone 2e-5 Head 1e-4 Results — Real mmHg Scale
SBP
MAE  : 5.788
RMSE : 8.199
R2   : 0.6681

DBP
MAE  : 4.232
RMSE : 5.710
R2   : 0.5664

Table format:
5.79/4.23

Average MAE:
5.010


In [134]:
metrics_twostage_sbp_15ep = {
    "model": "BENDR",
    "method": tuning_method_twostage_sbp_15ep,
    "best_stage": checkpoint_twostage_sbp_15ep["stage"],
    "best_epoch": int(checkpoint_twostage_sbp_15ep["epoch"]),
    "best_val_loss": float(checkpoint_twostage_sbp_15ep["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_twostage_sbp_15ep),
    "DBP_MAE": float(dbp_mae_twostage_sbp_15ep),
    "AVG_MAE": float(avg_mae_twostage_sbp_15ep),
    "SBP_RMSE": float(sbp_rmse_twostage_sbp_15ep),
    "DBP_RMSE": float(dbp_rmse_twostage_sbp_15ep),
    "SBP_R2": float(sbp_r2_twostage_sbp_15ep),
    "DBP_R2": float(dbp_r2_twostage_sbp_15ep),
    "table_format": f"{sbp_mae_twostage_sbp_15ep:.2f}/{dbp_mae_twostage_sbp_15ep:.2f}",
    "stage1_epochs": 15,
    "stage1_head_lr": 1e-4,
    "stage2_epochs": 30,
    "stage2_backbone_lr": 2e-5,
    "stage2_head_lr": 1e-4,
    "optimizer": checkpoint_twostage_sbp_15ep["optimizer"],
    "trainable_parameters": int(checkpoint_twostage_sbp_15ep["trainable_parameters"]),
    "total_parameters": int(checkpoint_twostage_sbp_15ep["total_parameters"]),
    "seed": int(checkpoint_twostage_sbp_15ep["seed"])
}

with open(metrics_path_twostage_sbp_15ep, "w") as f:
    json.dump(metrics_twostage_sbp_15ep, f, indent=4)

metrics_twostage_sbp_15ep

{'model': 'BENDR',
 'method': 'two_stage_head1e-4_15ep_then_full_ft_backbone2e-5_head1e-4',
 'best_stage': 'stage2_full_ft_discriminative_lr',
 'best_epoch': 30,
 'best_val_loss': 0.12590843802159346,
 'SBP_MAE': 5.7875752449035645,
 'DBP_MAE': 4.231672763824463,
 'AVG_MAE': 5.009624004364014,
 'SBP_RMSE': 8.198937268125677,
 'DBP_RMSE': 5.709868658529393,
 'SBP_R2': 0.6681277751922607,
 'DBP_R2': 0.5663542747497559,
 'table_format': '5.79/4.23',
 'stage1_epochs': 15,
 'stage1_head_lr': 0.0001,
 'stage2_epochs': 30,
 'stage2_backbone_lr': 2e-05,
 'stage2_head_lr': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

In [135]:
import gc
import torch

for var_name in list(globals().keys()):
    if (
        var_name.startswith("model_")
        or var_name.startswith("optimizer_")
        or var_name.startswith("checkpoint_")
    ):
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage_sbp_b3e5 = "two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head1e-4"

best_model_path_twostage_sbp_b3e5 = "bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head1e-4_m4_seed42.pth"
history_path_twostage_sbp_b3e5 = "history_bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head1e-4_m4_seed42.csv"
metrics_path_twostage_sbp_b3e5 = "metrics_bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head1e-4_m4_seed42.json"

model_twostage_sbp_b3e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage_sbp_b3e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage_sbp_b3e5(x_dummy)

criterion_twostage_sbp_b3e5 = nn.MSELoss()

print("Two-stage backbone 3e-5 setup ready")

Device: cuda
Two-stage backbone 3e-5 setup ready


In [136]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        param.requires_grad = is_head_parameter(name)


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [138]:
set_stage1_head_only(model_twostage_sbp_b3e5)

total_params_stage1_sbp_b3e5, trainable_params_stage1_sbp_b3e5 = count_trainable(model_twostage_sbp_b3e5)

head_params_stage1_sbp_b3e5 = []

for name, param in model_twostage_sbp_b3e5.named_parameters():
    if param.requires_grad:
        head_params_stage1_sbp_b3e5.append(param)

optimizer_stage1_sbp_b3e5 = torch.optim.AdamW(
    head_params_stage1_sbp_b3e5,
    lr=1e-4,
    weight_decay=1e-2
)

history_twostage_sbp_b3e5 = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1_sbp_b3e5))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [139]:
import time
import pandas as pd
import torch

num_epochs_stage1_sbp_b3e5 = 10
best_val_loss_twostage_sbp_b3e5 = float("inf")

start_time_twostage_sbp_b3e5 = time.time()

for epoch in range(num_epochs_stage1_sbp_b3e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_b3e5,
        train_loader,
        optimizer_stage1_sbp_b3e5,
        criterion_twostage_sbp_b3e5,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_b3e5,
        val_loader,
        criterion_twostage_sbp_b3e5,
        device
    )

    history_twostage_sbp_b3e5.append({
        "method": tuning_method_twostage_sbp_b3e5,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1_sbp_b3e5),
        "total_parameters": int(total_params_stage1_sbp_b3e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_b3e5:
        best_val_loss_twostage_sbp_b3e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage_sbp_b3e5,
            "model_state_dict": model_twostage_sbp_b3e5.state_dict(),
            "optimizer_state_dict": optimizer_stage1_sbp_b3e5.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_b3e5),
            "backbone_lr": 0.0,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1_sbp_b3e5),
            "total_parameters": int(total_params_stage1_sbp_b3e5),
            "seed": SEED
        }, best_model_path_twostage_sbp_b3e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_sbp_b3e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/10] | Train Loss: 0.900320 | Val Loss: 0.798457 | Time: 10.43s saved
Stage 1 Epoch [2/10] | Train Loss: 0.779926 | Val Loss: 0.743457 | Time: 11.02s saved
Stage 1 Epoch [3/10] | Train Loss: 0.738593 | Val Loss: 0.680086 | Time: 11.14s saved
Stage 1 Epoch [4/10] | Train Loss: 0.702387 | Val Loss: 0.647188 | Time: 11.19s saved
Stage 1 Epoch [5/10] | Train Loss: 0.677625 | Val Loss: 0.617528 | Time: 11.46s saved
Stage 1 Epoch [6/10] | Train Loss: 0.656732 | Val Loss: 0.601040 | Time: 11.43s saved
Stage 1 Epoch [7/10] | Train Loss: 0.639655 | Val Loss: 0.580404 | Time: 11.45s saved
Stage 1 Epoch [8/10] | Train Loss: 0.623452 | Val Loss: 0.571478 | Time: 11.29s saved
Stage 1 Epoch [9/10] | Train Loss: 0.610968 | Val Loss: 0.545834 | Time: 11.25s saved
Stage 1 Epoch [10/10] | Train Loss: 0.599625 | Val Loss: 0.541374 | Time: 11.26s saved


In [140]:
set_stage2_full_ft(model_twostage_sbp_b3e5)

total_params_stage2_sbp_b3e5, trainable_params_stage2_sbp_b3e5 = count_trainable(model_twostage_sbp_b3e5)

backbone_params_stage2_sbp_b3e5 = []
head_params_stage2_sbp_b3e5 = []

for name, param in model_twostage_sbp_b3e5.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2_sbp_b3e5.append(param)
    else:
        backbone_params_stage2_sbp_b3e5.append(param)

optimizer_stage2_sbp_b3e5 = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2_sbp_b3e5, "lr": 3e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2_sbp_b3e5, "lr": 1e-4, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2_sbp_b3e5))
print("Head params:", sum(p.numel() for p in head_params_stage2_sbp_b3e5))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [141]:
num_epochs_stage2_sbp_b3e5 = 30

for epoch in range(num_epochs_stage2_sbp_b3e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_sbp_b3e5,
        train_loader,
        optimizer_stage2_sbp_b3e5,
        criterion_twostage_sbp_b3e5,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_sbp_b3e5,
        val_loader,
        criterion_twostage_sbp_b3e5,
        device
    )

    history_twostage_sbp_b3e5.append({
        "method": tuning_method_twostage_sbp_b3e5,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 3e-5,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2_sbp_b3e5),
        "total_parameters": int(total_params_stage2_sbp_b3e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_sbp_b3e5:
        best_val_loss_twostage_sbp_b3e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage_sbp_b3e5,
            "model_state_dict": model_twostage_sbp_b3e5.state_dict(),
            "optimizer_state_dict": optimizer_stage2_sbp_b3e5.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_sbp_b3e5),
            "backbone_lr": 3e-5,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2_sbp_b3e5),
            "total_parameters": int(total_params_stage2_sbp_b3e5),
            "seed": SEED
        }, best_model_path_twostage_sbp_b3e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_sbp_b3e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage_sbp_b3e5).to_csv(history_path_twostage_sbp_b3e5, index=False)

print("Best val loss:", best_val_loss_twostage_sbp_b3e5)
print("Total training time:", time.time() - start_time_twostage_sbp_b3e5)

Stage 2 Epoch [1/30] | Train Loss: 0.406083 | Val Loss: 0.327318 | Time: 15.03s saved
Stage 2 Epoch [2/30] | Train Loss: 0.291281 | Val Loss: 0.271044 | Time: 15.09s saved
Stage 2 Epoch [3/30] | Train Loss: 0.240318 | Val Loss: 0.231269 | Time: 15.17s saved
Stage 2 Epoch [4/30] | Train Loss: 0.208803 | Val Loss: 0.204293 | Time: 15.03s saved
Stage 2 Epoch [5/30] | Train Loss: 0.185896 | Val Loss: 0.199146 | Time: 15.06s saved
Stage 2 Epoch [6/30] | Train Loss: 0.168793 | Val Loss: 0.187673 | Time: 15.02s saved
Stage 2 Epoch [7/30] | Train Loss: 0.157354 | Val Loss: 0.174265 | Time: 15.06s saved
Stage 2 Epoch [8/30] | Train Loss: 0.148149 | Val Loss: 0.168329 | Time: 15.05s saved
Stage 2 Epoch [9/30] | Train Loss: 0.140998 | Val Loss: 0.167719 | Time: 15.14s saved
Stage 2 Epoch [10/30] | Train Loss: 0.134141 | Val Loss: 0.163668 | Time: 15.12s saved
Stage 2 Epoch [11/30] | Train Loss: 0.126982 | Val Loss: 0.160690 | Time: 15.12s saved
Stage 2 Epoch [12/30] | Train Loss: 0.122050 | Val L

In [142]:
checkpoint_twostage_sbp_b3e5 = torch.load(
    best_model_path_twostage_sbp_b3e5,
    map_location="cpu"
)

model_twostage_sbp_b3e5 = BENDR_BP(n_times=625, n_channels_in=1)

model_twostage_sbp_b3e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].float()
    _ = model_twostage_sbp_b3e5(x_dummy)

model_twostage_sbp_b3e5.load_state_dict(checkpoint_twostage_sbp_b3e5["model_state_dict"])

model_twostage_sbp_b3e5 = model_twostage_sbp_b3e5.to(device)
model_twostage_sbp_b3e5.eval()

print("Loaded best two-stage backbone 3e-5 model")
print("Best stage:", checkpoint_twostage_sbp_b3e5["stage"])
print("Best epoch:", checkpoint_twostage_sbp_b3e5["epoch"])
print("Best val loss:", checkpoint_twostage_sbp_b3e5["best_val_loss"])

Loaded best two-stage backbone 3e-5 model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 27
Best val loss: 0.1302157832763425


In [143]:
y_true_twostage_sbp_b3e5, y_pred_twostage_sbp_b3e5 = evaluate_model(
    model_twostage_sbp_b3e5,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage_sbp_b3e5 shape:", y_true_twostage_sbp_b3e5.shape)
print("y_pred_twostage_sbp_b3e5 shape:", y_pred_twostage_sbp_b3e5.shape)

print("y_true min/max:", y_true_twostage_sbp_b3e5.min(), y_true_twostage_sbp_b3e5.max())
print("y_pred min/max:", y_pred_twostage_sbp_b3e5.min(), y_pred_twostage_sbp_b3e5.max())

y_true_twostage_sbp_b3e5 shape: (5064, 625)
y_pred_twostage_sbp_b3e5 shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 46.253098 184.21739


In [144]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage_sbp_b3e5, dbp_true_twostage_sbp_b3e5 = [], []
sbp_pred_twostage_sbp_b3e5, dbp_pred_twostage_sbp_b3e5 = [], []

for true_sig, pred_sig in zip(y_true_twostage_sbp_b3e5, y_pred_twostage_sbp_b3e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage_sbp_b3e5.append(sbp_t)
    dbp_true_twostage_sbp_b3e5.append(dbp_t)
    sbp_pred_twostage_sbp_b3e5.append(sbp_p)
    dbp_pred_twostage_sbp_b3e5.append(dbp_p)

sbp_true_twostage_sbp_b3e5 = np.array(sbp_true_twostage_sbp_b3e5)
dbp_true_twostage_sbp_b3e5 = np.array(dbp_true_twostage_sbp_b3e5)
sbp_pred_twostage_sbp_b3e5 = np.array(sbp_pred_twostage_sbp_b3e5)
dbp_pred_twostage_sbp_b3e5 = np.array(dbp_pred_twostage_sbp_b3e5)

sbp_mae_twostage_sbp_b3e5 = mean_absolute_error(sbp_true_twostage_sbp_b3e5, sbp_pred_twostage_sbp_b3e5)
dbp_mae_twostage_sbp_b3e5 = mean_absolute_error(dbp_true_twostage_sbp_b3e5, dbp_pred_twostage_sbp_b3e5)

sbp_rmse_twostage_sbp_b3e5 = np.sqrt(mean_squared_error(sbp_true_twostage_sbp_b3e5, sbp_pred_twostage_sbp_b3e5))
dbp_rmse_twostage_sbp_b3e5 = np.sqrt(mean_squared_error(dbp_true_twostage_sbp_b3e5, dbp_pred_twostage_sbp_b3e5))

sbp_r2_twostage_sbp_b3e5 = r2_score(sbp_true_twostage_sbp_b3e5, sbp_pred_twostage_sbp_b3e5)
dbp_r2_twostage_sbp_b3e5 = r2_score(dbp_true_twostage_sbp_b3e5, dbp_pred_twostage_sbp_b3e5)

avg_mae_twostage_sbp_b3e5 = (sbp_mae_twostage_sbp_b3e5 + dbp_mae_twostage_sbp_b3e5) / 2

print("BENDR Two-Stage Head 1e-4 10ep Then Full FT Backbone 3e-5 Head 1e-4 Results — Real mmHg Scale")
print("======================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage_sbp_b3e5:.3f}")
print(f"RMSE : {sbp_rmse_twostage_sbp_b3e5:.3f}")
print(f"R2   : {sbp_r2_twostage_sbp_b3e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage_sbp_b3e5:.3f}")
print(f"RMSE : {dbp_rmse_twostage_sbp_b3e5:.3f}")
print(f"R2   : {dbp_r2_twostage_sbp_b3e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage_sbp_b3e5:.2f}/{dbp_mae_twostage_sbp_b3e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage_sbp_b3e5:.3f}")

BENDR Two-Stage Head 1e-4 10ep Then Full FT Backbone 3e-5 Head 1e-4 Results — Real mmHg Scale
SBP
MAE  : 5.426
RMSE : 7.719
R2   : 0.7058

DBP
MAE  : 4.300
RMSE : 5.868
R2   : 0.5420

Table format:
5.43/4.30

Average MAE:
4.863


In [145]:
metrics_twostage_sbp_b3e5 = {
    "model": "BENDR",
    "method": tuning_method_twostage_sbp_b3e5,
    "best_stage": checkpoint_twostage_sbp_b3e5["stage"],
    "best_epoch": int(checkpoint_twostage_sbp_b3e5["epoch"]),
    "best_val_loss": float(checkpoint_twostage_sbp_b3e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_twostage_sbp_b3e5),
    "DBP_MAE": float(dbp_mae_twostage_sbp_b3e5),
    "AVG_MAE": float(avg_mae_twostage_sbp_b3e5),
    "SBP_RMSE": float(sbp_rmse_twostage_sbp_b3e5),
    "DBP_RMSE": float(dbp_rmse_twostage_sbp_b3e5),
    "SBP_R2": float(sbp_r2_twostage_sbp_b3e5),
    "DBP_R2": float(dbp_r2_twostage_sbp_b3e5),
    "table_format": f"{sbp_mae_twostage_sbp_b3e5:.2f}/{dbp_mae_twostage_sbp_b3e5:.2f}",
    "stage1_epochs": 10,
    "stage1_head_lr": 1e-4,
    "stage2_epochs": 30,
    "stage2_backbone_lr": 3e-5,
    "stage2_head_lr": 1e-4,
    "optimizer": checkpoint_twostage_sbp_b3e5["optimizer"],
    "trainable_parameters": int(checkpoint_twostage_sbp_b3e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_twostage_sbp_b3e5["total_parameters"]),
    "seed": int(checkpoint_twostage_sbp_b3e5["seed"])
}

with open(metrics_path_twostage_sbp_b3e5, "w") as f:
    json.dump(metrics_twostage_sbp_b3e5, f, indent=4)

metrics_twostage_sbp_b3e5

{'model': 'BENDR',
 'method': 'two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head1e-4',
 'best_stage': 'stage2_full_ft_discriminative_lr',
 'best_epoch': 27,
 'best_val_loss': 0.1302157832763425,
 'SBP_MAE': 5.426015377044678,
 'DBP_MAE': 4.299540996551514,
 'AVG_MAE': 4.862778186798096,
 'SBP_RMSE': 7.719172786608775,
 'DBP_RMSE': 5.868078496864312,
 'SBP_R2': 0.7058306932449341,
 'DBP_R2': 0.541990339756012,
 'table_format': '5.43/4.30',
 'stage1_epochs': 10,
 'stage1_head_lr': 0.0001,
 'stage2_epochs': 30,
 'stage2_backbone_lr': 3e-05,
 'stage2_head_lr': 0.0001,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}

Stage 1: Head-only, 10 epochs, Head LR = 1e-4
Stage 2: Full FT, Backbone LR = 3e-5, Head LR = 5e-5

In [146]:
import gc
import torch

for var_name in list(globals().keys()):
    if (
        var_name.startswith("model_")
        or var_name.startswith("optimizer_")
        or var_name.startswith("checkpoint_")
    ):
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tuning_method_twostage_b3e5_h5e5 = "two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head5e-5"

best_model_path_twostage_b3e5_h5e5 = "bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head5e-5_m4_seed42.pth"
history_path_twostage_b3e5_h5e5 = "history_bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head5e-5_m4_seed42.csv"
metrics_path_twostage_b3e5_h5e5 = "metrics_bendr_two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head5e-5_m4_seed42.json"

model_twostage_b3e5_h5e5 = BENDR_BP(n_times=625, n_channels_in=1).to(device)

model_twostage_b3e5_h5e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].to(device).float()
    _ = model_twostage_b3e5_h5e5(x_dummy)

criterion_twostage_b3e5_h5e5 = nn.MSELoss()

print("Two-stage backbone 3e-5 head 5e-5 setup ready")

Device: cuda
Two-stage backbone 3e-5 head 5e-5 setup ready


In [147]:
def is_head_parameter(name):
    return not name.startswith("bendr.")


def set_stage1_head_only(model):
    for name, param in model.named_parameters():
        param.requires_grad = is_head_parameter(name)


def set_stage2_full_ft(model):
    for param in model.parameters():
        param.requires_grad = True


def count_trainable(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total parameters:", total_params)
    print("Trainable parameters:", trainable_params)
    print("Trainable percentage:", 100 * trainable_params / total_params)
    return total_params, trainable_params

In [148]:
set_stage1_head_only(model_twostage_b3e5_h5e5)

total_params_stage1_b3e5_h5e5, trainable_params_stage1_b3e5_h5e5 = count_trainable(model_twostage_b3e5_h5e5)

head_params_stage1_b3e5_h5e5 = []

for name, param in model_twostage_b3e5_h5e5.named_parameters():
    if param.requires_grad:
        head_params_stage1_b3e5_h5e5.append(param)

optimizer_stage1_b3e5_h5e5 = torch.optim.AdamW(
    head_params_stage1_b3e5_h5e5,
    lr=1e-4,
    weight_decay=1e-2
)

history_twostage_b3e5_h5e5 = []

print("Stage 1 optimizer ready")
print("Head params:", sum(p.numel() for p in head_params_stage1_b3e5_h5e5))

Total parameters: 157970996
Trainable parameters: 829434
Trainable percentage: 0.5250546119238243
Stage 1 optimizer ready
Head params: 829434


In [149]:
import time
import pandas as pd
import torch

num_epochs_stage1_b3e5_h5e5 = 10
best_val_loss_twostage_b3e5_h5e5 = float("inf")

start_time_twostage_b3e5_h5e5 = time.time()

for epoch in range(num_epochs_stage1_b3e5_h5e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_b3e5_h5e5,
        train_loader,
        optimizer_stage1_b3e5_h5e5,
        criterion_twostage_b3e5_h5e5,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_b3e5_h5e5,
        val_loader,
        criterion_twostage_b3e5_h5e5,
        device
    )

    history_twostage_b3e5_h5e5.append({
        "method": tuning_method_twostage_b3e5_h5e5,
        "stage": "stage1_head_only",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 0.0,
        "head_lr": 1e-4,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage1_b3e5_h5e5),
        "total_parameters": int(total_params_stage1_b3e5_h5e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_b3e5_h5e5:
        best_val_loss_twostage_b3e5_h5e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage1_head_only",
            "method": tuning_method_twostage_b3e5_h5e5,
            "model_state_dict": model_twostage_b3e5_h5e5.state_dict(),
            "optimizer_state_dict": optimizer_stage1_b3e5_h5e5.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_b3e5_h5e5),
            "backbone_lr": 0.0,
            "head_lr": 1e-4,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage1_b3e5_h5e5),
            "total_parameters": int(total_params_stage1_b3e5_h5e5),
            "seed": SEED
        }, best_model_path_twostage_b3e5_h5e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 1 Epoch [{epoch+1}/{num_epochs_stage1_b3e5_h5e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

Stage 1 Epoch [1/10] | Train Loss: 0.919795 | Val Loss: 0.815187 | Time: 10.45s saved
Stage 1 Epoch [2/10] | Train Loss: 0.785822 | Val Loss: 0.754247 | Time: 10.86s saved
Stage 1 Epoch [3/10] | Train Loss: 0.742694 | Val Loss: 0.689760 | Time: 11.04s saved
Stage 1 Epoch [4/10] | Train Loss: 0.708323 | Val Loss: 0.652091 | Time: 11.15s saved
Stage 1 Epoch [5/10] | Train Loss: 0.683835 | Val Loss: 0.625876 | Time: 11.32s saved
Stage 1 Epoch [6/10] | Train Loss: 0.663790 | Val Loss: 0.604383 | Time: 11.44s saved
Stage 1 Epoch [7/10] | Train Loss: 0.645398 | Val Loss: 0.591312 | Time: 11.43s saved
Stage 1 Epoch [8/10] | Train Loss: 0.630737 | Val Loss: 0.572361 | Time: 11.28s saved
Stage 1 Epoch [9/10] | Train Loss: 0.617268 | Val Loss: 0.550857 | Time: 11.17s saved
Stage 1 Epoch [10/10] | Train Loss: 0.605106 | Val Loss: 0.534173 | Time: 11.11s saved


In [150]:
set_stage2_full_ft(model_twostage_b3e5_h5e5)

total_params_stage2_b3e5_h5e5, trainable_params_stage2_b3e5_h5e5 = count_trainable(model_twostage_b3e5_h5e5)

backbone_params_stage2_b3e5_h5e5 = []
head_params_stage2_b3e5_h5e5 = []

for name, param in model_twostage_b3e5_h5e5.named_parameters():
    if not param.requires_grad:
        continue

    if is_head_parameter(name):
        head_params_stage2_b3e5_h5e5.append(param)
    else:
        backbone_params_stage2_b3e5_h5e5.append(param)

optimizer_stage2_b3e5_h5e5 = torch.optim.AdamW(
    [
        {"params": backbone_params_stage2_b3e5_h5e5, "lr": 3e-5, "weight_decay": 1e-2},
        {"params": head_params_stage2_b3e5_h5e5, "lr": 5e-5, "weight_decay": 1e-2}
    ]
)

print("Stage 2 optimizer ready")
print("Backbone params:", sum(p.numel() for p in backbone_params_stage2_b3e5_h5e5))
print("Head params:", sum(p.numel() for p in head_params_stage2_b3e5_h5e5))

Total parameters: 157970996
Trainable parameters: 157970996
Trainable percentage: 100.0
Stage 2 optimizer ready
Backbone params: 157141562
Head params: 829434


In [151]:
num_epochs_stage2_b3e5_h5e5 = 30

for epoch in range(num_epochs_stage2_b3e5_h5e5):
    epoch_start = time.time()

    train_loss = train_one_epoch(
        model_twostage_b3e5_h5e5,
        train_loader,
        optimizer_stage2_b3e5_h5e5,
        criterion_twostage_b3e5_h5e5,
        device
    )

    val_loss = validate_one_epoch(
        model_twostage_b3e5_h5e5,
        val_loader,
        criterion_twostage_b3e5_h5e5,
        device
    )

    history_twostage_b3e5_h5e5.append({
        "method": tuning_method_twostage_b3e5_h5e5,
        "stage": "stage2_full_ft_discriminative_lr",
        "epoch": epoch + 1,
        "train_loss": float(train_loss),
        "val_loss": float(val_loss),
        "backbone_lr": 3e-5,
        "head_lr": 5e-5,
        "optimizer": "AdamW",
        "trainable_parameters": int(trainable_params_stage2_b3e5_h5e5),
        "total_parameters": int(total_params_stage2_b3e5_h5e5),
        "seed": SEED
    })

    if val_loss < best_val_loss_twostage_b3e5_h5e5:
        best_val_loss_twostage_b3e5_h5e5 = val_loss

        torch.save({
            "epoch": epoch + 1,
            "stage": "stage2_full_ft_discriminative_lr",
            "method": tuning_method_twostage_b3e5_h5e5,
            "model_state_dict": model_twostage_b3e5_h5e5.state_dict(),
            "optimizer_state_dict": optimizer_stage2_b3e5_h5e5.state_dict(),
            "best_val_loss": float(best_val_loss_twostage_b3e5_h5e5),
            "backbone_lr": 3e-5,
            "head_lr": 5e-5,
            "optimizer": "AdamW",
            "trainable_parameters": int(trainable_params_stage2_b3e5_h5e5),
            "total_parameters": int(total_params_stage2_b3e5_h5e5),
            "seed": SEED
        }, best_model_path_twostage_b3e5_h5e5)

        status = "saved"
    else:
        status = ""

    print(
        f"Stage 2 Epoch [{epoch+1}/{num_epochs_stage2_b3e5_h5e5}] | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"Time: {time.time() - epoch_start:.2f}s {status}"
    )

pd.DataFrame(history_twostage_b3e5_h5e5).to_csv(history_path_twostage_b3e5_h5e5, index=False)

print("Best val loss:", best_val_loss_twostage_b3e5_h5e5)
print("Total training time:", time.time() - start_time_twostage_b3e5_h5e5)

Stage 2 Epoch [1/30] | Train Loss: 0.405106 | Val Loss: 0.338561 | Time: 14.85s saved
Stage 2 Epoch [2/30] | Train Loss: 0.295639 | Val Loss: 0.267593 | Time: 14.90s saved
Stage 2 Epoch [3/30] | Train Loss: 0.246248 | Val Loss: 0.250422 | Time: 15.02s saved
Stage 2 Epoch [4/30] | Train Loss: 0.217253 | Val Loss: 0.214274 | Time: 15.01s saved
Stage 2 Epoch [5/30] | Train Loss: 0.194611 | Val Loss: 0.212369 | Time: 14.89s saved
Stage 2 Epoch [6/30] | Train Loss: 0.179960 | Val Loss: 0.186676 | Time: 14.88s saved
Stage 2 Epoch [7/30] | Train Loss: 0.166223 | Val Loss: 0.226370 | Time: 13.83s 
Stage 2 Epoch [8/30] | Train Loss: 0.157258 | Val Loss: 0.194768 | Time: 13.86s 
Stage 2 Epoch [9/30] | Train Loss: 0.147740 | Val Loss: 0.172081 | Time: 15.05s saved
Stage 2 Epoch [10/30] | Train Loss: 0.140558 | Val Loss: 0.168584 | Time: 14.93s saved
Stage 2 Epoch [11/30] | Train Loss: 0.135288 | Val Loss: 0.166243 | Time: 14.97s saved
Stage 2 Epoch [12/30] | Train Loss: 0.129337 | Val Loss: 0.168

In [152]:
checkpoint_twostage_b3e5_h5e5 = torch.load(
    best_model_path_twostage_b3e5_h5e5,
    map_location="cpu"
)

model_twostage_b3e5_h5e5 = BENDR_BP(n_times=625, n_channels_in=1)

model_twostage_b3e5_h5e5.eval()
with torch.no_grad():
    first_batch = next(iter(train_loader))
    x_dummy = first_batch[0].float()
    _ = model_twostage_b3e5_h5e5(x_dummy)

model_twostage_b3e5_h5e5.load_state_dict(checkpoint_twostage_b3e5_h5e5["model_state_dict"])

model_twostage_b3e5_h5e5 = model_twostage_b3e5_h5e5.to(device)
model_twostage_b3e5_h5e5.eval()

print("Loaded best two-stage backbone 3e-5 head 5e-5 model")
print("Best stage:", checkpoint_twostage_b3e5_h5e5["stage"])
print("Best epoch:", checkpoint_twostage_b3e5_h5e5["epoch"])
print("Best val loss:", checkpoint_twostage_b3e5_h5e5["best_val_loss"])

Loaded best two-stage backbone 3e-5 head 5e-5 model
Best stage: stage2_full_ft_discriminative_lr
Best epoch: 30
Best val loss: 0.13509840220102584


In [153]:
y_true_twostage_b3e5_h5e5, y_pred_twostage_b3e5_h5e5 = evaluate_model(
    model_twostage_b3e5_h5e5,
    test_loader,
    y_scaler,
    device
)

print("y_true_twostage_b3e5_h5e5 shape:", y_true_twostage_b3e5_h5e5.shape)
print("y_pred_twostage_b3e5_h5e5 shape:", y_pred_twostage_b3e5_h5e5.shape)

print("y_true min/max:", y_true_twostage_b3e5_h5e5.min(), y_true_twostage_b3e5_h5e5.max())
print("y_pred min/max:", y_pred_twostage_b3e5_h5e5.min(), y_pred_twostage_b3e5_h5e5.max())

y_true_twostage_b3e5_h5e5 shape: (5064, 625)
y_pred_twostage_b3e5_h5e5 shape: (5064, 625)
y_true min/max: 60.0 179.5625
y_pred min/max: 44.842888 184.06995


In [154]:
from scipy.signal import find_peaks
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import json

def extract_sbp_dbp(abp_segment):
    abp_segment = np.asarray(abp_segment).squeeze()

    peaks, _ = find_peaks(abp_segment, distance=30)

    if len(peaks) == 0:
        sbp = abp_segment.max()
    else:
        sbp = abp_segment[peaks].mean()

    troughs, _ = find_peaks(-abp_segment, distance=15)

    if len(troughs) == 0:
        dbp = abp_segment.min()
    else:
        dbp = abp_segment[troughs].mean()

    return sbp, dbp


sbp_true_twostage_b3e5_h5e5, dbp_true_twostage_b3e5_h5e5 = [], []
sbp_pred_twostage_b3e5_h5e5, dbp_pred_twostage_b3e5_h5e5 = [], []

for true_sig, pred_sig in zip(y_true_twostage_b3e5_h5e5, y_pred_twostage_b3e5_h5e5):
    sbp_t, dbp_t = extract_sbp_dbp(true_sig)
    sbp_p, dbp_p = extract_sbp_dbp(pred_sig)

    sbp_true_twostage_b3e5_h5e5.append(sbp_t)
    dbp_true_twostage_b3e5_h5e5.append(dbp_t)
    sbp_pred_twostage_b3e5_h5e5.append(sbp_p)
    dbp_pred_twostage_b3e5_h5e5.append(dbp_p)

sbp_true_twostage_b3e5_h5e5 = np.array(sbp_true_twostage_b3e5_h5e5)
dbp_true_twostage_b3e5_h5e5 = np.array(dbp_true_twostage_b3e5_h5e5)
sbp_pred_twostage_b3e5_h5e5 = np.array(sbp_pred_twostage_b3e5_h5e5)
dbp_pred_twostage_b3e5_h5e5 = np.array(dbp_pred_twostage_b3e5_h5e5)

sbp_mae_twostage_b3e5_h5e5 = mean_absolute_error(sbp_true_twostage_b3e5_h5e5, sbp_pred_twostage_b3e5_h5e5)
dbp_mae_twostage_b3e5_h5e5 = mean_absolute_error(dbp_true_twostage_b3e5_h5e5, dbp_pred_twostage_b3e5_h5e5)

sbp_rmse_twostage_b3e5_h5e5 = np.sqrt(mean_squared_error(sbp_true_twostage_b3e5_h5e5, sbp_pred_twostage_b3e5_h5e5))
dbp_rmse_twostage_b3e5_h5e5 = np.sqrt(mean_squared_error(dbp_true_twostage_b3e5_h5e5, dbp_pred_twostage_b3e5_h5e5))

sbp_r2_twostage_b3e5_h5e5 = r2_score(sbp_true_twostage_b3e5_h5e5, sbp_pred_twostage_b3e5_h5e5)
dbp_r2_twostage_b3e5_h5e5 = r2_score(dbp_true_twostage_b3e5_h5e5, dbp_pred_twostage_b3e5_h5e5)

avg_mae_twostage_b3e5_h5e5 = (sbp_mae_twostage_b3e5_h5e5 + dbp_mae_twostage_b3e5_h5e5) / 2

print("BENDR Two-Stage Head 1e-4 10ep Then Full FT Backbone 3e-5 Head 5e-5 Results — Real mmHg Scale")
print("======================================================================================================")

print("SBP")
print(f"MAE  : {sbp_mae_twostage_b3e5_h5e5:.3f}")
print(f"RMSE : {sbp_rmse_twostage_b3e5_h5e5:.3f}")
print(f"R2   : {sbp_r2_twostage_b3e5_h5e5:.4f}")

print("\nDBP")
print(f"MAE  : {dbp_mae_twostage_b3e5_h5e5:.3f}")
print(f"RMSE : {dbp_rmse_twostage_b3e5_h5e5:.3f}")
print(f"R2   : {dbp_r2_twostage_b3e5_h5e5:.4f}")

print("\nTable format:")
print(f"{sbp_mae_twostage_b3e5_h5e5:.2f}/{dbp_mae_twostage_b3e5_h5e5:.2f}")

print("\nAverage MAE:")
print(f"{avg_mae_twostage_b3e5_h5e5:.3f}")

BENDR Two-Stage Head 1e-4 10ep Then Full FT Backbone 3e-5 Head 5e-5 Results — Real mmHg Scale
SBP
MAE  : 5.547
RMSE : 7.804
R2   : 0.6994

DBP
MAE  : 4.622
RMSE : 6.198
R2   : 0.4890

Table format:
5.55/4.62

Average MAE:
5.085


In [155]:
metrics_twostage_b3e5_h5e5 = {
    "model": "BENDR",
    "method": tuning_method_twostage_b3e5_h5e5,
    "best_stage": checkpoint_twostage_b3e5_h5e5["stage"],
    "best_epoch": int(checkpoint_twostage_b3e5_h5e5["epoch"]),
    "best_val_loss": float(checkpoint_twostage_b3e5_h5e5["best_val_loss"]),
    "SBP_MAE": float(sbp_mae_twostage_b3e5_h5e5),
    "DBP_MAE": float(dbp_mae_twostage_b3e5_h5e5),
    "AVG_MAE": float(avg_mae_twostage_b3e5_h5e5),
    "SBP_RMSE": float(sbp_rmse_twostage_b3e5_h5e5),
    "DBP_RMSE": float(dbp_rmse_twostage_b3e5_h5e5),
    "SBP_R2": float(sbp_r2_twostage_b3e5_h5e5),
    "DBP_R2": float(dbp_r2_twostage_b3e5_h5e5),
    "table_format": f"{sbp_mae_twostage_b3e5_h5e5:.2f}/{dbp_mae_twostage_b3e5_h5e5:.2f}",
    "stage1_epochs": 10,
    "stage1_head_lr": 1e-4,
    "stage2_epochs": 30,
    "stage2_backbone_lr": 3e-5,
    "stage2_head_lr": 5e-5,
    "optimizer": checkpoint_twostage_b3e5_h5e5["optimizer"],
    "trainable_parameters": int(checkpoint_twostage_b3e5_h5e5["trainable_parameters"]),
    "total_parameters": int(checkpoint_twostage_b3e5_h5e5["total_parameters"]),
    "seed": int(checkpoint_twostage_b3e5_h5e5["seed"])
}

with open(metrics_path_twostage_b3e5_h5e5, "w") as f:
    json.dump(metrics_twostage_b3e5_h5e5, f, indent=4)

metrics_twostage_b3e5_h5e5

{'model': 'BENDR',
 'method': 'two_stage_head1e-4_10ep_then_full_ft_backbone3e-5_head5e-5',
 'best_stage': 'stage2_full_ft_discriminative_lr',
 'best_epoch': 30,
 'best_val_loss': 0.13509840220102584,
 'SBP_MAE': 5.547004699707031,
 'DBP_MAE': 4.622171401977539,
 'AVG_MAE': 5.084588050842285,
 'SBP_RMSE': 7.8036969446947815,
 'DBP_RMSE': 6.198327965005367,
 'SBP_R2': 0.6993532180786133,
 'DBP_R2': 0.4889869689941406,
 'table_format': '5.55/4.62',
 'stage1_epochs': 10,
 'stage1_head_lr': 0.0001,
 'stage2_epochs': 30,
 'stage2_backbone_lr': 3e-05,
 'stage2_head_lr': 5e-05,
 'optimizer': 'AdamW',
 'trainable_parameters': 157970996,
 'total_parameters': 157970996,
 'seed': 42}